# Warehouse Activity Profiling and KPI Assessment — Mikel Barrena and June Ona

This notebook analyzes the picking and inventory operations of the Philadelphia
distribution center.
**Structure:**
- Part 0 — Data loading and cleaning
- Part I — Activity Profiles (Heatmap, Customer Order, Item Activity, Inventory)
- Part II — KPI Assessment (Time, Quality, Productivity)

## Project structure

This notebook is the entry point. Run it from the project root (see `README.md`). It checks the original files in `data/raw/`, runs every analysis (WH-AS1 to WH-AS5) and saves each table (`.csv`, and `.tex` for the report tables) and figure (`.png` and `.pdf`) used in the report to `artifacts/`.

**Settings recorded in the code:** the scenarios S0 to S7 are generated from the three decisions (section 1 of WH-AS5), the model assumptions are in the first cells of WH-AS4 and WH-AS5 (also saved in `as5_common_basis.csv`), and the random draws of the worst case use the seeds 0 to 4. The analysis has no other random step.


In [1]:
# Colab only: upload warehouse-project.zip when asked, then unzip it.
# Outside Colab (or if the project folder is already here) this cell does nothing.
import os, zipfile
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB and not os.path.exists('src'):
    files.upload()                                  # choose warehouse-project.zip
    zipfile.ZipFile("warehouse-project.zip").extractall(".")
    os.chdir("warehouse-project")
    get_ipython().system('pip install -q xlrd==2.0.1 seaborn')

In [2]:
# --- Project setup: paths and helpers to save evidence --------------------
import sys
import textwrap
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / "data" / "sources.json").is_file():   # e.g. notebook started in a sub-folder
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

DATA_DIR = ROOT / "data" / "raw"          # original files (never edited)
ARTIFACTS = ROOT / "artifacts"            # every table and figure used in the report
ARTIFACTS.mkdir(exist_ok=True)


def _tex(s):
    """Make a text safe for LaTeX."""
    s = str(s)
    for a, b in [('\\', ' '), ('&', '\\&'), ('%', '\\%'), ('_', '\\_'), ('#', '\\#'), ('–', '--'),
                 ('Δ', '$\\Delta$'), ('×', '$\\times$'), ('→', '$\\to$'), ('²', '\\textsuperscript{2}'), ('≥', '$\\geq$'), ('≤', '$\\leq$')]:
        s = s.replace(a, b)
    return s


def save_table(df, name, index=False, caption=None, label=None, wide=False, col_format=None, digits=1):
    """Save a table (the data behind a result) as artifacts/<name>.csv.
    With a caption it also saves artifacts/<name>.tex (booktabs table for the report)."""
    path = ARTIFACTS / f"{name}.csv"
    df.to_csv(path, index=index)
    print(f"[saved] {path.relative_to(ROOT)}")
    if caption is not None:
        t = df.reset_index() if index else df.copy()
        for c in t.columns:
            if pd.api.types.is_float_dtype(t[c]):
                t[c] = t[c].map(lambda v: f"{v:.{digits}f}")
        def _head(c):                                      # break long headers into short lines
            lines = textwrap.wrap(_tex(c), 13, break_long_words=False)
            return "\\shortstack{" + "\\\\".join(lines) + "}" if len(lines) > 1 else _tex(c)
        t.columns = [_head(c) for c in t.columns]
        t = t.map(_tex) if hasattr(t, "map") else t.applymap(_tex)
        tex = t.to_latex(index=False, escape=False, caption=_tex(caption), label=label, position="t",
                         column_format=col_format or "l" + "c" * (len(t.columns) - 1))
        if wide:                                            # full-width table for a two-column page
            tex = tex.replace("{table}", "{table*}").replace("\\begin{table*}[t]", "\\begin{table*}[t]\n\\small")
        # scale the table down if it is wider than the column (needs \fitwidth in main.tex)
        width = "\\textwidth" if wide else "\\columnwidth"
        tex = tex.replace("\\begin{tabular}", "\\fitwidth{" + width + "}{\\begin{tabular}", 1).replace("\\end{tabular}", "\\end{tabular}}", 1)
        (ARTIFACTS / f"{name}.tex").write_text(tex, encoding="utf-8")
        print(f"[saved] {(ARTIFACTS / f'{name}.tex').relative_to(ROOT)}")


def save_fig(name, fig=None):
    """Save a figure as artifacts/<name>.png and .pdf, then show it."""
    fig = fig or plt.gcf()
    for ext, kw in [("png", {"dpi": 200}), ("pdf", {})]:
        path = ARTIFACTS / f"{name}.{ext}"
        fig.savefig(path, bbox_inches="tight", **kw)
        print(f"[saved] {path.relative_to(ROOT)}")
    plt.show()


## Part 0.1 — Loading the raw files

We have 10 raw files. Some are simple (Excel files, or text files with a real
header row and tab separators). Others need special handling because they have
no header row and fixed-width columns — we will load those in the next step.

In this first cell we only load the 7 "easy" files:
- 5 Excel files: active locations, reserve cases, min/max replenishment levels,
  item dimensions, and the warehouse layout map
- 2 tab-separated text files: item master data and sales history

We print the shape (rows, columns) of each one so we can confirm nothing failed
silently.

In [3]:
import os
import pandas as pd

from src.pipeline import ensure_data

# Check (or download from the approved course URLs) every original file in data/raw/.
ensure_data(include_all=True)
folder_path = DATA_DIR

raw = {}  # Diccionario para almacenar las tablas

# --- 5 archivos Excel ---
excel_files = {
    'active':    'DC23ACTIVE AS OF 050210.xls',   # active picking locations
    'cases':     'DC23CASES AS OF 050210.xls',    # reserve locations
    'minmax':    'DC23MINMAX2.xls',               # min/max replenishment rules
    'dims':      'Phily_Dim_Status2.xls',         # SKU dimensions and status
    'bcf_zones': 'BCF-zones.xls',                 # warehouse map (no header, raw grid)
}

for key, fname in excel_files.items():
    path = os.path.join(folder_path, fname)
    if os.path.exists(path):
        header_param = None if key == 'bcf_zones' else 0
        raw[key] = pd.read_excel(path, header=header_param)
        print(f"{key:12s} -> {raw[key].shape}   (desde {fname})")
    else:
        print(f"❌ No se encontró: {fname}")

# --- 2 archivos de texto (TSV) ---
text_files = {
    'items': 'ITEMDATAV2.txt',
    'sales': 'dc23Sales04.txt',   # ~190MB
}

for key, fname in text_files.items():
    path = os.path.join(folder_path, fname)
    if os.path.exists(path):
        raw[key] = pd.read_csv(path, sep='\t', low_memory=False)
        print(f"{key:12s} -> {raw[key].shape}   (desde {fname})")
    else:
        print(f"❌ No se encontró: {fname}")

Using checked local file: data\raw\BCF-zones.xls
Using checked local file: data\raw\DC23CASES AS OF 050210.xls
Using checked local file: data\raw\DC23ACTIVE AS OF 050210.xls
Using checked local file: data\raw\Phily_Dim_Status2.xls
Using checked local file: data\raw\ITEMDATAV2.txt
Using checked local file: data\raw\DC23MINMAX2.xls


Using checked local file: data\raw\dc23Sales04.txt
Using checked local file: data\raw\TRCART23.txt
Using checked local file: data\raw\CDCART23.txt
Using checked local file: data\raw\CHCART23.txt
Using checked local file: data\raw\batch-key.txt
Checked 11 approved source files in data\raw


active       -> (20805, 7)   (desde DC23ACTIVE AS OF 050210.xls)


cases        -> (49030, 9)   (desde DC23CASES AS OF 050210.xls)


minmax       -> (21193, 4)   (desde DC23MINMAX2.xls)


dims         -> (22904, 21)   (desde Phily_Dim_Status2.xls)
bcf_zones    -> (48, 181)   (desde BCF-zones.xls)
items        -> (35637, 47)   (desde ITEMDATAV2.txt)


sales        -> (2249641, 10)   (desde dc23Sales04.txt)


## Part 0.2 — Loading the 3 fixed-width files

`CDCART23.txt`, `CHCART23.txt`, and `TRCART23.txt` don't have a header row or
a separator — each field is just at a fixed position in the text. We checked
the real positions by hand and used `pd.read_fwf` to read them.

We load everything as text first (not numbers), because these are IDs and
codes, not real numbers. If we convert too early, we could lose things like
leading zeros. We'll turn the columns we actually need into numbers/dates
later.

In [4]:
# --- CDCART23.txt: carton detail (15 fields) ---
cd_colspecs = [
    (0, 8), (10, 30), (32, 34), (34, 37), (37, 42), (42, 54),
    (59, 65), (70, 76), (76, 77), (80, 83), (84, 85), (88, 89), (92, 93),
    (98, 106), (108, 114)
]
cd_names = [
    "pickticket_nbr", "carton_nbr", "carton_line_nbr", "style", "color",
    "style_sfx", "to_be_packed_units", "units_packed", "zone", "aisle",
    "bay", "level", "position", "date_created", "time_created"
]

# --- CHCART23.txt: carton header (7 fields) ---
ch_colspecs = [(0, 20), (20, 28), (30, 41), (41, 43), (43, 46), (48, 56), (58, 64)]
ch_names = [
    "carton_nbr", "pickticket_nbr", "print_wave", "pack_code",
    "truck_plan_id", "date_created", "time_created"
]

# --- TRCART23.txt: picking trace / cart assignment (18 fields) ---
tr_colspecs = [
    (0, 10), (10, 14), (14, 15), (18, 21), (22, 23), (26, 27), (30, 31),
    (34, 37), (37, 42), (42, 54), (56, 58), (63, 69), (69, 89), (89, 97),
    (103, 105), (105, 116), (118, 126), (128, 134)
]
tr_names = [
    "truck_nbr", "area_ignore", "zone", "aisle", "bay", "level", "position",
    "style", "color", "style_sfx", "carton_line_nbr", "ctn_pkt_qty",
    "carton_nbr", "pickticket_nbr", "pkt_line_nbr", "print_wave",
    "date_created", "time_created"
]

fwf_specs = {
    'CDCART23.txt': (cd_colspecs, cd_names, 'cdcart'),
    'CHCART23.txt': (ch_colspecs, ch_names, 'chcart'),
    'TRCART23.txt': (tr_colspecs, tr_names, 'trcart'),
}

for fname, (colspecs, names, key) in fwf_specs.items():
    path = os.path.join(folder_path, fname)
    df_fwf = pd.read_fwf(path, colspecs=colspecs, names=names, dtype=str, encoding='latin1')
    for col in df_fwf.columns:
        df_fwf[col] = df_fwf[col].str.strip()   # remove padding spaces
    raw[key] = df_fwf
    print(f"{key:8s} -> {raw[key].shape}   (from {fname})")

cdcart   -> (182128, 15)   (from CDCART23.txt)


chcart   -> (126033, 7)   (from CHCART23.txt)


trcart   -> (138674, 18)   (from TRCART23.txt)


## Part 0.3 — Cleaning `items` and `sales`

**`items`**: column names are already clean, we just lowercase them and
make sure `sku` is a clean string, since we use it to join with other files.

**`sales`**: needs a bit more work:
- Rename columns to clearer names (`item` → `sku`, `ordqty` → `qty_ordered`, etc.)
- Turn `txndate` into a real date, so we can get year, month, hour later
- Drop rows with a broken date (there are always a few in real data)
- Flag "bulk" orders (top 1% biggest orders). These are big wholesale
  orders, very different from a normal order, and could mess up our charts
  if we don't separate them.

In [5]:
# --- Clean items ---
df_items = raw['items'].copy()
df_items.columns = df_items.columns.str.strip().str.lower().str.replace(' ', '_').str.replace('/', '_')
df_items.rename(columns={'item_number': 'sku'}, inplace=True)
df_items['sku'] = df_items['sku'].astype(str).str.strip()

print("items columns:", list(df_items.columns)[:10], "...")
print("items shape:", df_items.shape)

# --- Clean sales ---
df_sales = raw['sales'].copy()
df_sales.rename(columns={
    'item': 'sku',
    'ordnbr': 'order_id',
    'linenbr': 'line_num',
    'cusnbr': 'customer_id',
    'ordqty': 'qty_ordered',
    'shpqty': 'qty_shipped',
    'txndate': 'transaction_date'
}, inplace=True)

df_sales['sku'] = df_sales['sku'].astype(str).str.strip()
df_sales['order_id'] = df_sales['order_id'].astype(str)

rows_before = len(df_sales)
df_sales['transaction_date'] = pd.to_datetime(df_sales['transaction_date'], errors='coerce')
df_sales = df_sales.dropna(subset=['transaction_date']).copy()
rows_dropped = rows_before - len(df_sales)
print(f"sales: dropped {rows_dropped} rows ({rows_dropped/rows_before:.3%}) with an unparseable date")

df_sales['year'] = df_sales['transaction_date'].dt.year
df_sales['month'] = df_sales['transaction_date'].dt.month
df_sales['hour'] = df_sales['transaction_date'].dt.hour

# Flag the top 1% largest orders (by quantity shipped) as "bulk"
qty_threshold = df_sales['qty_shipped'].quantile(0.99)
df_sales['is_bulk_order'] = df_sales['qty_shipped'] > qty_threshold
print(f"sales: bulk-order threshold = {qty_threshold:.0f} units, {df_sales['is_bulk_order'].sum()} rows flagged as bulk")

print("sales shape:", df_sales.shape)
print("date range:", df_sales['transaction_date'].min(), "to", df_sales['transaction_date'].max())

items columns: ['sku', 'description', 'vendor_id', 'hazmat_code', 'selling_uom', 'stocking_status', 'item_status', 'selling_length', 'selling_width', 'selling_height'] ...
items shape: (35637, 47)


sales: dropped 1 rows (0.000%) with an unparseable date


sales: bulk-order threshold = 60 units, 21051 rows flagged as bulk
sales shape: (2249640, 14)
date range: 2004-01-01 15:17:00 to 2005-01-02 22:25:00


## Part 0.4 — Filtering item status and cleaning dimensions

The assignment says to drop discontinued items: keep only `item_status = A`
(active), and exclude `D`, `X`, `P`. For `stocking_status`, we keep `Y` and
`X`, and exclude `N`.

We also found a value we didn't expect: `'R'` in `stocking_status` (40
rows). It's not explained anywhere, so we play it safe and exclude it too.
This is a good example of the "messy data" the assignment talks about.

We also clean **`dims`**, which has the size of each SKU. We'll need this
later to calculate volume (cube) for one of the Item Activity profiles.

In [6]:
# --- Filter items to active, stocked SKUs only ---
print("item_status counts:\n", df_items['item_status'].value_counts(dropna=False))
print("\nstocking_status counts:\n", df_items['stocking_status'].value_counts(dropna=False))

df_items_active = df_items[
    (df_items['item_status'] == 'A') &
    (df_items['stocking_status'].isin(['Y', 'X']))
].copy()

print(f"\nitems: kept {len(df_items_active)} of {len(df_items)} rows "
      f"({len(df_items_active)/len(df_items):.1%}) after filtering discontinued/unstocked SKUs")

# --- Clean dims (Phily_Dim_Status2) ---
df_dims = raw['dims'].copy()
df_dims.columns = df_dims.columns.str.strip().str.lower()
df_dims['sku'] = df_dims['sku'].astype(str).str.strip()

# Selling-unit cube in cubic feet (dimensions are in inches: 1728 in^3 = 1 ft^3)
df_dims['unit_cube_ft3'] = (df_dims['selllen'] * df_dims['sellwid'] * df_dims['sellhgt']) / 1728

print("\ndims shape:", df_dims.shape)
print(df_dims[['sku', 'selllen', 'sellwid', 'sellhgt', 'unit_cube_ft3']].head())
print("missing unit_cube_ft3:", df_dims['unit_cube_ft3'].isna().sum())

item_status counts:
 item_status
A    29054
D     3418
X     2672
P      493
Name: count, dtype: int64

stocking_status counts:
 stocking_status
Y    23537
N    11389
X      671
R       40
Name: count, dtype: int64

items: kept 22652 of 35637 rows (63.6%) after filtering discontinued/unstocked SKUs

dims shape: (22904, 22)
        sku  selllen  sellwid  sellhgt  unit_cube_ft3
0  AAG47294      6.2      6.1      1.7       0.037207
1  AAG47342     11.9     12.0      2.4       0.198333
2  AAG47344      8.6      5.9      1.5       0.044045
3  AAG47346     10.3      8.1      1.9       0.091734
4  AAG47348     13.7     12.0      2.5       0.237847
missing unit_cube_ft3: 0


## Part 0.5 — Cleaning the location-level inventory files

`active`, `cases`, and `minmax` all describe **where SKUs physically live** in
the warehouse:
- `active`: current quantity in each active (pick-face) location
- `cases`: cartons/pallets sitting in reserve locations
- `minmax`: the replenishment trigger (min) and target (max) for each active location

We just lowercase column names and make sure `sku`, `zone`, and `aisle` are
clean strings (not floats, not padded with spaces) so they join correctly
with everything else later.

In [7]:
# --- active ---
df_active = raw['active'].copy()
df_active.columns = df_active.columns.str.strip().str.lower()
df_active['sku'] = df_active['sku'].astype(str).str.strip()
df_active['zone'] = df_active['zone'].astype(str).str.strip()
print("active:", df_active.shape)
print(df_active.head(3))

# --- cases ---
df_cases = raw['cases'].copy()
df_cases.columns = df_cases.columns.str.strip().str.lower().str.replace(' ', '_')
df_cases['sku'] = df_cases['sku'].astype(str).str.strip()
df_cases['zone'] = df_cases['zone'].astype(str).str.strip()
print("\ncases:", df_cases.shape)
print(df_cases.head(3))
print("location_type values:", df_cases['location_type'].value_counts(dropna=False))

# --- minmax ---
df_minmax = raw['minmax'].copy()
df_minmax.columns = df_minmax.columns.str.strip().str.lower()
df_minmax['sku'] = df_minmax['sku'].astype(str).str.strip()
print("\nminmax:", df_minmax.shape)
print(df_minmax.head(3))

active: (20805, 7)
  zone  aisle bay  level position          sku  quantity
0    A  102.0   A    2.0        B  SPRSP2-4915      22.0
1    A  102.0   A    5.0        B     AVE05731      83.0
2    A  102.0   B    1.0        B     WLJ36610      15.0

cases: (49030, 9)
   case_number           sku  quantity zone  aisle bay level position  \
0   2300378238  SPRSP111-1/5         5    B    202   A     T        T   
1   2300378239  SPRSP111-1/5         5    B    202   A     T        T   
2   2350105143      SPR02321         6    B    202   A     T        T   

  location_type  
0           BLU  
1           BLU  
2           BLU  
location_type values: location_type
KCN    41897
BLU     3870
PLC     3202
PLP       61
Name: count, dtype: int64

minmax: (21193, 4)
           sku location  max  min
0  SPRSP2-4915  A102A2B   54   14
1     AVE05731  A102A5B  114   29
2     WLJ36610  A102B1B   48    6


## Part 0.6 — Cleaning the picking trace files (`cdcart`, `chcart`, `trcart`)

These three files show what was picked, when, and from where.

- **`trcart`**: every pick, grouped into a picking cart. Only active
  locations show up here (reserve picks are not batched this way).
- **`cdcart`**: every carton line, active and reserve.
- **`chcart`**: one row per carton, telling us if it's from **reserve**
  (`pack_code == '01'`) or **active** (`pack_code` blank).

For each file we:
1. Build the SKU code from `style` + `style_sfx` (same format we saw in `sales`)
2. Build `pick_datetime` from the separate date and time columns
3. Turn quantity columns into numbers

At the end, we add `pick_source` (Active/Reserve) from `chcart` onto
`cdcart`, using `carton_nbr`, so later we can split activity by zone into
active vs. reserve.

In [8]:
import numpy as np

# --- trcart ---
df_trc = raw['trcart'].copy()
df_trc['sku'] = (df_trc['style'].fillna('') + df_trc['style_sfx'].fillna('')).str.strip()
df_trc['pick_qty'] = pd.to_numeric(df_trc['ctn_pkt_qty'], errors='coerce')
df_trc['zone'] = df_trc['zone'].astype(str).str.strip()
df_trc['aisle'] = df_trc['aisle'].astype(str).str.strip()
df_trc['wave_id'] = df_trc['print_wave']

d_str = df_trc['date_created'].str.zfill(8)
t_str = df_trc['time_created'].str.zfill(6)
df_trc['pick_datetime'] = pd.to_datetime(d_str + t_str, format='%Y%m%d%H%M%S', errors='coerce')

print("trcart:", df_trc.shape, "| unparseable datetimes:", df_trc['pick_datetime'].isna().sum())
print(df_trc[['sku', 'zone', 'aisle', 'level', 'pick_qty', 'wave_id', 'pick_datetime']].head(3))

# --- chcart ---
df_chc = raw['chcart'].copy()
d_str = df_chc['date_created'].str.zfill(8)
t_str = df_chc['time_created'].str.zfill(6)
df_chc['pick_datetime'] = pd.to_datetime(d_str + t_str, format='%Y%m%d%H%M%S', errors='coerce')
df_chc['pick_source'] = np.where(df_chc['pack_code'] == '01', 'Reserve', 'Active')

print("\nchcart:", df_chc.shape)
print(df_chc['pick_source'].value_counts(normalize=True).mul(100).round(1))

# --- cdcart ---
df_cdc = raw['cdcart'].copy()
df_cdc['sku'] = (df_cdc['style'].fillna('') + df_cdc['style_sfx'].fillna('')).str.strip()
df_cdc['to_be_packed_units'] = pd.to_numeric(df_cdc['to_be_packed_units'], errors='coerce')
df_cdc['units_packed'] = pd.to_numeric(df_cdc['units_packed'], errors='coerce')
df_cdc['zone'] = df_cdc['zone'].astype(str).str.strip()
df_cdc['aisle'] = df_cdc['aisle'].astype(str).str.strip()

# Attach pick_source (Active/Reserve) from chcart via carton_nbr
df_cdc = df_cdc.merge(
    df_chc[['carton_nbr', 'pick_source']].drop_duplicates('carton_nbr'),
    on='carton_nbr', how='left'
)

print("\ncdcart:", df_cdc.shape)
print("cdcart pick_source coverage:", df_cdc['pick_source'].notna().mean().round(3))
print(df_cdc[['sku', 'zone', 'aisle', 'to_be_packed_units', 'units_packed', 'pick_source']].head(3))

trcart:

 (138674, 22) | unparseable datetimes: 1
           sku zone aisle level  pick_qty      wave_id       pick_datetime
0  BICVCG11-BE    B   231     3      12.0  20050225001 2005-02-25 12:30:21
1    ESSH10U13    A   101     1       1.0  20050225032 2005-02-25 19:04:54
2     ELD24411    A   108     3       1.0  20050225032 2005-02-25 19:04:54

chcart: (126033, 9)
pick_source
Active     64.8
Reserve    35.2
Name: proportion, dtype: float64



cdcart: (182128, 17)
cdcart pick_source coverage: 1.0
           sku zone aisle  to_be_packed_units  units_packed pick_source
0  BICVCG11-BE    B   231                12.0          12.0      Active
1    ESSH10U13    A   101                 1.0           1.0      Active
2     ELD24411    A   108                 1.0           1.0      Active


## Part I — Activity Profiles




### AP2 — Customer Order Profile: Lines-per-Order Distribution

This profile shows how many order lines (distinct items) a typical order
contains. It matters because **single-line orders can be batch-picked** much
more efficiently than multi-line orders (one picker collects many single-line
orders in one aisle pass, instead of walking the whole warehouse per order).

We use the full `sales` table (not just active picks) since an order's line
count is a property of the *order*, not of where it was picked from.

In [9]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

order_lines = (
    df_sales.groupby('order_id')['line_num']
    .nunique()
    .reset_index(name='line_count')
)

total_orders = len(order_lines)
single_line_orders = (order_lines['line_count'] == 1).sum()
single_line_pct = single_line_orders / total_orders * 100

print(f"Total orders: {total_orders:,}")
print(f"Single-line orders: {single_line_orders:,} ({single_line_pct:.1f}%)")
print(f"Average lines per order: {order_lines['line_count'].mean():.2f}")
print(f"Median lines per order: {order_lines['line_count'].median():.0f}")

lines_dist = order_lines['line_count'].value_counts().reset_index()
lines_dist.columns = ['lines', 'count']
lines_dist = lines_dist[lines_dist['lines'] <= 15].sort_values('lines')

fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=lines_dist, x='lines', y='count', color='teal', ax=ax)
ax.set_title('Lines per Order Distribution (orders with 1-15 lines shown)', fontsize=12, fontweight='bold')
ax.set_xlabel('Number of Lines per Order')
ax.set_ylabel('Number of Orders')
plt.tight_layout()
save_fig('ap2_lines_per_order')
save_table(lines_dist, 'ap2_lines_per_order')

Total orders: 883,910
Single-line orders: 545,720 (61.7%)
Average lines per order: 2.40
Median lines per order: 1


[saved] artifacts\ap2_lines_per_order.png


[saved] artifacts\ap2_lines_per_order.pdf
[saved] artifacts\ap2_lines_per_order.csv


C:\Users\joneo\AppData\Local\Temp\ipykernel_15512\2522381755.py:62: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**AP2 — Lines-per-order profile.** Based on 883,910 orders (2004). **61.7%
of orders (545,720) have just one line**, and the numbers drop fast after
that (2 lines: 15.8%, 3 lines: 6.9%). Median = 1 line, average = 2.40.

*Improvement idea:* single-line orders can be batch-picked together instead
of one at a time. A picker collecting 10-15 orders per pass, instead of one
trip per order, could cut travel time a lot for the 6 out of 10 orders that
only need one line.

### AP3 — Customer Order Profile: Hour-of-Day Activity Distribution

This profile shows **when** orders come in during the day. It's useful for
staffing: if orders spike at certain hours, picking staff should be
scheduled for those hours instead of spread out evenly.

We use the `hour` column we already made from `transaction_date` in Part 0.3.

In [10]:
hourly_orders = (
    df_sales.groupby('hour')['order_id']
    .nunique()
    .reset_index(name='order_count')
)

peak_hour = hourly_orders.loc[hourly_orders['order_count'].idxmax()]

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=hourly_orders, x='hour', y='order_count', color='navy', ax=ax)
ax.set_title('Order Arrivals by Hour of Day', fontsize=12, fontweight='bold')
ax.set_xlabel('Hour of Day (0-23)')
ax.set_ylabel('Number of Distinct Orders')
plt.tight_layout()
save_fig('ap3_hour_of_day')
hourly_orders = (
    df_sales.groupby('hour')['order_id']
    .nunique()
    .reset_index(name='order_count')
)

total_orders = hourly_orders['order_count'].sum()
peak_hour = hourly_orders.loc[hourly_orders['order_count'].idxmax()]

print(f"Total orders: {total_orders:,}")
print(f"Peak hour: {int(peak_hour['hour']):02d}:00, with {int(peak_hour['order_count']):,} distinct orders "
      f"({peak_hour['order_count']/total_orders:.1%})")

afternoon = hourly_orders[hourly_orders['hour'].isin([15, 16, 17])]['order_count'].sum()
print(f"Orders in 15:00-17:00: {afternoon:,} ({afternoon/total_orders:.1%})")

print(hourly_orders.sort_values('order_count', ascending=False).head(5).to_string(index=False))
save_table(hourly_orders, 'ap3_orders_by_hour', index=True)

[saved] artifacts\ap3_hour_of_day.png
[saved] artifacts\ap3_hour_of_day.pdf


C:\Users\joneo\AppData\Local\Temp\ipykernel_15512\2522381755.py:62: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Total orders: 957,627
Peak hour: 16:00, with 188,321 distinct orders (19.7%)
Orders in 15:00-17:00: 453,482 (47.4%)
 hour  order_count
   16       188321
   15       146622
   17       118539
   14       112968
   11        87437
[saved] artifacts\ap3_orders_by_hour.csv


**AP3 — Hour-of-day order profile.** We counted 957,627 order-hour records
(some orders have lines placed in more than one hour, so they get counted
more than once here). Orders are very concentrated in the afternoon:
**15:00-17:00 alone is 47.4% of all order activity**, with a peak at
**16:00 (19.7%)**. Almost no orders come in before 07:00 or after 18:00.

*Context:* SPR accepts orders until around 5 PM, so this peak makes sense (customers order right before the cutoff).

*Improvement idea:* picking staff have to handle almost half of the day's
order activity in just 3 hours (15:00-18:00). Adding extra staff just for
this window, or encouraging big customers to order earlier, could ease the
pressure.

### AP4 — Item Activity Profile: SKU Popularity (ABC Pareto)

This profile ranks SKUs by total units shipped, to see if a few SKUs drive
most of the demand. If they do, those SKUs should go in the easiest-to-reach
spots ("golden zone").

We exclude the bulk orders (top 1%, flagged in Part 0.3), so a few huge
wholesale orders don't skew the ranking.

In [11]:


df_std = df_sales[~df_sales['is_bulk_order']].copy()

sku_activity = (
    df_std.groupby('sku')['qty_shipped']
    .sum()
    .sort_values(ascending=False)
    .reset_index(name='total_units')
)

sku_activity['cum_units'] = sku_activity['total_units'].cumsum()
sku_activity['cum_pct'] = sku_activity['cum_units'] / sku_activity['total_units'].sum() * 100
sku_activity['sku_pct'] = (sku_activity.index + 1) / len(sku_activity) * 100

conditions = [sku_activity['cum_pct'] <= 80.0, sku_activity['cum_pct'] <= 95.0]
sku_activity['abc_class'] = np.select(conditions, ['A', 'B'], default='C')

class_summary = sku_activity.groupby('abc_class').agg(
    num_skus=('sku', 'count'),
    total_units=('total_units', 'sum')
)
class_summary['pct_of_skus'] = class_summary['num_skus'] / len(sku_activity) * 100
class_summary['pct_of_volume'] = class_summary['total_units'] / sku_activity['total_units'].sum() * 100
print(class_summary)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(sku_activity['sku_pct'], sku_activity['cum_pct'], color='navy', linewidth=2)
ax.axhline(80, color='red', linestyle='--', label='80% of volume')
ax.axhline(95, color='orange', linestyle='--', label='95% of volume')
ax.set_title('SKU Pareto Curve — Cumulative Volume Concentration', fontsize=12, fontweight='bold')
ax.set_xlabel('Cumulative % of SKUs (ranked by volume)')
ax.set_ylabel('Cumulative % of Units Shipped')
ax.legend(loc='lower right')
plt.tight_layout()
save_fig('ap4_abc_pareto')
save_table(class_summary, 'ap4_abc_summary', index=True)

           num_skus  total_units  pct_of_skus  pct_of_volume
abc_class                                                   
A              4880    7604331.0    18.431091      79.999148
B              6415    1425829.0    24.228576      15.000018
C             15182     475355.0    57.340333       5.000834
[saved] artifacts\ap4_abc_pareto.png
[saved] artifacts\ap4_abc_pareto.pdf


[saved] artifacts\ap4_abc_summary.csv


C:\Users\joneo\AppData\Local\Temp\ipykernel_15512\2522381755.py:62: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**AP4 — SKU popularity profile (ABC Pareto).** Class A: just 4,880
SKUs (18.4%) make up 80% of units shipped. Class B adds 6,415 SKUs (24.2%)
for the next 15%. Finally, Class C, with 15,182 SKUs (57.3%),  only moves 5% of volume.

*Improvement idea:* put the 4,880 Class A SKUs in the easiest "golden zone"
spots near pack/ship, since they cover 80% of all picks. Class C SKUs could
move to denser, lower-priority storage to free up prime floor space.

### AP5 — Item Activity Profile: Popularity vs. Cube-Movement Profile

A SKU can sell a lot but still be small (like a pen), or sell a lot and be
bulky (like a box of paper). These need different shelving even if they
sell the same amount. This profile compares, for each ABC class from AP4,
its share of units shipped vs. its share of space (cube) moved. If a class
takes up much more space than its sales share, it's bulkier than it looks.

We reuse `sku_activity` and `abc_class` from AP4, and add each SKU's size
from `dims` to get `total_cube_ft3 = total_units x unit_cube_ft3`.

In [12]:
sku_cube = sku_activity.merge(
    df_dims[['sku', 'unit_cube_ft3']], on='sku', how='inner'
)
sku_cube['total_cube_ft3'] = sku_cube['total_units'] * sku_cube['unit_cube_ft3']

print(f"SKUs matched to dimension data: {len(sku_cube):,} of {len(sku_activity):,} "
      f"({len(sku_cube)/len(sku_activity):.1%})")

# Build one simple comparison table: % of units shipped vs. % of cube moved, per class
cube_by_class = sku_cube.groupby('abc_class').agg(
    num_skus=('sku', 'count'),
    total_units=('total_units', 'sum'),
    total_cube_ft3=('total_cube_ft3', 'sum')
).reset_index()

cube_by_class['pct_of_units'] = cube_by_class['total_units'] / cube_by_class['total_units'].sum() * 100
cube_by_class['pct_of_cube'] = cube_by_class['total_cube_ft3'] / cube_by_class['total_cube_ft3'].sum() * 100
cube_by_class['avg_cube_per_unit_ft3'] = cube_by_class['total_cube_ft3'] / cube_by_class['total_units']

print(cube_by_class.round(2))

# Reshape for a simple grouped bar chart: one bar for "% of units", one for "% of cube", per class
plot_data = cube_by_class.melt(
    id_vars='abc_class', value_vars=['pct_of_units', 'pct_of_cube'],
    var_name='metric', value_name='pct'
)
plot_data['metric'] = plot_data['metric'].map({
    'pct_of_units': '% of Units Shipped', 'pct_of_cube': '% of Cube (Space) Moved'
})

fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=plot_data, x='abc_class', y='pct', hue='metric', ax=ax)
ax.set_title('Popularity Share vs. Physical Space Share, by ABC Class', fontsize=12, fontweight='bold')
ax.set_xlabel('ABC Class')
ax.set_ylabel('% of Total (Units or Cube)')
ax.legend(title='')
plt.tight_layout()
save_fig('ap5_popularity_vs_cube')
save_table(cube_by_class, 'ap5_cube_by_class', index=True)

SKUs matched to dimension data: 21,734 of 26,477 (82.1%)
  abc_class  num_skus  total_units  total_cube_ft3  pct_of_units  pct_of_cube  \
0         A      4706    7473990.0       487478.43         81.09        53.56   
1         B      5944    1327955.0       241947.41         14.41        26.58   
2         C     11084     415209.0       180752.79          4.50        19.86   

   avg_cube_per_unit_ft3  
0                   0.07  
1                   0.18  
2                   0.44  
[saved] artifacts\ap5_popularity_vs_cube.png


[saved] artifacts\ap5_popularity_vs_cube.pdf
[saved] artifacts\ap5_cube_by_class.csv


C:\Users\joneo\AppData\Local\Temp\ipykernel_15512\2522381755.py:62: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**AP5 — Popularity vs. cube-movement profile.** Based on 21,734 SKUs (82.1%
of the assortment) matched to size data Class A makes up 81.1% of units
shipped but only 53.6% of space (cube) moved, so Class A items are
smaller on average (0.07 ft³/unit). Class C is the opposite: only 4.5% of
units shipped but 19.9% of cube (0.44 ft³/unit, 6x bigger than Class A).

*Improvement idea:* Class A's small size fits dense shelving near the pack
area. Class C takes up too much space for how little it sells, a good
candidate to move to reserve/overstock, freeing up prime shelf space.

### AP6 — Inventory Profile: Item-Family Inventory Footprint

This profile groups SKUs into families (based on keywords in the
description) and shows how much stock each family holds. It shows where
storage space is being used.

We build the family from `description` and `hazmat_code`. For stock, we
average the 12 monthly `_oh` columns (end-of-month inventory), instead of
using just one month, to get a more representative number.

In [13]:
def categorize_family(row):
    if str(row.get('hazmat_code', 'N')).strip().upper() == 'Y':
        return 'Hazardous Material'
    desc = str(row.get('description', '')).upper()
    if any(k in desc for k in ['CRTDG', 'TNR', 'INK', 'CARTRIDGE', 'ELECTRONIC', 'CABLE', 'PRINTER']):
        return 'Consumables / Tech'
    elif any(k in desc for k in ['PAPER', 'REFILL', 'PLNR', 'NOTEBOOK', 'BINDER', 'PAD', 'FOLDER']):
        return 'Paper / Binders'
    elif any(k in desc for k in ['CHAIR', 'DESK', 'FURNITURE', 'TABLE', 'SHELF', 'CABINET']):
        return 'Furniture / Equipment'
    else:
        return 'General / Other'

df_items_active['item_family'] = df_items_active.apply(categorize_family, axis=1)

oh_cols = [c for c in df_items_active.columns if c.endswith('oh')]
df_items_active['avg_inventory_units'] = df_items_active[oh_cols].mean(axis=1)

family_summary = (
    df_items_active.groupby('item_family')
    .agg(num_skus=('sku', 'count'), total_inventory_units=('avg_inventory_units', 'sum'))
    .reset_index()
    .sort_values('total_inventory_units', ascending=False)
)
family_summary['pct_of_inventory'] = family_summary['total_inventory_units'] / family_summary['total_inventory_units'].sum() * 100
family_summary['inventory_per_sku'] = family_summary['total_inventory_units'] / family_summary['num_skus']
print(family_summary.round(1))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.barplot(data=family_summary, x='total_inventory_units', y='item_family', color='steelblue', ax=axes[0])
axes[0].set_title('Total On-Hand Inventory by Item Family', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Total Units in Stock (On-Hand, 12-month avg)')
axes[0].set_ylabel('')

sns.barplot(data=family_summary.sort_values('inventory_per_sku', ascending=False),
            x='inventory_per_sku', y='item_family', color='darkorange', ax=axes[1])
axes[1].set_title('Inventory per SKU by Item Family', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Average Units in Stock per SKU')
axes[1].set_ylabel('')

plt.tight_layout()
save_fig('ap6_family_inventory')
save_table(family_summary, 'ap6_family_summary', index=True)

             item_family  num_skus  total_inventory_units  pct_of_inventory  \
2        General / Other     16157              1765463.7              65.8   
4        Paper / Binders      2584               474652.5              17.7   
3     Hazardous Material      1943               394424.4              14.7   
0     Consumables / Tech      1281                38943.0               1.5   
1  Furniture / Equipment       687                 9172.3               0.3   

   inventory_per_sku  
2              109.3  
4              183.7  
3              203.0  
0               30.4  
1               13.4  


[saved] artifacts\ap6_family_inventory.png


[saved] artifacts\ap6_family_inventory.pdf
[saved] artifacts\ap6_family_summary.csv


C:\Users\joneo\AppData\Local\Temp\ipykernel_15512\2522381755.py:62: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**AP6 — Item-family inventory profile.**  By total stock, `General / Other` dominates with 1,765,464
units (65.8%), but that's mostly because it has the most SKUs (71.3% of
all). By stock *per SKU*, the picture flips: `Hazardous Material` (203.0
units/SKU) and `Paper / Binders` (183.7) hold much more than `General /
Other` (109.3), while `Consumables / Tech` (30.4) and `Furniture` (13.4)
hold the least.

*Improvement idea:* `General / Other`'s big total is just a SKU-count
effect, so it should be sloted SKU-by-SKU (using the ABC classes from AP4)
instead of as one group. `Hazardous Material` is the real concern, high
stock per SKU plus special storage needs makes it worth reviewing its
min/max levels.

### AP7 — Inventory Profile: Handling-Unit Reserve Profile

This profile splits reserve locations into two types:

* **Case Reserve** (`BLU`/`KCN`): boxes have their own case ID, so workers
  can pull individual cartons.
* **Pallet Reserve** (`PLC`/`PLP`): one case ID covers a whole pallet, so
  workers can only pull full pallets.

This shows if reserve storage matches how items actually get replenished,
case by case, or pallet by pallet.

In [14]:
df_cases_valid = df_cases.copy()
df_cases_valid['location_type'] = df_cases_valid['location_type'].astype(str).str.strip().str.upper()

handling_map = {'BLU': 'Case Reserve', 'KCN': 'Case Reserve', 'PLC': 'Pallet Reserve', 'PLP': 'Pallet Reserve'}
df_cases_valid['handling_type'] = df_cases_valid['location_type'].map(handling_map)

unmapped = df_cases_valid[df_cases_valid['handling_type'].isna()]
print(f"Unmapped location_type rows: {len(unmapped)} ({len(unmapped)/len(df_cases_valid):.2%})")
print(unmapped['location_type'].value_counts())

df_cases_valid = df_cases_valid.dropna(subset=['handling_type']).copy()

handling_summary = df_cases_valid.groupby('handling_type').agg(
    num_locations=('case_number', 'count'),
    total_units=('quantity', 'sum'),
    num_skus=('sku', 'nunique')
).reset_index()
handling_summary['pct_of_units'] = handling_summary['total_units'] / handling_summary['total_units'].sum() * 100
print(handling_summary)

zone_handling = pd.crosstab(
    df_cases_valid['zone'], df_cases_valid['handling_type'],
    values=df_cases_valid['quantity'], aggfunc='sum'
).fillna(0)
zone_handling_pct = zone_handling.div(zone_handling.sum(axis=1), axis=0) * 100
print(zone_handling_pct.round(1))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.barplot(data=handling_summary, x='handling_type', y='total_units', color='seagreen', ax=axes[0])
axes[0].set_title('Reserve Inventory by Handling Type', fontsize=12, fontweight='bold')
axes[0].set_xlabel('')
axes[0].set_ylabel('Total Units in Reserve')

zone_handling_pct.plot(kind='bar', stacked=True, ax=axes[1], color=['seagreen', 'sandybrown'])
axes[1].set_title('Handling-Type Mix by Zone', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Zone')
axes[1].set_ylabel('% of Reserve Units')
axes[1].legend(title='')

plt.tight_layout()
save_fig('ap7_handling_unit_reserve')
save_table(handling_summary, 'ap7_handling_summary', index=True)
save_table(zone_handling, 'ap7_zone_handling', index=True)

Unmapped location_type rows: 0 (0.00%)
Series([], Name: count, dtype: int64)
    handling_type  num_locations  total_units  num_skus  pct_of_units
0    Case Reserve          45767       545696      8064     59.522984
1  Pallet Reserve           3263       371086      1403     40.477016
handling_type  Case Reserve  Pallet Reserve
zone                                       
B                     100.0             0.0
F                       3.8            96.2
R                     100.0             0.0
W                       6.6            93.4


[saved] artifacts\ap7_handling_unit_reserve.png
[saved] artifacts\ap7_handling_unit_reserve.pdf
[saved] artifacts\ap7_handling_summary.csv
[saved] artifacts\ap7_zone_handling.csv


C:\Users\joneo\AppData\Local\Temp\ipykernel_15512\2522381755.py:62: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**AP7 — Handling-unit reserve profile.** **Case Reserve holds 59.5% of units** (45,767 locations),
**Pallet Reserve holds 40.5%** (3,263 locations). Pallet spots store way
more each on average (113.7 units/location vs. 11.9), this means pallets are for
bulk, cases are for easier restocking.

Each zone sticks to one type: **B and R are 100% Case Reserve**, **F (96.2%)
and W (93.4%) are almost all Pallet Reserve**. Zone C has no reserve at all.

*Improvement idea:* a SKU in Zone F or W can only be restocked a full
pallet at a time, even if it doesn't sell fast. Checking if any Class A/B
SKUs (from AP4) are stuck there could show items that should move to a
Case Reserve zone instead.

### AP1 — Heatmap: Where Picking Activity Happens in the Real Warehouse

`BCF-zones.xls` is a grid version of the warehouse floor. Each real location
(like "Zone B, Aisle 215") is written as a short code (`B215`), split across
a few stacked cells. Some cells just spell out words like "RESERVE", not
real locations, so we ignore those.

**What the code does:**
1. Pull out only the cells that look like a real code (1 letter + 3 numbers).
2. Count picks per Zone+Aisle, using `trcart` (active-location picks only).
3. Plot each location at its real grid position, colored by pick count,
   so the heatmap is shaped like the real warehouse, not just a table.

**Note:** this file only covers Zones B, C, F, and R (that's what the name
"BCF" means).

In [15]:
import re
import matplotlib.pyplot as plt
import pandas as pd

# Step 1: decode real bay codes from the layout grid
df_bcf = raw['bcf_zones'].copy()
records = []

for col in df_bcf.columns:
    filled_rows = df_bcf[col][df_bcf[col].notna()].index.tolist()
    if not filled_rows:
        continue
    # group rows that are next to each other (a code is written top to bottom)
    runs, current_run = [], [filled_rows[0]]
    for row_idx in filled_rows[1:]:
        if row_idx == current_run[-1] + 1:
            current_run.append(row_idx)
        else:
            runs.append(current_run)
            current_run = [row_idx]
    runs.append(current_run)

    for run in runs:
        code = ''.join(str(df_bcf[col][r]) for r in run)
        if re.fullmatch(r'[A-Z]\d{3}', code):  # keep only real "Zone+Aisle" codes
            records.append({'row': run[0], 'col': col, 'zone': code[0], 'aisle': code[1:]})

bay_locations = pd.DataFrame(records).drop_duplicates(subset=['zone', 'aisle'])
print(f"Real bay locations found: {len(bay_locations)}")

# Step 2: count picks per Zone+Aisle
zone_aisle_activity = df_trc.groupby(['zone', 'aisle']).size().reset_index(name='pick_count')
heat_data = bay_locations.merge(zone_aisle_activity, on=['zone', 'aisle'], how='left')
heat_data['pick_count'] = heat_data['pick_count'].fillna(0)
print(f"Bays with at least 1 pick: {(heat_data['pick_count'] > 0).sum()} of {len(heat_data)}")

# Step 3: plot it at the real grid position
fig, ax = plt.subplots(figsize=(14, 5))
scatter = ax.scatter(heat_data['col'], heat_data['row'], c=heat_data['pick_count'],
                      cmap='YlOrRd', s=180, marker='s')

for _, row in heat_data.iterrows():
    ax.text(row['col'], row['row'], row['zone'],
            ha='center', va='center', fontsize=6, fontweight='bold', color='black')

ax.invert_yaxis()
plt.colorbar(scatter, label='Pick Count')
ax.set_title('AP1 — Spatial Picking Heatmap (Zones B, C, F, R)', fontsize=12, fontweight='bold')
ax.set_xlabel('Warehouse Layout — Column Position')
ax.set_ylabel('Row Position')
plt.tight_layout()
save_fig('ap1_heatmap')
save_table(zone_aisle_activity, 'ap1_zone_aisle_activity', index=True)

Real bay locations found: 140
Bays with at least 1 pick: 100 of 140


[saved] artifacts\ap1_heatmap.png
[saved] artifacts\ap1_heatmap.pdf
[saved] artifacts\ap1_zone_aisle_activity.csv


C:\Users\joneo\AppData\Local\Temp\ipykernel_15512\2522381755.py:62: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**AP1 — Spatial Picking Heatmap Analysis**

* **Coverage & Spatial Distribution:**
  * **100 of 140 bays (71%)** registered active picking activity.
  * **Zone R (40 bays at the bottom):** Shows **0 picks** across all locations (color gradient at 0), confirming its role as 100% reserve storage.

* **Activity Hotspots:**
  * **Zone B Concentration:** Displays the highest pick density, reaching peak activity levels of up to **~2,500 picks per bay** (dark red nodes on the top-left and mid-left aisles).
  * **Zone C vs Zone B:** Zone C (middle and top-right) shows low-to-moderate activity (light yellow/orange), despite occupying similar horizontal layout space.
  * **Zone F:** Shows minimal picking activity (single bay on top-left).

* **Optimization Opportunity:** Redistribute high-demand Class A SKUs from the saturated Zone B aisles to underutilized Zone C bays to reduce congestion and travel bottlenecks.

**Summary figure for the report.** The next figure puts the three most important profiles of AS1 side by side. It reuses the data of AP2, AP3 and AP4, so nothing new is calculated.

In [16]:
# --- Figure for the report (AS1): order profile, arrival profile and SKU Pareto in one figure ---
fig, axes = plt.subplots(1, 3, figsize=(7.2, 2.5))

# (a) lines per order (orders with 1-15 lines)
axes[0].bar(lines_dist['lines'], lines_dist['count'] / 1000, color='#4C78A8')
axes[0].set_xlabel('Lines per order')
axes[0].set_ylabel('Orders (thousands)')
axes[0].set_title('(a) Order size', fontsize=9)

# (b) order arrivals by hour of the day
axes[1].bar(hourly_orders['hour'], hourly_orders['order_count'] / 1000, color='#4C78A8')
axes[1].set_xlabel('Hour of the day')
axes[1].set_ylabel('Orders (thousands)')
axes[1].set_title('(b) Arrivals by hour', fontsize=9)

# (c) Pareto curve of SKU volume
axes[2].plot(sku_activity['sku_pct'], sku_activity['cum_pct'], color='#4C78A8', lw=2)
axes[2].axhline(80, color='#E45756', ls='--', lw=1)
axes[2].text(52, 82, '80% of units', color='#E45756', fontsize=7)
axes[2].set_xlabel('SKUs, ranked by volume (%)')
axes[2].set_ylabel('Cumulative units shipped (%)')
axes[2].set_title('(c) SKU Pareto', fontsize=9)

for ax in axes:
    ax.tick_params(labelsize=7)
    ax.xaxis.label.set_size(8); ax.yaxis.label.set_size(8)
plt.tight_layout()
save_fig('as1_summary_profiles')

[saved] artifacts\as1_summary_profiles.png
[saved] artifacts\as1_summary_profiles.pdf


C:\Users\joneo\AppData\Local\Temp\ipykernel_15512\2522381755.py:62: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


- **(a) Order size:** 61.7% of the orders have a single line. This is why batch picking and a good route matter.
- **(b) Arrivals:** most orders arrive in the afternoon. 47.4% of the order activity comes between 15:00 and 17:00, with a peak at 16:00. Staffing has to follow this peak.
- **(c) Pareto:** 18.4% of the SKUs (class A, 4,880 SKUs) give 80% of the units shipped. These are the SKUs that the storage assignment (AS2) puts closest to the packing area.

## Part II — KPI Assessment

### KPI1 — Time: Order Lead Time

We first tried "picking wave duration" (how long a wave takes). But we
checked the real timestamps in `TRCART23`, and found a problem: all picks in
the same wave happen within a few seconds of each other, even for a wave
with 900+ picks. So this field is not "when each item was picked", it's
just "when the wave was created". It cannot give us a real duration.

So we measure **Order Lead Time** instead, the time between when
the customer places the order (`sales`) and when the order's box is finished
(`chcart`). We connect both files using the order number (`order_id` in
`sales` = `pickticket_nbr` in `chcart`).

**One bug to fix first:** when we load the full `sales` file, the order
number becomes `23926870.0` instead of `23926870` (Python reads it as a
decimal number). We remove the `.0` so the join works.

**One thing to expect:** `sales` is from 2004, but `chcart` is from 2005 —
different periods. So the lead time will look very large (months, not
hours). This is expected and mentioned in the assignment's own FAQ, so we
just report it honestly.

In [17]:
# Order placement time
order_placed = (
    df_sales.groupby('order_id')['transaction_date']
    .min()
    .reset_index(name='order_time')
)

# Fix order_id format
order_placed['order_id'] = (
    order_placed['order_id']
    .astype(str)
    .str.replace(r'\.0$', '', regex=True)
)

# Order completion time
order_completed = (
    df_chc.groupby('pickticket_nbr')['pick_datetime']
    .max()
    .reset_index(name='ship_time')
    .rename(columns={'pickticket_nbr': 'order_id'})
)

# Match both datasets
lead_time = order_placed.merge(
    order_completed,
    on='order_id',
    how='inner'
)

# Calculate lead time in days
lead_time['lead_time_days'] = (
    lead_time['ship_time'] - lead_time['order_time']
).dt.total_seconds() / 86400

# Results
print(f"Orders matched: {len(lead_time):,} of {len(order_placed):,}")
print(f"Negative lead times: {(lead_time['lead_time_days'] < 0).sum()}")
print(lead_time['lead_time_days'].describe())

kpi1_median_lead_days = lead_time['lead_time_days'].median()

print(f"\nKPI1 - Median Order Lead Time: {kpi1_median_lead_days:.1f} days")


Orders matched: 59,382 of 883,910
Negative lead times: 0
count    59382.000000
mean       322.236620
std          1.044454
min        314.950278
25%        321.940081
50%        322.068252
75%        322.231644
max        329.885949
Name: lead_time_days, dtype: float64

KPI1 - Median Order Lead Time: 322.1 days


**KPI1 — Order Lead Time.** We could match 59,382 orders between `sales`
and `chcart`, out of 883,910 total orders (most don't match because the two
files are from different time periods). The result: a median of **322.1
days**, and almost every order gives a very similar number (between 315 and
330 days).

**Why this number is not real:** that ~322 days is just the calendar gap
between the `sales` file (2004) and the `chcart` file (2005), not a real
delivery time. The fact that almost all orders give a similar number is
actually a good sign: if our join had an error, we would see random, mixed
values, not values this close together.

**What this means for the report:** we can't give a real Lead Time with this
data. What we can show is that our method (joining `sales` and `chcart` by
order number, and fixing the ".0" bug) is correct, if the two files were
from the same time period, this same code would give a real, trustworthy
result.

### KPI2 & KPI3 — Quality: Order Fill Rate and Stockout Line Rate

These two KPIs use the `sales` file: for each order line, we compare
`qty_ordered` (what the customer asked for) with `qty_shipped` (what we
actually sent).

- **Fill Rate**: what % of the ordered quantity was actually shipped, on
  average. 100% means every line was fully shipped.
- **Stockout Line Rate**: what % of lines were shipped with less than the
  full ordered amount (a partial or missing shipment).

**One small data issue we check first:** the file description says
`qty_shipped` should never be bigger than `qty_ordered`. We check this, if
it happens, it's a data error, and we cap the fill rate at 100% so these
rows don't inflate the result.

In [18]:
# Data check: shipped quantity should never exceed ordered quantity
over_shipped = (df_sales['qty_shipped'] > df_sales['qty_ordered']).sum()
print(f"Lines where qty_shipped > qty_ordered (data error, should be 0): {over_shipped}")

# KPI2: Order Fill Rate (capped at 100% per line)
df_sales['line_fill_rate'] = (df_sales['qty_shipped'] / df_sales['qty_ordered']).clip(upper=1.0)
kpi2_fill_rate = df_sales['line_fill_rate'].mean() * 100

# KPI3: Stockout Line Rate
df_sales['is_stockout'] = df_sales['qty_shipped'] < df_sales['qty_ordered']
kpi3_stockout_rate = df_sales['is_stockout'].mean() * 100

print(f"\nKPI2 - Order Fill Rate: {kpi2_fill_rate:.2f}%")
print(f"KPI3 - Stockout Line Rate: {kpi3_stockout_rate:.2f}%")

Lines where qty_shipped > qty_ordered (data error, should be 0): 5

KPI2 - Order Fill Rate: 93.31%
KPI3 - Stockout Line Rate: 7.18%


**KPI2 — Order Fill Rate: 93.31%.** On average, customers received 93.31% of what they ordered per line.
We found 5 lines where the shipped quantity was
higher than ordered(a data entry error, not a real over-shipment) and
capped these at 100% so they don't distort the average upward.

**KPI3 — Stockout Line Rate: 7.18%.** This means about 1 in every 14 order
lines was shipped with less than the full ordered quantity, most likely due
to the item being out of stock or short at the moment of picking.

Improvement opportunity: The 7.18% stockout rate causes the 93.31% fill rate. Reducing stockouts would move the fill rate closer to 100%. We can compare stockouts with the ABC classes from AP4 to see if they affect fast-moving (high-demand) or slow-moving (low-demand) products. This can show if the problem is mainly replenishment or forecasting.

In [19]:
# --- Cross-Analysis: KPI3 (Stockouts) x AP4 (ABC Classification) ---

# 1. Join ABC class back to df_sales
df_sales_abc = df_sales.merge(
    sku_activity[['sku', 'abc_class']],
    on='sku',
    how='left'
)

# 2. Group by ABC class and compute stockout metrics
stockout_by_abc = df_sales_abc.groupby('abc_class').agg(
    total_lines=('is_stockout', 'count'),
    stockout_lines=('is_stockout', 'sum'),
    stockout_rate=('is_stockout', 'mean')
).reset_index()

# 3. Format percentages
stockout_by_abc['stockout_rate'] = stockout_by_abc['stockout_rate'] * 100
stockout_by_abc['pct_of_total_stockouts'] = (
    stockout_by_abc['stockout_lines'] / stockout_by_abc['stockout_lines'].sum()
) * 100

print(stockout_by_abc.round(2).to_string(index=False))
save_table(stockout_by_abc, 'kpi3_stockout_by_abc')

abc_class  total_lines  stockout_lines  stockout_rate  pct_of_total_stockouts
        A      1391663           59718           4.29                   36.98
        B       558998           42786           7.65                   26.49
        C       298975           58984          19.73                   36.53
[saved] artifacts\kpi3_stockout_by_abc.csv


### Stockout Root Cause Analysis (KPI3 x AP4)

The stockout rate is very different by class: 4.3% of the Class A lines, 7.7% of Class B and 19.7% of Class C. Class A and Class C cause about the same share of all stockout lines (37% each). The data show **where** the stockouts happen, but not **why**. The explanations below are hypotheses, not results.

* **Class A items (37% of all stockouts)**
  *Hypothesis:* these popular items sell so fast that the picking shelves may run empty before workers can restock them from reserve storage.
  * *Possible fix:* increase reorder points so active picking shelves get restocked sooner. AS3 tests a related measure: more active locations for the fast SKUs, which means fewer refills.

* **Class C items (37% of all stockouts)**
  *Hypothesis:* the demand of these slow items is hard to predict, so stock levels may be kept too low.
  * *Possible fix:* update purchasing rules or have suppliers ship these items directly to customers.

*How to test the hypotheses:* compare stock on hand at the time of each stockout line with the min/max levels of the location. We did not do this.

### KPI4 & KPI5 — Productivity: Units Picked per Hour and Pick Hits per Hour

These two KPIs measure how much work gets done, using `trcart` (active
picks). We group all picks into 1-hour time buckets (using the date and hour
of `pick_datetime`), then look at the average per hour.

- **Units Picked per Hour (UPH)**: total quantity picked, per hour.
- **Pick Hits per Hour**: total number of pick lines, per hour (this is
  different from UPH, one pick line could be 1 unit or 50 units).

We only count hours that actually have picking activity (empty hours, like
overnight, are automatically excluded once we group by hour).
**Note:** UPH is the **total number of units picked by all active pickers per hour**, not the productivity of one picker.



In [20]:
df_trc_valid = df_trc.dropna(subset=['pick_datetime']).copy()

hourly_activity = (
    df_trc_valid.groupby(df_trc_valid['pick_datetime'].dt.floor('h'))
    .agg(total_units=('pick_qty', 'sum'), total_picks=('pick_qty', 'size'))
)

print(f"Number of active hour-buckets: {len(hourly_activity)}")
print(hourly_activity.describe())

kpi4_uph = hourly_activity['total_units'].mean()
kpi5_picks_per_hour = hourly_activity['total_picks'].mean()

print(f"\nKPI4 - Units Picked per Hour (UPH): {kpi4_uph:.1f}")
print(f"KPI5 - Pick Hits per Hour: {kpi5_picks_per_hour:.1f}")

Number of active hour-buckets: 172
        total_units  total_picks
count    172.000000   172.000000
mean    3709.412791   806.238372
std     4002.064590   848.375280
min        1.000000     1.000000
25%      110.500000    22.000000
50%     2955.500000   660.500000
75%     5070.250000  1114.250000
max    21569.000000  3998.000000

KPI4 - Units Picked per Hour (UPH): 3709.4
KPI5 - Pick Hits per Hour: 806.2


In [21]:
# --- Save the five KPIs in one table ---
import pandas as pd

kpi_table = pd.DataFrame([
    ('KPI1 Time', 'Median order lead time', kpi1_median_lead_days, 'days'),
    ('KPI2 Quality', 'Order fill rate', kpi2_fill_rate, '%'),
    ('KPI3 Quality', 'Stockout line rate', kpi3_stockout_rate, '%'),
    ('KPI4 Productivity', 'Units picked per hour', kpi4_uph, 'units/h'),
    ('KPI5 Productivity', 'Pick hits per hour', kpi5_picks_per_hour, 'picks/h'),
], columns=['KPI', 'Name', 'Value', 'Unit'])
save_table(kpi_table, 'kpi_summary')


[saved] artifacts\kpi_summary.csv


KPI4 & KPI5- These KPIs are based on **172 active hours** in `TRCART23` (Feb 25–Mar 25, 2005).

There is a big difference between hours. The median was only **2,955.5 units** and **660.5 picks**, which is lower than the average. A few very busy hours increase the overall average, with up to **21,569 units** and **3,998 picks** in one hour.

**Improvement opportunity:** A single average can be misleading for staffing. Orders are highest between **15:00–17:00**, so more workers should be scheduled during these busy hours instead of keeping the same staffing level throughout the shift.



## Key Takeaways & Action Plan

### 1. Warehouse Layout & Slotting

- **Traffic (trade-off):** Zone B is crowded with Class A picks, and moving some Class A items to Zone C would balance the traffic. But AS2 to AS5 show that putting the fast SKUs close together cuts walking and refill work, and raises the load of the busiest aisle (115 to 232 lines per day). The two goals conflict. AS5 weighs them.
- **Optimize placement:** Class A items generate **80% of shipped volume** and are mostly small. Place them on easy-to-reach shelves near packing. Move bulky, slow-moving items further away.
- **Improve reserve storage:** Check that Class A/B items are not in pallet-only reserve zones (F/W), so they can be replenished more easily.

### 2. Labor & Order Processing

- **Batch picking:** Over **60% of orders are single-line orders**. Grouping them into batches can reduce travel time.
- **Peak staffing:** Order activity is highest between **15:00–17:00**. Schedule more workers during these hours to match demand.

### 3. Inventory & Quality

- **Reduce stockouts:** The **7.18% stockout rate** leads to a **93.31% fill rate**. Stockouts affect both fast-moving Class A and slow-moving Class C items, but for different reasons.
- **Class A (hypothesis):** These items sell very quickly, so picking locations may run empty before they are replenished. **Increase reorder points and replenish them earlier.**
- **Class C (hypothesis):** These items have less predictable demand and may be unavailable when customers suddenly order them. **Review purchasing and safety stock rules, or consider supplier drop-shipping for some items.**
- **Review inventory levels:** Categories such as **Hazardous Materials** have high stock per SKU. Reviewing their min/max levels could free up storage space.

### 4. Data Validation

- **KPI1 limitation:** The **~322-day Order Lead Time** is not a real lead time because `sales` is from 2004 and `chcart` is from 2005.
- The **join and calculation method were correctly validated** and can be used with matching data from the same period.


## WH-AS2 — Storage Location Assignment (Zones B & C)
**1. Model Scope**

Before building the model, we need to decide which part of the warehouse we are working with. The full warehouse is too big to optimize all at once, so we focus on Zones B and C, which are the two zones this part of the project is about.

We take every active storage location in these two zones, and every item (SKU) that is currently stored in them. We also check that each location is counted only once, and that each location holds only one item, which is how this part of the warehouse is organized today.

One thing to note: this model only looks at **picking trips** (walking to grab an item for an order). We don't have real data on how often items get restocked from reserve storage, so restocking trips are not included. We'll mention this limitation again in the final recommendation.

In [22]:
# --- Which SKUs and locations are we working with? ---
SCOPE_ZONES = ['B', 'C']

# Keep only active locations in zones B and
active_scope = df_active[df_active['zone'].isin(SCOPE_ZONES)].copy()

# Make sure each physical location only appears once
active_scope = active_scope.drop_duplicates(subset=['zone', 'aisle', 'bay', 'level', 'position'])

# The SKUs that live in these locations
skus_in_scope = active_scope['sku'].unique()

print(f"Active locations in zones B/C: {len(active_scope):,}")
print(f"Distinct SKUs currently active in zones B/C: {len(skus_in_scope):,}")
print("\nLocations per zone:")
print(active_scope['zone'].value_counts())

# Quick check: does every location really hold just one SKU?
dupes = active_scope.groupby(['zone', 'aisle', 'bay', 'level', 'position'])['sku'].nunique()
print(f"\nLocations with more than one SKU (should be 0): {(dupes > 1).sum()}")

Active locations in zones B/C: 14,575


Distinct SKUs currently active in zones B/C: 14,575

Locations per zone:
zone
C    8961
B    5614
Name: count, dtype: int64

Locations with more than one SKU (should be 0): 0


We found 14,575 active locations across Zones B and C (5,614 in Zone B and 8,961 in Zone C), holding 14,575 different SKUs. There are no locations shared by more than one item, which confirms the current system follows a simple one-item-per-location rule.

This gives us the full size of the problem: this is how many items and locations we need to work with in the next steps.

**2. Distance Matrix**

To know which locations are close or far from the packing area, we need a coordinate for every location, and a way to measure distance.

We reuse the aisle coordinates we already built in AP1 (from the warehouse map), and give every location in Zones B and C the coordinate of its own aisle. The packing area is treated as one single point, where every picker starts and ends their trip. Distance is measured the simple way warehouses actually walk — along aisles, not in a straight line (this is called rectilinear, or "Manhattan," distance) — and converted from grid squares into meters.

In [23]:
# --- Reuse the aisle-level coordinates from AP1 ---
aisle_coords = bay_locations[bay_locations['zone'].isin(SCOPE_ZONES)][['zone', 'aisle', 'row', 'col']].copy()

# Fix: make sure both tables use the same type (text) for aisle, so they can be joined
active_scope['aisle'] = active_scope['aisle'].astype(int).astype(str).str.zfill(3)
aisle_coords['aisle'] = aisle_coords['aisle'].astype(str).str.zfill(3)

# Give every active location the coordinate of its aisle
loc_scope = active_scope.merge(aisle_coords, on=['zone', 'aisle'], how='left')

# How many locations have no coordinate? (not on the original map)
no_coord = loc_scope['row'].isna()
print(f"Active B/C locations with no map coordinate: {no_coord.sum():,} of {len(loc_scope):,} "
      f"({no_coord.mean():.1%}) -- these are left out of the model")

loc_scope = loc_scope.dropna(subset=['row', 'col']).copy()
loc_scope['row'] = loc_scope['row'].astype(int)
loc_scope['col'] = loc_scope['col'].astype(int)

print(f"Locations with a coordinate, in scope: {len(loc_scope):,}")

# --- I/O point: the packing area, one single point ---
GRID_TO_M = 42 * 0.0254  # 42-inch bay = 1 grid square, converted to meters

io_point = {
    'row': int(loc_scope['row'].median()),
    'col': loc_scope['col'].min(),   # leftmost column = packing side
}
print("\nI/O point grid position (packing area, assumed):", io_point)

# --- Distance from the I/O point to every location ---
loc_scope['dist_m'] = (
    (loc_scope['row'] - io_point['row']).abs() + (loc_scope['col'] - io_point['col']).abs()
) * GRID_TO_M

print("\nDistance summary (meters):")
print(loc_scope['dist_m'].describe().round(1))

Active B/C locations with no map coordinate: 0 of 14,575 (0.0%) -- these are left out of the model
Locations with a coordinate, in scope: 14,575

I/O point grid position (packing area, assumed): {'row': 7, 'col': np.int64(10)}

Distance summary (meters):
count    14575.0
mean       103.4
std         44.7
min          0.0
25%         71.5
50%        101.3
75%        138.7
max        187.8
Name: dist_m, dtype: float64


Every one of the 14,575 locations in Zones B and C has a coordinate on the map, so none had to be left out. On average, a location is about 103 meters from the packing area, with the closest locations right next to it (0 meters) and the farthest about 188 meters away.

This distance is what we use next to decide which locations are "good" (close, for popular items) and which are "bad" (far, for items that don't move much).

**3. Weights**

Now we need to know which SKUs matter most. The idea is simple: items that get picked often should get the best (closest) locations, and items that barely get picked can go further away.

To measure "how often," we count real picking trips for each SKU in Zones B and C, using only the active (not reserve) locations. This gives us $f_i$, the picking frequency for each item — the first piece of the weight formula.

In [24]:
# --- Frequency f_i: real picking trips per SKU in zones B/C, active locations only ---
bc_active_picks = df_cdc[
    (df_cdc['zone'].isin(SCOPE_ZONES)) &
    (df_cdc['pick_source'] == 'Active')
].copy()

freq = (
    bc_active_picks.groupby('sku')
    .size()
    .reset_index(name='f_i')
)

print(f"SKUs with at least 1 active B/C pick recorded: {len(freq):,}")
print(f"Of the {len(skus_in_scope):,} SKUs with a location in B/C, "
      f"{len(set(freq['sku']) & set(skus_in_scope)):,} have at least one recorded pick")

print("\nf_i distribution (picks per SKU):")
print(freq['f_i'].describe())

SKUs with at least 1 active B/C pick recorded: 9,475
Of the 14,575 SKUs with a location in B/C, 9,457 have at least one recorded pick

f_i distribution (picks per SKU):
count    9475.000000
mean        9.825224
std        18.809553
min         1.000000
25%         2.000000
50%         5.000000
75%        11.000000
max       478.000000
Name: f_i, dtype: float64


- 9,457 of 14,575 SKUs got picked at least once; about 5,100 never did.
- Picking is uneven: half of the active SKUs were picked 5 times or fewer, but the busiest SKU was picked 478 times.

This unevenness is why location assignment matters — a SKU picked 478 times shouldn't sit as far away as one picked only once.

**Why shrink the model**

SKUs with zero picks gain nothing from a better location — there's no activity to save time on. To keep the model solvable, we check how concentrated picking activity is among the SKUs that *were* picked, which tells us how many we actually need to optimize.

In [25]:
# Ties in f_i are broken by SKU code, so the 3,500 SKUs are the same on every computer.
# --- How concentrated is picking activity? (same idea as AS1's AP4) ---
freq_sorted = freq.sort_values(['f_i', 'sku'], ascending=[False, True]).reset_index(drop=True)
freq_sorted['cum_pct'] = freq_sorted['f_i'].cumsum() / freq_sorted['f_i'].sum() * 100

for pct in [50, 80, 90, 95]:
    n_needed = (freq_sorted['cum_pct'] <= pct).sum() + 1
    print(f"SKUs needed for {pct}% of B/C picking activity: {n_needed:,} of {len(freq_sorted):,} "
          f"({n_needed/len(freq_sorted):.1%})")

# How many total trips do we lose if we only model, say, the top 90%-of-activity SKUs?
n_90 = (freq_sorted['cum_pct'] <= 90).sum() + 1
print(f"\nIf we model only the top {n_90:,} SKUs (90% of activity), "
      f"we cover {freq_sorted.loc[:n_90-1, 'f_i'].sum():,} of {freq_sorted['f_i'].sum():,} total picks")

SKUs needed for 50% of B/C picking activity: 1,119 of 9,475 (11.8%)
SKUs needed for 80% of B/C picking activity: 3,500 of 9,475 (36.9%)
SKUs needed for 90% of B/C picking activity: 5,152 of 9,475 (54.4%)
SKUs needed for 95% of B/C picking activity: 6,500 of 9,475 (68.6%)

If we model only the top 5,152 SKUs (90% of activity), we cover 83,786 of 93,094 total picks


Picking activity is very concentrated: just 1,119 SKUs (11.8% of all SKUs with picks) already account for half of all picking trips. To reach 80% of all activity, we only need 3,500 SKUs — about a third of the SKUs that were ever picked.

This confirms the model can be made much smaller without losing much accuracy. We choose to keep the top 3,500 SKUs (80% of activity) in our model. This cuts the problem size a lot, while still covering the large majority of real picking trips — so the optimization result will reflect what actually happens in the warehouse, not a long tail of items that barely move.

**Final item scope**

Based on that, we now lock in the final scope: the top 3,500 SKUs by picking activity. These become the items (**I**) in our model, and the 14,575 active locations in Zones B and C stay as the full set of locations (**J**) they can be placed in.

In [26]:
# --- Final item scope: top 80%-of-activity SKUs ---
SCOPE_PCT = 80
n_scope_skus = (freq_sorted['cum_pct'] <= SCOPE_PCT).sum() + 1

item_scope = freq_sorted.iloc[:n_scope_skus].copy()
print(f"Final item scope: {len(item_scope):,} SKUs "
      f"({len(item_scope)/len(freq_sorted):.1%} of active SKUs, "
      f"{item_scope['f_i'].sum():,} of {freq_sorted['f_i'].sum():,} picks = "
      f"{item_scope['f_i'].sum()/freq_sorted['f_i'].sum():.1%} of activity)")

print("\nFinal f_i distribution (SKUs in scope):")
print(item_scope['f_i'].describe())

Final item scope: 3,500 SKUs (36.9% of active SKUs, 74,476 of 93,094 picks = 80.0% of activity)



Final f_i distribution (SKUs in scope):
count    3500.000000
mean       21.278857
std        27.275985
min         7.000000
25%        10.000000
50%        14.000000
75%        23.000000
max       478.000000
Name: f_i, dtype: float64


These 3,500 SKUs are only 36.9% of the SKUs that had any picks, but they cover 80% of all picking activity. Every SKU in this smaller group was picked at least 7 times, so we're only keeping items that genuinely matter for travel distance.

This is the final group of items we optimize the location of in the next steps.

**Space requirement (S_i)**

Each SKU doesn't just need a location — it needs enough space for its stock. We don't have the exact size of each location in the data, so we estimate it: since almost every SKU today fits in exactly one location, we treat a "typical" location's capacity as the 75th percentile of all `Max` stock values. Any SKU needing much more space than that gets more than one location.

This is a simplification — a more precise model would use real shelf sizes — but it's a reasonable estimate given what we have.

In [27]:
# --- Space requirement S_i, from DC23MINMAX2's Max stock level ---
minmax = raw['minmax'][['SKU', 'Max']].rename(columns={'SKU': 'sku'}).drop_duplicates(subset='sku').copy()

item_scope = item_scope.merge(minmax, on='sku', how='left')

missing_max = item_scope['Max'].isna()
print(f"SKUs in scope with no Max value found: {missing_max.sum():,} of {len(item_scope):,}")

# For any SKU with no Max on file, fall back to the smallest typical requirement (1 location)
item_scope['Max'] = item_scope['Max'].fillna(item_scope['Max'].median())

# "Typical" single-location capacity, estimated from the data itself (see markdown above)
LOCATION_CAPACITY = item_scope['Max'].quantile(0.75)
print(f"Assumed single-location capacity: {LOCATION_CAPACITY:.0f} units")

item_scope['S_i'] = np.ceil(item_scope['Max'] / LOCATION_CAPACITY).clip(lower=1).astype(int)

print(f"\nSKUs needing more than 1 location: {(item_scope['S_i'] > 1).sum():,} of {len(item_scope):,}")
print("\nS_i distribution:")
print(item_scope['S_i'].describe())
print(f"\nTotal locations needed for all {len(item_scope):,} SKUs in scope: {item_scope['S_i'].sum():,}")

SKUs in scope with no Max value found: 12 of 3,500
Assumed single-location capacity: 240 units

SKUs needing more than 1 location: 838 of 3,500

S_i distribution:
count     3500.000000
mean        14.735429
std        491.388200
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max      25001.000000
Name: S_i, dtype: float64

Total locations needed for all 3,500 SKUs in scope: 51,574


Something looks wrong here: the model says we'd need 51,574 locations in total — far more than the 14,575 we actually have. One SKU alone would need over 25,000 locations.

This is a sign of a data problem, not a modeling problem. We check it in the next step.

**Checking the data**

Before fixing anything, we check what's actually causing this. We look at the full distribution of `Max` values, and specifically at the SKUs with the highest values, to see if they make sense.

In [28]:
# --- Diagnostic: is 'Max' reasonable, or do we have data outliers? ---
print("Max value distribution (raw, before capping):")
print(item_scope['Max'].describe())

print("\nTop 10 SKUs by Max value:")
print(item_scope[['sku', 'Max', 'f_i']].sort_values('Max', ascending=False).head(10))

print("\nHow many SKUs have Max > 10,000?", (item_scope['Max'] > 10000).sum())
print("How many SKUs have Max > 1,000?", (item_scope['Max'] > 1000).sum())

Max value distribution (raw, before capping):
count    3.500000e+03
mean     3.393657e+03
std      1.179339e+05
min      3.000000e+00
25%      5.000000e+01
50%      1.000000e+02
75%      2.400000e+02
max      6.000080e+06
Name: Max, dtype: float64

Top 10 SKUs by Max value:
                  sku        Max  f_i
2677         SWI40701  6000080.0   10
1063         COX22236  2520432.0   20
596          COX13387  2520156.0   28
77          MMM654-YW    18312.0   85
12         MMM6549-YW     7000.0  198
29    MMM810-3/4X1296     5000.0  140
8            SPR19783     4320.0  218
649        MMM6569-YW     4000.0   27
14    MMM62003/4X1296     4000.0  173
3329         SPR02072     3600.0    8

How many SKUs have Max > 10,000? 4
How many SKUs have Max > 1,000? 139


We found the problem: 3 SKUs have `Max` values in the millions (one is 6,000,080 units), but they only have 10–28 recorded picks. That doesn't make sense — a SKU barely picked at all wouldn't realistically need millions of units of stock.

These look like data entry errors, not real stock targets. Of the 4 SKUs with `Max` above 10,000, only one (`MMM654-YW`, 18,312 units, 85 picks) looks plausible. The other 135 SKUs with `Max` above 1,000 also have many more picks, so they look normal. We fix the broken values in the next step.

**Fixing the outliers**

To stop these broken values from distorting the model, we cap `Max` at its 99th percentile — the same approach we used in AS1 for extreme outliers. Any SKU above that value gets capped down to it; everyone else stays the same. Then we recalculate the space requirement with the cleaned values.

In [29]:
# --- Cap extreme Max values (data quality control, same approach as AS1) ---
cap_value = item_scope['Max'].quantile(0.99)
n_capped = (item_scope['Max'] > cap_value).sum()
print(f"Capping Max at the 99th percentile: {cap_value:.0f} units")
print(f"SKUs affected by the cap: {n_capped} of {len(item_scope):,}")

item_scope['Max_capped'] = item_scope['Max'].clip(upper=cap_value)

# Re-estimate location capacity and S_i using the capped values
LOCATION_CAPACITY = item_scope['Max_capped'].quantile(0.75)
print(f"\nAssumed single-location capacity: {LOCATION_CAPACITY:.0f} units")

item_scope['S_i'] = np.ceil(item_scope['Max_capped'] / LOCATION_CAPACITY).clip(lower=1).astype(int)

print(f"\nSKUs needing more than 1 location: {(item_scope['S_i'] > 1).sum():,} of {len(item_scope):,}")
print("\nS_i distribution:")
print(item_scope['S_i'].describe())
print(f"\nTotal locations needed for all {len(item_scope):,} SKUs in scope: {item_scope['S_i'].sum():,}")
print(f"Physical locations available in B/C: {len(loc_scope):,}")

Capping Max at the 99th percentile: 2200 units


SKUs affected by the cap: 34 of 3,500

Assumed single-location capacity: 240 units

SKUs needing more than 1 location: 838 of 3,500

S_i distribution:
count    3500.000000
mean        1.552857
std         1.414943
min         1.000000
25%         1.000000
50%         1.000000
75%         1.000000
max        10.000000
Name: S_i, dtype: float64

Total locations needed for all 3,500 SKUs in scope: 5,435
Physical locations available in B/C: 14,575


After capping, the numbers make sense: the model now says we need 5,435 locations in total, well within the 14,575 we actually have. Only 34 SKUs were affected by the cap, and the biggest space requirement for any single SKU is now 10 locations, not 25,000.

This confirms the space requirement step is working correctly, and we have more than enough room in Zones B and C to fit the plan.

**Handling cost (c_i)**

The last piece we need is the handling cost — how expensive it is to move one unit of a SKU. We don't have real cost data per SKU, so we use a simple, honest assumption: every SKU costs the same to handle ($c_i = 1$). This keeps the model focused on what we *do* know well: how often each SKU is picked, and how far each location is.

In [30]:
# --- Handling cost c_i: uniform (see markdown for why) ---
item_scope['c_i'] = 1

print("c_i set to 1 for all SKUs in scope (uniform handling cost).")
print(f"\nFinal item_scope table ready with: f_i, Max, S_i, c_i for {len(item_scope):,} SKUs")
print(item_scope[['sku', 'f_i', 'Max_capped', 'S_i', 'c_i']].head(10))

c_i set to 1 for all SKUs in scope (uniform handling cost).

Final item_scope table ready with: f_i, Max, S_i, c_i for 3,500 SKUs
            sku  f_i  Max_capped  S_i  c_i
0      SPR87002  478      2200.0   10    1
1      SPR85001  459      2200.0   10    1
2      SPR85009  440      1600.0    7    1
3      SPR87005  436      1200.0    5    1
4      SPRHB210  422      1500.0    7    1
5  SPRSP111-1/3  394       300.0    2    1
6    HEWC6656AN  270       504.0    3    1
7     HEW51645A  246       500.0    3    1
8      SPR19783  218      2200.0   10    1
9      SPRW2011  213       216.0    1    1


With this, every piece of the weight formula is ready: $f_i$ (how often it's picked), $S_i$ (how much space it needs), $c_i$ (handling cost, the same for everyone), and $d_j$ (distance, from Step 2). We now have everything we need to build the actual optimization model.

**4. Mathematical Model**

Now we put everything into a formal model.

**Sets**
- $I$: the 3,500 SKUs in scope
- $J$: the 14,575 active locations in zones B/C

**Parameters**
- $f_i$: picking trips for item $i$ (from `cdcart23_clean`)
- $c_i = 1$: handling cost (uniform, stated assumption)
- $S_i$: locations needed for item $i$ (from `DC23MINMAX2`, capped)
- $d_j$: distance from the packing area to location $j$ (meters)
- $w_{ij} = \dfrac{f_i \cdot c_i \cdot d_j}{S_i}$: cost of placing one unit of item $i$'s stock at location $j$

**Decision variable**
- $x_{ij} \in \{0,1\}$: 1 if item $i$ is assigned to location $j$

**Objective** — minimize total weighted travel cost:
$$\min \sum_{i \in I} \sum_{j \in J} w_{ij} \, x_{ij}$$

**Constraints**
- $\sum_{j \in J} x_{ij} = S_i \quad \forall i$ (each item gets exactly as many locations as it needs)
- $\sum_{i \in I} x_{ij} \le 1 \quad \forall j$ (each location holds at most one item — the compatibility rule from Step 1)
- $x_{ij} \in \{0,1\}$

**How we solve it.** This problem is too large for a generic solver to search freely (3,500 × 14,575 possible pairs). But our cost has a special shape: $w_{ij} = r_i \cdot d_j$, where $r_i = f_i c_i / S_i$ depends only on the item and $d_j$ only on the location. For costs that multiply apart like this, the optimal solution is always: **give the items with the highest $r_i$ the closest locations, in order** — similar to how you'd pack a truck, putting the most urgent boxes closest to the door. So instead of a slow generic solver, we sort both lists and match them directly — this gives the exact same optimal answer, much faster.

In [31]:
# --- Cost of the current assignment (what SPR does today) ---
current_dist = loc_scope[['sku', 'dist_m']].drop_duplicates(subset='sku')
item_current = item_scope.merge(current_dist, on='sku', how='left')

missing_current = item_current['dist_m'].isna().sum()
print(f"SKUs in scope with no current location found: {missing_current} of {len(item_current):,}")
item_current = item_current.dropna(subset=['dist_m'])

current_cost = (item_current['f_i'] * item_current['c_i'] * item_current['dist_m']).sum()
print(f"\nCurrent assignment cost: {current_cost:,.0f} (picks x meters)")

# --- Optimal assignment, via sorting (see markdown for why this is optimal) ---
item_opt = item_scope.copy()
item_opt['r_i'] = item_opt['f_i'] * item_opt['c_i'] / item_opt['S_i']
item_opt = item_opt.sort_values(['r_i', 'sku'], ascending=[False, True]).reset_index(drop=True)

sorted_dist = np.sort(loc_scope['dist_m'].values)
total_locations_needed = item_opt['S_i'].sum()
print(f"\nLocations needed: {total_locations_needed:,} of {len(sorted_dist):,} available")

proposed_cost = 0.0
pointer = 0
for _, row in item_opt.iterrows():
    s = int(row['S_i'])
    assigned_d = sorted_dist[pointer:pointer + s]
    proposed_cost += row['r_i'] * assigned_d.sum()
    pointer += s

print(f"Proposed (optimal) assignment cost: {proposed_cost:,.0f} (picks x meters)")

improvement = (current_cost - proposed_cost) / current_cost * 100
print(f"\nImprovement: {improvement:.1f}% lower total travel cost")
save_table(item_opt, 'as2_assignment')

SKUs in scope with no current location found: 3 of 3,500

Current assignment cost: 4,463,039 (picks x meters)

Locations needed: 5,435 of 14,575 available
Proposed (optimal) assignment cost: 2,726,257 (picks x meters)

Improvement: 38.9% lower total travel cost
[saved] artifacts\as2_assignment.csv


The current setup costs 4,463,039 (picks × meters) in the model. The optimal setup, using the same 5,435 locations, costs 2,726,257. That is **38.9% less**.

**How to read this number.** It is the value of the model objective. The model counts every pick line as a separate one-way trip from the packing area. It does not include that a picker collects many lines in one batch tour. So the number shows that the proposed assignment puts the busy SKUs much closer to the packing area, but it is **not the real walking saving**. With the real batches, AS4 finds 17% less walking when the new storage is combined with a better route, because a tour is as long as its farthest stop.

**5. Layout Comparison**

Numbers alone don't show *where* the improvement comes from, so we draw a picture of the warehouse. Each location is colored by the "priority" of the SKU sitting there — how often it's picked, relative to how much space it needs. We draw this twice: once for today's assignment, and once for the proposed one, so we can see whether high-priority items actually move closer to the packing area.

In [32]:
# --- Rebuild the proposed assignment, this time keeping WHICH location each SKU gets ---
loc_sorted = loc_scope.sort_values(['dist_m', 'zone', 'aisle', 'bay', 'level', 'position']).reset_index(drop=True)
loc_sorted['assigned_sku'] = None

pointer = 0
for _, row in item_opt.iterrows():
    s = int(row['S_i'])
    loc_sorted.loc[pointer:pointer + s - 1, 'assigned_sku'] = row['sku']
    pointer += s

# Map each assigned location back to its item's priority (r_i), for coloring
r_i_map = item_opt.set_index('sku')['r_i']
loc_sorted['r_i_proposed'] = loc_sorted['assigned_sku'].map(r_i_map)

# Current assignment: each location's priority, using today's sku-to-location mapping
loc_scope['r_i_current'] = loc_scope['sku'].map(r_i_map)  # NaN for SKUs outside our 3,500-item scope

# --- Plot: current vs. proposed, side by side ---
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

for ax, data, col, title in [
    (axes[0], loc_scope, 'r_i_current', 'Current assignment'),
    (axes[1], loc_sorted, 'r_i_proposed', 'Proposed (optimal) assignment'),
]:
    plotted = data.dropna(subset=[col])
    sc_plot = ax.scatter(plotted['col'], plotted['row'], c=plotted[col],
                     cmap='YlOrRd', s=25, marker='s')
    ax.invert_yaxis()
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Warehouse layout — column (packing area on the left)')
    ax.set_ylabel('Row')
    plt.colorbar(sc_plot, ax=ax, label='Item priority (r_i = picks / locations needed)')

plt.tight_layout()
save_fig('as2_layout_comparison')

print(f"Locations shown, current: {loc_scope['r_i_current'].notna().sum():,}")
print(f"Locations shown, proposed: {loc_sorted['r_i_proposed'].notna().sum():,}")

[saved] artifacts\as2_layout_comparison.png


[saved] artifacts\as2_layout_comparison.pdf
Locations shown, current: 3,497
Locations shown, proposed: 5,435


C:\Users\joneo\AppData\Local\Temp\ipykernel_15512\2522381755.py:62: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


The picture confirms what the numbers already told us: in the current layout, high-priority SKUs are scattered across the whole width of Zones B and C. In the proposed layout, those same high-priority SKUs are pulled in much closer to the packing area, while low-priority SKUs get pushed further out.

The proposed plan uses more locations overall (5,435 vs. today's 3,497), because several fast-moving SKUs need more than one location to hold their stock — something the current setup doesn't account for. Even with more locations in use, the model travel cost still drops by 38.9%, because the right items are now in the right spots.

**6. Storage Policy and Feasibility**

Having a good plan on paper isn't enough — we need to check it actually works in practice. This section does that in five parts: how locations are assigned, and then four feasibility checks (capacity, physical fit, replenishment, and service risk).

**6A. Storage Policy**

Each SKU's location is decided by one number: its priority ratio $r_i = f_i \cdot c_i / S_i$ — how often it's picked, divided by how much space it needs. SKUs are ranked by this number and matched to locations in distance order: the SKU with the highest priority gets the closest spot, and so on down the list.

If a new SKU is added later, or an existing one's demand changes, we just recompute its $r_i$ and slot it into its new position — no need to redo the whole plan. We recommend recalculating the full assignment every quarter, or right after a big demand shift, since picking activity can become concentrated differently over time.

**6B. Capacity Check**

This check asks: does the space we gave each SKU actually cover how much stock it needs to hold? Since we built $S_i$ specifically so that $S_i$ locations would cover each SKU's `Max` stock level, this should hold true — we just confirm it here.

In [33]:
# --- Capacity check: does the assigned space actually cover each SKU's Max? ---
item_opt['space_assigned'] = item_opt['S_i'] * LOCATION_CAPACITY
item_opt['capacity_ok'] = item_opt['space_assigned'] >= item_opt['Max_capped']

print(f"SKUs where assigned space covers their Max requirement: "
      f"{item_opt['capacity_ok'].sum():,} of {len(item_opt):,} "
      f"({item_opt['capacity_ok'].mean():.1%})")

shortfall = item_opt[~item_opt['capacity_ok']]
if len(shortfall) > 0:
    print(f"\nSKUs with a shortfall (should be none or near-none):")
    print(shortfall[['sku', 'Max_capped', 'space_assigned']].head(10))

SKUs where assigned space covers their Max requirement: 3,500 of 3,500 (100.0%)


Every single SKU (3,500 of 3,500) has enough assigned space to cover its stock requirement. This makes sense, since we built the space requirement this way from the start — but it's good to confirm nothing broke along the way.

**6C. Compatibility Check**

This check asks: does the product actually fit in the shelf space it was given? Each item has real dimensions (length, width, height), and each location has a fixed shelf depth. If an item is bigger than the shelf depth, it doesn't physically fit, even if there's "enough space" on paper.

We compare each item's largest dimension to the shelf depth. If it's smaller, it fits — if not, we flag it as needing a different kind of spot, like a deeper shelf or a pallet position.

In [34]:

# Assumption: shelf depth is 30 inches in both Zone B and Zone C (standard shelf bay depth
# used across the DC for these zones, based on the BCF-zones layout). If your BCF legend
# shows a different number for one of the zones, just change LOCATION_DEPTH_IN below.
LOCATION_DEPTH_IN = 30

dims = raw['dims'].copy()
dims.columns = [c.strip() for c in dims.columns]
dims = dims.rename(columns={'SKU': 'sku'})

# Keep only the dimension columns we need
dims_small = dims[['sku', 'SellLen', 'SellWid', 'SellHgt']].copy()

# Merge item dimensions onto our optimized item table
item_opt = item_opt.merge(dims_small, on='sku', how='left')

# An item's "footprint depth" is the largest of its three dimensions
item_opt['max_dim_in'] = item_opt[['SellLen', 'SellWid', 'SellHgt']].max(axis=1)

# Compatibility: does the item fit within the location depth?
item_opt['compat_ok'] = item_opt['max_dim_in'] <= LOCATION_DEPTH_IN

n_missing = item_opt['max_dim_in'].isna().sum()
n_checked = item_opt['max_dim_in'].notna().sum()
n_ok = item_opt.loc[item_opt['max_dim_in'].notna(), 'compat_ok'].sum()

print(f"Items with dimension data: {n_checked:,} (missing: {n_missing:,})")
print(f"Items that fit the {LOCATION_DEPTH_IN}-inch location depth: "
      f"{n_ok:,} of {n_checked:,} ({n_ok/n_checked:.1%})")

Items with dimension data: 3,488 (missing: 12)
Items that fit the 30-inch location depth: 3,474 of 3,488 (99.6%)


Almost every item (99.6%) fits in the shelf space it was given. Only 14 items out of 3,488 are too big for a standard 30-inch shelf — probably bulky products that need a different kind of storage, like a pallet spot. We also had 12 items with no size data, so we couldn't check those.

This confirms the plan works in practice, not just on paper. The small number of exceptions is worth a one-line mention in the report, but it doesn't change the overall result.

**6D. Replenishment Check**

This check asks: will staff need to refill this location too often? We measure how many **days** the assigned space lasts at the average demand of the 30 days of data. We use **units picked** (the same unit as the space, and the same as in AS3), not the number of pick lines.

Days of stock = 30 × assigned space (units) ÷ units picked in 30 days. A small number means many refills. We flag a SKU if the space lasts less than 1 day, which means more than one refill per day.

In [35]:
# --- 6D. Replenishment: how many days does the assigned space last? (units picked in 30 days, same unit as AS3) ---
units = bc_active_picks.groupby('sku')['units_packed'].sum().reset_index(name='u_i')
item_opt = item_opt.drop(columns=['u_i'], errors='ignore').merge(units, on='sku', how='left')
item_opt['u_i'] = item_opt['u_i'].fillna(0)

item_opt['refills_30d'] = item_opt['u_i'] / item_opt['space_assigned']                            # refills in 30 days (average demand)
item_opt['days_of_stock'] = 30 * item_opt['space_assigned'] / item_opt['u_i'].replace(0, np.nan)  # days the space lasts
item_opt['frequent_replenishment'] = item_opt['days_of_stock'] < 1     # more than one refill per day on average

print(f"Median days of stock in the assigned space: {item_opt['days_of_stock'].median():.1f}")
print(f"Median refills per SKU in 30 days: {item_opt['refills_30d'].median():.1f}")
print(f"Total refills in 30 days for the {len(item_opt):,} SKUs: {item_opt['refills_30d'].sum():,.0f} "
      f"(about {item_opt['refills_30d'].sum() / 30:,.0f} per day)")
print(f"SKUs needing more than one refill per day (days of stock < 1): "
      f"{item_opt['frequent_replenishment'].sum():,} ({item_opt['frequent_replenishment'].mean():.1%})")
save_table(item_opt[['sku', 'u_i', 'S_i', 'space_assigned', 'refills_30d', 'days_of_stock']], 'as2_replenishment_check')

Median days of stock in the assigned space: 218.2
Median refills per SKU in 30 days: 0.1
Total refills in 30 days for the 3,500 SKUs: 744 (about 25 per day)
SKUs needing more than one refill per day (days of stock < 1): 0 (0.0%)
[saved] artifacts\as2_replenishment_check.csv


**Results (replenishment).**

- **No SKU needs more than one refill per day.** The space assigned by AS2 lasts a median of **218 days** at the average demand of the 30 days of data, and the median SKU needs only 0.1 refills in 30 days. This check now uses units picked, the same unit as the space. In the previous version it divided units by pick lines, so that result was not valid.
- **The fast SKUs are the ones that need refills.** The 3,500 SKUs together need about **744 refills in 30 days (25 per day)**, and most of them come from a small group of fast SKUs. This is the same problem that AS3 studies, so AS3 adds extra locations for the fast SKUs to reduce these trips.
- **Limits.** The estimate assumes average demand and that a refill brings the location back to full. It counts refills of the 3,500 SKUs of the scope (with the space assigned by AS2, 240 units per location), so it should not be compared directly with the 881 trips of AS3, which are for the 838 SKUs that need more than one location.

**6E. Service Check**

This check asks: could a location run out of stock in the middle of a picking wave, before anyone has a chance to refill it? We already know most items have plenty of stock coverage, but averages can hide risk for a few very fast-moving items. So here we look closely at the items with the lowest coverage — the ones closest to running empty — to make sure none of them are dangerously low.

In [36]:
# --- 6E. Service: the SKUs whose assigned space lasts the fewest days ---
riskiest = item_opt.nsmallest(10, 'days_of_stock')[['sku', 'u_i', 'S_i', 'space_assigned', 'days_of_stock']]

print("10 SKUs with the fewest days of stock (highest risk of running empty):")
print(riskiest.round(1).to_string(index=False))

print(f"\nFewest days of stock overall: {item_opt['days_of_stock'].min():.1f}")
print(f"SKUs with less than 3 days of stock (worth watching): "
      f"{(item_opt['days_of_stock'] < 3).sum():,} ({(item_opt['days_of_stock'] < 3).mean():.1%})")
save_table(riskiest, 'as2_service_check')

10 SKUs with the fewest days of stock (highest risk of running empty):
     sku    u_i  S_i  space_assigned  days_of_stock
SPR01821  701.0    1           240.0           10.3
SPR87005 3471.0    5          1200.0           10.4
SPR01817 1347.0    2           480.0           10.7
SPR05711  608.0    1           240.0           11.8
BSP44104  585.0    1           240.0           12.3
SPR85009 4075.0    7          1680.0           12.4
SPR87002 5344.0   10          2400.0           13.5
SPR85001 5050.0   10          2400.0           14.3
SPR05731  428.0    1           240.0           16.8
CCS24305  428.0    1           240.0           16.8

Fewest days of stock overall: 10.3
SKUs with less than 3 days of stock (worth watching): 0 (0.0%)
[saved] artifacts\as2_service_check.csv


**Results (service).**

- **No SKU is close to running empty.** The SKUs with the fewest days of stock still have at least **10.3 days** of space, and no SKU has less than 3 days. At average demand the riskiest locations need a refill about every 10 to 17 days.
- **They are the fast SKUs.** The list is led by `SPR01821`, `SPR87005` and `SPR01817`, followed by `SPR05711` and `BSP44104`, and then the fast SKUs `SPR85009`, `SPR87002` and `SPR85001`. The last three are the ones that AS3 gives up to 10 locations. For `SPR87002` the result (13.5 days, about 2.2 refills in 30 days) is close to the AS3 value for the proposed layout (2.4 trips): AS2 gives it 10 locations of 240 units (2,400 units), and AS3 uses its `Max` of 2,200 units.
- **What to do.** Staff should check these 10 SKUs first and refill them a little ahead of the min level.
- **Limit.** The check uses average demand. A busy day can empty a location sooner, and we did not test peak days.

**7. Management KPIs**

Here we turn the technical results into numbers that make sense to a manager, not just an analyst.

1. **Travel distance saved** — how much less the model travel cost is for the new plan, compared to today (every pick counted as a separate trip, see above).
2. **Space used** — how many of the available locations in Zones B and C the new plan actually uses.
3. **Model travel time saved** — the same saving written in hours at 1 m/s. It is **not** a real labour saving, because it assumes one trip per pick line. The real labour saving is measured in AS4 and AS5.

In [37]:


# 1. Travel distance saved (%)
travel_saved_pct = (current_cost - proposed_cost) / current_cost

# 2. Space utilization: locations used vs locations available in scope
locations_used = item_opt['S_i'].sum()
locations_available = len(active_scope)  # 14,575 from Step 1
space_utilization_pct = locations_used / locations_available

# 3. Rough labor time saved: assume average travel speed of 1 meter/second
#    (typical walking pace with a cart in a warehouse aisle)
AVG_SPEED_M_PER_S = 1.0
time_saved_seconds = (current_cost - proposed_cost) / AVG_SPEED_M_PER_S
time_saved_hours = time_saved_seconds / 3600

print(f"1. Travel distance saved: {travel_saved_pct:.1%}")
print(f"2. Locations used: {locations_used:,.0f} of {locations_available:,} available ({space_utilization_pct:.1%})")
print(f"3. Model travel time saved (one trip per pick line, 1 m/s): {time_saved_hours:,.1f} hours  [not a real labour saving, see AS4 and AS5]")

1. Travel distance saved: 38.9%
2. Locations used: 5,435 of 14,575 available (37.3%)
3. Model travel time saved (one trip per pick line, 1 m/s): 482.4 hours  [not a real labour saving, see AS4 and AS5]


In [38]:
# --- Save the AS2 management KPIs ---
as2_kpis = pd.DataFrame([
    ('Current travel cost', current_cost, 'picks x m'),
    ('Proposed travel cost', proposed_cost, 'picks x m'),
    ('Travel distance saved', travel_saved_pct * 100, '%'),
    ('Locations used', locations_used, 'locations'),
    ('Locations available', locations_available, 'locations'),
    ('Space utilization', space_utilization_pct * 100, '%'),
    ('Model travel time saved (not a real labour saving)', time_saved_hours, 'hours'),
], columns=['KPI', 'Value', 'Unit'])
save_table(as2_kpis, 'as2_management_kpis')


[saved] artifacts\as2_management_kpis.csv


Moving to the new storage plan cuts the model travel cost by 38.9%. That is a big improvement for a change that needs no new equipment and no building changes.

The plan only needs 37.3% of the available locations in Zones B and C, leaving plenty of room for inventory growth or new items.

The 482 hours are the same saving written in hours, but they assume one trip per pick line. That is more than all the real walking time of the baseline (282 h in 22 days, AS4), so it cannot be a real saving. With the real batch routes, AS4 and AS5 find 82 to 90 hours less in total (9% to 10% of the labour time), after adding the better route and the new layout.

**Summary — WH-AS2**

We looked at Zones B and C (14,575 locations, 14,575 SKUs) and reduced the scope to the top 3,500 SKUs by activity (80% of total picks), because the full problem was too large to solve directly.

We built a model that assigns each item to the number of locations it needs, based on how often it's picked, how costly it is to handle, and how close the location is to the shipping point. Because the cost formula splits neatly into an item part and a location part, we solved it by simply sorting and matching — no complex solver needed.

The new plan:
- Cuts the model travel cost by 38.9% (every pick counted as a separate trip). With the real batch routes, AS4 and AS5 find 17% less walking when the plan is combined with a better route.
- Uses only 37.3% of available space, leaving room to grow
- Passes the capacity (100%) and physical fit (99.6%) checks
- Passes the replenishment check (no SKU needs more than one refill per day; the space lasts a median of 218 days) and the service check (the fastest SKUs still have at least 10 days of stock)

The storage reassignment is a strong improvement in the model. Its real effect is smaller, and it only works well together with a route that follows distance (AS4, AS5).

# WH-AS3 — Warehouse Layout Design (Zones B & C)

## **1. Layout Problem and Scope**

**1.1 The problem.** In AS2 we saw that today every SKU has exactly one active picking location. But some SKUs need more space than that to hold their stock. For these SKUs, the single location runs out fast, and staff must bring more stock from reserve storage again and again.

We saw the same signal in AS1: Class A items cause 37% of all stockouts, because their pick shelves run empty before staff can refill them.

**1.2 The question.** Should we give these fast SKUs more active locations?

**1.3 The scope.**
- Area: Zones B and C.
- Products: the SKUs that need more than one location (the ones with $S_i > 1$, where $S_i$ is the number of locations a SKU needs).
- Period: 30 days of picking data (25 February to 26 March 2005).

First, we select these SKUs and measure how much extra space they need.

In [39]:
# --- Scope for AS3: SKUs that need more than 1 active location ---
gap = item_opt[item_opt['S_i'] > 1].copy()

# Today every SKU has exactly 1 active location (confirmed in AS2, Step 1)
gap['current_locations'] = 1
gap['extra_locations'] = gap['S_i'] - gap['current_locations']

total_picks_scope = item_opt['f_i'].sum()

print(f"SKUs needing more than 1 active location: {len(gap):,} of {len(item_opt):,}")
print(f"Extra active locations needed in total: {gap['extra_locations'].sum():,}")
print(f"Picks covered by these SKUs: {gap['f_i'].sum():,} of {total_picks_scope:,} "
      f"({gap['f_i'].sum()/total_picks_scope:.1%})")

print("\nExtra locations per SKU:")
print(gap['extra_locations'].describe().round(2))

print(f"\nTotal active locations in B/C: {len(loc_scope):,}")
print(f"Locations used by the AS2 plan (top 3,500 SKUs): {item_opt['S_i'].sum():,}")
print(f"Locations currently held by SKUs outside our 3,500: "
      f"{len(loc_scope) - len(item_opt):,}")

SKUs needing more than 1 active location: 838 of 3,500
Extra active locations needed in total: 1,935
Picks covered by these SKUs: 27,036 of 74,476 (36.3%)

Extra locations per SKU:
count    838.00
mean       2.31
std        2.08
min        1.00
25%        1.00
50%        1.00
75%        2.75
max        9.00
Name: extra_locations, dtype: float64

Total active locations in B/C: 14,575
Locations used by the AS2 plan (top 3,500 SKUs): 5,435
Locations currently held by SKUs outside our 3,500: 11,075


We found 838 SKUs (24% of our 3,500) that need more than one active location. Together they need 1,935 extra locations. Most of them need just 1 extra spot (the middle value is 1), but a few busy SKUs need up to 9 extra.

These SKUs are important: they cause 36.3% of all picks in our scope (27,036 of 74,476). So the space problem hits a big part of the daily work.

Where could the extra space come from? Today, 11,075 locations in Zones B and C are held by SKUs outside our top 3,500. These are low-activity items. The 1,935 extra locations we need are a small part of that space (about 17%).

## **2. Space and Handling Requirements**

In this part we turn the numbers into floor space and workload.

**2.1 Bay height and locations per bay.**
First, a quick data check: how tall are the bays, and how many locations does a bay hold? We need this to turn locations into floor space.

In [40]:
# --- How many levels does a bay have in zones B/C? ---
print(active_scope['level'].value_counts().sort_index())
print("\nLocations per bay (average):")
per_bay = active_scope.groupby(['zone', 'aisle', 'bay']).size()
print(per_bay.describe().round(1))

level
1.0    1954
2.0    3068
3.0    3140
4.0    2687
5.0    1965
6.0    1139
7.0     484
8.0      93
9.0      45
Name: count, dtype: int64

Locations per bay (average):
count    1829.0
mean        8.0
std         4.6
min         1.0
25%         5.0
50%         7.0
75%        10.0
max        41.0
dtype: float64


Most locations are on levels 2 to 4, and bays have up to 9 levels. A bay holds 8 locations on average (the middle value is 7). We use 8 to convert locations into floor space.

**2.2 Floor space.** Each bay is 42 inches wide, and shelves are 30 inches deep (the same values we used in AS2). One bay stacks about 8 locations on top of each other. So the floor space of one location is the floor area of a bay divided by 8. With this we turn the 1,935 extra locations into square meters.

In [41]:
# --- Floor space of locations (storage footprint) ---
BAY_WIDTH_M = 42 * 0.0254   # 42-inch bay
BAY_DEPTH_M = 30 * 0.0254   # 30-inch shelf depth (same as AS2)
BAY_AREA_M2 = BAY_WIDTH_M * BAY_DEPTH_M

locs_per_bay = per_bay.mean()                  # average locations per bay (levels x positions)
area_per_location = BAY_AREA_M2 / locs_per_bay

extra_locs = gap['extra_locations'].sum()
extra_area = extra_locs * area_per_location
total_area = len(active_scope) * area_per_location
outside_area = (len(loc_scope) - len(item_opt)) * area_per_location

print(f"Floor area of one bay: {BAY_AREA_M2:.2f} m2")
print(f"Average locations per bay: {locs_per_bay:.1f}")
print(f"Floor area per location: {area_per_location:.3f} m2")
print(f"\nExtra locations needed: {extra_locs:,} -> {extra_area:,.0f} m2")
print(f"All active locations in B/C: {len(active_scope):,} -> {total_area:,.0f} m2")
print(f"Locations held by SKUs outside our 3,500: {outside_area:,.0f} m2")

Floor area of one bay: 0.81 m2
Average locations per bay: 8.0
Floor area per location: 0.102 m2

Extra locations needed: 1,935 -> 197 m2
All active locations in B/C: 14,575 -> 1,487 m2
Locations held by SKUs outside our 3,500: 1,130 m2


One location needs only about 0.10 m² of floor space. The 1,935 extra locations need about 197 m². That is only 13% of the 1,487 m² used by all active locations in Zones B and C.

The low-activity SKUs outside our top 3,500 hold about 1,130 m². The space we need is about 17% of it. So the extra space is small, and it can come from inside the same zones.

The extra locations are not new shelves. They are existing locations that change owner, inside bays and aisles that already exist. So the access space (aisles) does not change, and the floor area of the proposal is only the storage that changes use (197 m² if we give all 1,935 extra locations). The shelf levels are already counted, because we use the average of 8 locations per bay.

**2.3 Workload.** For each SKU we add up the units picked from its active location in the 30 days of data. This is $u_i$. A SKU with high $u_i$ empties its location fast, so staff must bring stock from reserve often.

**2.4 Adjacency (what must be close to what).** We rate how important it is for two areas to be close:
- **A** = absolutely necessary
- **E** = especially important
- **I** = important

| Pair of areas | Rating | Why |
|---|---|---|
| Active picking – packing area | A | Every pick ends here |
| Active picking – reserve storage | E | Staff refill active locations from reserve |
| Reserve storage – packing area | I | Reserve is only visited for refills |


**Why we need this.** The ratings tell us which areas must stay close when we change the layout:
- Picking and packing (**A**) must stay close, so we never move picking locations far from packing.
- Picking and reserve (**E**) should stay close, so refills stay short.
- Reserve and packing (**I**) matter least, so reserve can be a bit farther away.

In our proposal we only change who uses each existing location, so no area moves and all these relations are kept.

In [42]:
# --- Workload: units picked per SKU from active locations in B/C ---
units = (
    bc_active_picks.groupby('sku')['units_packed']
    .sum()
    .reset_index(name='u_i')
)

item_opt = item_opt.drop(columns=['u_i'], errors='ignore').merge(units, on='sku', how='left')
item_opt['u_i'] = item_opt['u_i'].fillna(0)

gap = item_opt[item_opt['S_i'] > 1].copy()
gap['extra_locations'] = gap['S_i'] - 1

print(f"Units picked, all 3,500 SKUs: {item_opt['u_i'].sum():,.0f}")
print(f"Units picked, the {len(gap)} SKUs with S_i > 1: {gap['u_i'].sum():,.0f} "
      f"({gap['u_i'].sum()/item_opt['u_i'].sum():.1%})")

print(f"\nUnits picked per SKU ({len(gap)} SKUs):")
print(gap['u_i'].describe().round(1))

dates = pd.to_datetime(df_cdc['date_created'], format='%Y%m%d', errors='coerce').dropna()
n_days = (dates.max() - dates.min()).days + 1
print(f"\nPicking data period: {dates.min().date()} to {dates.max().date()} ({n_days} days)")

Units picked, all 3,500 SKUs: 333,469


Units picked, the 838 SKUs with S_i > 1: 211,509 (63.4%)

Units picked per SKU (838 SKUs):


count     838.0
mean      252.4
std       459.0
min         8.0
25%        62.0
50%       127.0
75%       253.0
max      5344.0
Name: u_i, dtype: float64

Picking data period: 2005-02-25 to 2005-03-26 (30 days)


The 838 SKUs that need more space make up only 24% of our SKUs, but they account for 63.4% of all units picked (211,509 of 333,469). They are the fast movers.

A typical one (the middle value) has 127 units picked, while the busiest has 5,344. Because these SKUs sell so many units from a single location, the location empties fast. This is why they need many refills from reserve.

All numbers cover the 30 days of picking data (25 February to 26 March 2005).

## **3. Quantitative Model: Replenishment Work**

We use a simple calculation model. It counts how many times staff must refill the active location of each SKU in the 30 days of data.

**3.1 Inputs** (for each SKU $i$)
- $u_i$: units picked in 30 days
- $C_i$: units the active location(s) can hold

**3.2 How it works.** When a location is empty, staff refill it to full from reserve. So the number of refill trips is:

$$R_i = \frac{u_i}{C_i}$$

- **Current layout:** each SKU has 1 location, so $C_i$ = 240 units (the capacity of one location from AS2).
- **Proposed layout:** each SKU gets $S_i$ locations, so $C_i$ = its full stock target `Max` (capped, as in AS2).

**3.3 Assumptions and constraint**
- A refill brings the location back to full.
- Demand in the 30 days is typical.
- One trip refills one SKU.
- The extra locations (1,935) must fit in the space held by low-activity SKUs (11,075 locations).

**3.4 What we compare.** Total refill trips (current vs. proposed) and the extra floor space used.

In [43]:
# --- Replenishment model: refill trips in 30 days, current vs proposed ---
gap['C_cur'] = np.minimum(gap['Max_capped'], LOCATION_CAPACITY)   # 1 location today
gap['C_prop'] = gap['Max_capped']                                 # S_i locations in the proposal

gap['trips_cur'] = gap['u_i'] / gap['C_cur']
gap['trips_prop'] = gap['u_i'] / gap['C_prop']
gap['trips_saved'] = gap['trips_cur'] - gap['trips_prop']

trips_cur = gap['trips_cur'].sum()
trips_prop = gap['trips_prop'].sum()
saved = trips_cur - trips_prop

print(f"Refill trips in 30 days, current layout:  {trips_cur:,.0f}")
print(f"Refill trips in 30 days, proposed layout: {trips_prop:,.0f}")
print(f"Trips saved: {saved:,.0f} ({saved/trips_cur:.1%})")
print(f"Per day: {trips_cur/30:,.1f} -> {trips_prop/30:,.1f}")

print(f"\nTrips saved per extra location: {saved/extra_locs:.3f}")
print(f"Trips saved per extra m2: {saved/extra_area:.2f}")

print("\nTop 5 SKUs by trips saved:")
print(gap.nlargest(5, 'trips_saved')[['sku', 'u_i', 'S_i', 'trips_cur', 'trips_prop', 'trips_saved']].round(1).to_string(index=False))

Refill trips in 30 days, current layout:  881
Refill trips in 30 days, proposed layout: 301
Trips saved: 581 (65.9%)
Per day: 29.4 -> 10.0

Trips saved per extra location: 0.300
Trips saved per extra m2: 2.94

Top 5 SKUs by trips saved:
       sku    u_i  S_i  trips_cur  trips_prop  trips_saved
  SPR87002 5344.0   10       22.3         2.4         19.8
  SPR85001 5050.0   10       21.0         2.3         18.7
  SPR19783 4049.0   10       16.9         1.8         15.0
  SPR85009 4075.0    7       17.0         2.5         14.4
MMM6549-YW 3695.0   10       15.4         1.7         13.7


With the current layout, staff make about 881 refill trips in 30 days (29 per day) for these 838 SKUs. With the proposed layout, they make about 301 (10 per day). That is 581 fewer trips, a 65.9% drop.

The savings are not equal. The busiest SKUs give the biggest gains. For example, `SPR87002` goes from 22 trips to 2.4 trips. Each extra location saves 0.3 trips in 30 days on average, and each extra square meter saves about 3 trips.

Note: this is a model of average demand. It assumes a refill brings the location back to full. Real refills may also happen before a location is empty.

## **4. Sensitivity Analysis: How Much Extra Space Do We Need?**

Giving all 1,935 extra locations may not be necessary. Each new location saves fewer trips than the one before it. So we test smaller budgets: 10%, 25%, 50%, 75% and 100% of the extra locations.

For each budget, we give the extra locations first to the SKUs where they save the most trips. Because every new location of a SKU saves less than its previous one, this order gives the best result for any budget. Then we count the trips saved and the floor space used.

In [44]:
# --- Sensitivity: trips saved vs. number of extra locations given ---
rows = []
for _, r in gap.iterrows():
    for k in range(1, int(r['S_i'])):
        c_prev = min(r['Max_capped'], LOCATION_CAPACITY * k)
        c_new = min(r['Max_capped'], LOCATION_CAPACITY * (k + 1))
        rows.append((r['sku'], k, r['u_i'] / c_prev - r['u_i'] / c_new))

inc = pd.DataFrame(rows, columns=['sku', 'k', 'gain']).sort_values(['gain', 'sku', 'k'], ascending=[False, True, True]).reset_index(drop=True)
inc['cum_locs'] = np.arange(1, len(inc) + 1)
inc['cum_saved'] = inc['gain'].cumsum()

N_ALL = len(inc)                          # every extra location the 843 fast SKUs need
N_TARGETED = int(round(0.5 * N_ALL))      # targeted option = first half (the 50% option)

print(f"Check: trips saved with all extra locations = {inc['cum_saved'].iloc[-1]:,.0f} (should be {saved:,.0f})\n")

sens = []
for share in [0.10, 0.25, 0.50, 0.75, 1.00]:
    n = int(round(share * len(inc)))
    s = inc['cum_saved'].iloc[n - 1]
    sens.append({
        'share_of_extra_locations': f"{share:.0%}",
        'extra_locations': n,
        'extra_area_m2': round(n * area_per_location),
        'trips_saved_30d': round(s),
        'share_of_max_saving': f"{s / saved:.0%}",
    })
sens_df = pd.DataFrame(sens)
print(sens_df.to_string(index=False))
save_table(sens_df, 'as3_sensitivity')

Check: trips saved with all extra locations = 581 (should be 581)

share_of_extra_locations  extra_locations  extra_area_m2  trips_saved_30d share_of_max_saving
                     10%              194             20              320                 55%
                     25%              484             49              446                 77%
                     50%              968             99              535                 92%
                     75%             1451            148              570                 98%
                    100%             1935            197              581                100%


[saved] artifacts\as3_sensitivity.csv


Each new location saves fewer trips than the one before it. The first 194 extra locations (10%) already save 320 trips, which is 55% of the maximum saving. With 968 locations (50%) we get 92% of the saving. The last 967 locations add only 46 more trips.

So we do not need to give all 1,935 extra locations. A good choice is 968 extra locations: it uses only 99 m² of floor space and saves 535 refill trips in 30 days (about 61% fewer than today).

**Figure: refill trips against extra locations.** The curve shows how the refill trips in 30 days fall as we give more extra locations to the fast SKUs. We give the locations in order of the biggest saving first. It uses the same numbers as the sensitivity table.

In [45]:
# --- Figure for the report (AS3): refill trips against the number of extra locations ---
n_all = len(inc)
trips_left = trips_cur - inc['cum_saved'].to_numpy()          # refill trips in 30 days

fig, ax = plt.subplots(figsize=(3.5, 2.7))
ax.plot(np.r_[0, inc['cum_locs'].to_numpy()], np.r_[trips_cur, trips_left], color='#4C78A8', lw=2)
for n, label in [(N_TARGETED, 'Targeted (50%)'), (n_all, 'Full (100%)')]:
    t = trips_cur - inc['cum_saved'].iloc[n - 1]
    ax.scatter(n, t, color='#E45756', zorder=3)
    ax.annotate(f'{label}\n{t:.0f} trips ({(t / trips_cur - 1) * 100:.0f}%)', (n, t),
                xytext=(-8, 14), textcoords='offset points', ha='right', fontsize=7)
ax.set_xlabel('Extra active locations')
ax.set_ylabel('Refill trips in 30 days')
ax.set_ylim(0, trips_cur * 1.05)
sec = ax.secondary_xaxis('top', functions=(lambda n: n * area_per_location, lambda a: a / area_per_location))
sec.set_xlabel('Extra floor space (m$^2$)', fontsize=8)
sec.tick_params(labelsize=7)
ax.tick_params(labelsize=7)
ax.xaxis.label.set_size(8); ax.yaxis.label.set_size(8)
plt.tight_layout()
save_fig('as3_refill_trips_curve')

print(f"Today: {trips_cur:.0f} trips. All {n_all} extra locations: {trips_left[-1]:.0f} trips. "
      f"The first {N_TARGETED} locations give {inc['cum_saved'].iloc[N_TARGETED - 1] / inc['cum_saved'].iloc[-1]:.0%} of the maximum saving.")
save_table(pd.DataFrame({'extra_locations': np.r_[0, inc['cum_locs'].to_numpy()], 'refill_trips_30d': np.r_[trips_cur, trips_left]}), 'as3_refill_curve')

[saved] artifacts\as3_refill_trips_curve.png


[saved] artifacts\as3_refill_trips_curve.pdf
Today: 881 trips. All 1935 extra locations: 301 trips. The first 968 locations give 92% of the maximum saving.
[saved] artifacts\as3_refill_curve.csv


C:\Users\joneo\AppData\Local\Temp\ipykernel_15512\2522381755.py:62: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


The curve flattens quickly: each new location saves fewer trips than the one before. The targeted option (968 locations, 99 m²) cuts the trips from 881 to 346 (−61%). Giving all 1,935 locations only reaches 301 trips (−66%) and needs twice the space. The targeted option gives 92% of the maximum saving with half of the space. This is why we recommend it.

## **5. Feasibility Check**

**5.1 What we check.** Giving extra locations to fast SKUs has a cost: some other SKUs lose their active location and must live in reserve only. We check if this is possible.

We take the SKUs with zero picks in the 30 days. They do not need a pick location now. We check three things: (1) are there enough of them, (2) how many units must move to reserve, and (3) do they already have stock in reserve?

In [46]:
# --- Feasibility: who gives up an active location? ---
zero_pick_skus = set(skus_in_scope) - set(freq['sku'])    # no picks in 30 days
cand = active_scope[active_scope['sku'].isin(zero_pick_skus)].copy()
cand = cand.sort_values(['quantity', 'sku'])                       # easiest to move first (least stock)

reserve_skus = set(df_cases['sku'])
avg_units_per_case = df_cases.loc[df_cases['location_type'].isin(['BLU', 'KCN']), 'quantity'].mean()

print(f"SKUs in B/C with zero picks in 30 days: {len(cand):,}")
print(f"Average units per case in case reserve: {avg_units_per_case:.1f}\n")

rows = []
for label, n in [('50% option', N_TARGETED), ('100% option', N_ALL)]:
    moved = cand.head(n)
    units = moved['quantity'].sum()
    rows.append({
        'option': label,
        'skus_to_move': n,
        'enough_candidates': len(cand) >= n,
        'units_to_move': int(units),
        'cases_needed': int(round(units / avg_units_per_case)),
        'already_in_reserve': f"{moved['sku'].isin(reserve_skus).mean():.0%}",
    })
print(pd.DataFrame(rows).to_string(index=False))
save_table(pd.DataFrame(rows), 'as3_feasibility')

SKUs in B/C with zero picks in 30 days: 5,118
Average units per case in case reserve: 11.9

     option  skus_to_move  enough_candidates  units_to_move  cases_needed already_in_reserve
 50% option           968               True              0             0                 0%
100% option          1935               True           2362           198                 4%
[saved] artifacts\as3_feasibility.csv


**5.2 Results.**

There are 5,118 SKUs with no picks in 30 days, so there are more than enough candidates to give up their active location (we need 968 or 1,935).

The 968 SKUs with the least stock have no units in their active location at all, so moving them costs nothing. For the full option (1,935 SKUs), only 2,362 units must go to reserve. That is about 198 cases, a very small amount compared with the 45,767 case reserve locations.

Only 4% of these SKUs already have stock in reserve. This is expected, because they are slow or empty items. But it brings a risk: if one of them gets an order, it has no active location. Staff could pick it directly from reserve, which already happens for 35.2% of cartons today.

**5.3 Conditions management must check before implementing:**
- Reserve space is available for the moved SKUs. Zone C has no reserve, so they go to reserve in other zones.
- Aisle width and equipment clearance allow refills at the busiest locations (up to 10 locations per SKU).
- Slow SKUs that lose their active location can still be picked from reserve when an order arrives.
- Shelf weight and safety rules are respected on the levels that receive fast SKUs.

## **6. Comparison and Recommendation**

**6.1 The options.** We compare three options for the same 838 SKUs and the same 30 days:
- **Current:** 1 active location per SKU.
- **Targeted (50%):** 968 extra locations, given first to the SKUs where they save the most trips.
- **Full (100%):** all 1,935 extra locations.

We do not know how long one refill trip takes, so we show time saved for 3, 5 and 10 minutes per trip.

In [47]:
# --- Final comparison: current vs targeted (50%) vs full (100%) ---
def option_row(name, n_extra):
    s = inc['cum_saved'].iloc[n_extra - 1] if n_extra > 0 else 0
    trips = trips_cur - s
    return {
        'option': name,
        'extra_locations': n_extra,
        'extra_area_m2': round(n_extra * area_per_location),
        'trips_30d': round(trips),
        'trips_per_day': round(trips / 30, 1),
        'trips_saved': round(s),
        'change_%': f"{-s / trips_cur:.0%}",
        'hours_saved_3min': round(s * 3 / 60),
        'hours_saved_5min': round(s * 5 / 60),
        'hours_saved_10min': round(s * 10 / 60),
    }

comparison = pd.DataFrame([
    option_row('Current', 0),
    option_row('Targeted (50%)', N_TARGETED),
    option_row('Full (100%)', N_ALL),
])
print(comparison.to_string(index=False))
save_table(comparison, 'as3_comparison')

        option  extra_locations  extra_area_m2  trips_30d  trips_per_day  trips_saved change_%  hours_saved_3min  hours_saved_5min  hours_saved_10min
       Current                0              0        881           29.4            0       0%                 0                 0                  0
Targeted (50%)              968             99        346           11.5          535     -61%                27                45                 89
   Full (100%)             1935            197        301           10.0          581     -66%                29                48                 97
[saved] artifacts\as3_comparison.csv


**6.2 Results of the comparison.**

The targeted option cuts refill trips from 881 to 346 in 30 days (61% fewer, from 29 to 12 per day). It needs 968 extra locations, which is 99 m² of floor space. The full option cuts trips by 66%, but it needs twice the space for only 46 more saved trips.

In time, the targeted option saves about 45 hours per month if a refill trip takes 5 minutes (27 to 89 hours for 3 to 10 minutes). This is a real but modest gain, because the total number of refill trips is not very large.

**6.3 Layout view: current vs. proposed.** The map shows the warehouse grid of Zones B and C. Each square is an aisle, and the grey squares are all the aisles. The color shows how many active locations the 838 fast SKUs use in that aisle. The star is the packing area. Both maps use the same scale. The aisles, bays and packing area stay fixed. Only the owner of some locations changes.

In [48]:
# --- Layout view: where do the fast SKUs sit, today vs. targeted proposal ---
fast = set(gap['sku'])

# Current: each fast SKU has 1 location
cur_map = loc_scope[loc_scope['sku'].isin(fast)]

# Proposed: 1 + extra locations per SKU, taking the closest ones to the packing area
extra_by_sku = inc.head(N_TARGETED).groupby('sku').size()
prop = loc_sorted[loc_sorted['assigned_sku'].isin(fast)].copy()
prop['rank_in_sku'] = prop.groupby('assigned_sku').cumcount()
prop['n_keep'] = 1 + prop['assigned_sku'].map(extra_by_sku).fillna(0)
prop = prop[prop['rank_in_sku'] < prop['n_keep']]

print(f"Locations used by the fast SKUs, current: {len(cur_map):,}")
print(f"Locations used by the fast SKUs, proposed: {len(prop):,}")
print(f"Average distance to packing area, current: {cur_map['dist_m'].mean():.1f} m")
print(f"Average distance to packing area, proposed: {prop['dist_m'].mean():.1f} m")

cur_cnt = cur_map.groupby(['col', 'row']).size().reset_index(name='n')
prop_cnt = prop.groupby(['col', 'row']).size().reset_index(name='n')
vmax = max(cur_cnt['n'].max(), prop_cnt['n'].max())

fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharex=True, sharey=True)
for ax, data, title in [(axes[0], cur_cnt, f'Current ({len(cur_map):,} locations)'),
                        (axes[1], prop_cnt, f'Proposed, targeted ({len(prop):,} locations)')]:
    ax.scatter(loc_scope['col'], loc_scope['row'], c='lightgrey', s=60, marker='s')
    sc_plot = ax.scatter(data['col'], data['row'], c=data['n'], cmap='YlOrRd', s=90,
                    marker='s', vmin=0, vmax=vmax)
    ax.scatter(io_point['col'], io_point['row'], marker='*', s=300, c='navy', label='Packing area')
    ax.invert_yaxis()
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Warehouse layout, column')
    ax.set_ylabel('Row')
    ax.legend(loc='upper right')
fig.colorbar(sc_plot, ax=axes, label='Locations used by the fast SKUs (per aisle)')
save_fig('as3_layout_current_vs_proposed')

Locations used by the fast SKUs, current: 837
Locations used by the fast SKUs, proposed: 1,806
Average distance to packing area, current: 55.2 m
Average distance to packing area, proposed: 63.0 m


[saved] artifacts\as3_layout_current_vs_proposed.png


[saved] artifacts\as3_layout_current_vs_proposed.pdf


C:\Users\joneo\AppData\Local\Temp\ipykernel_15512\2522381755.py:62: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


On the left (today), the locations of the fast SKUs are spread along the whole width of the zone, up to column 180, far from the packing area (the star). On the right (proposed), the 1,806 locations are concentrated in a compact block of aisles between columns 55 and 85, plus the aisles close to the packing area. The dark squares are aisles with many fast-SKU locations (up to about 150 in one aisle). The grey aisles on the far right are left for slow SKUs.

One fast SKU has no current location in the data, so the left map shows 837 locations and not 838.

The average distance of these locations to the packing area goes up from 55.2 m to 63.0 m. This is expected: the proposal adds about 970 more locations, and the extra ones are further away than the first one of each SKU. We check what this means for picking walk in the next step.

**6.4 Side effect on picking walk.** More locations per SKU means that some picks happen at locations further away. We check the average walking distance per pick for the 838 fast SKUs, current vs. proposed. Each SKU's picks are shared equally between its locations (as in AS2), and SKUs with more picks count more.

In [49]:
# --- Picking walk per pick for the fast SKUs: current vs. targeted proposal ---
cur_d = cur_map.drop_duplicates('sku').set_index('sku')['dist_m']
prop_d = prop.groupby('assigned_sku')['dist_m'].mean()

chk = (gap.set_index('sku')[['f_i']]
       .join(cur_d.rename('d_cur'), how='inner')
       .join(prop_d.rename('d_prop'), how='inner'))
w = chk['f_i']

d_cur = (chk['d_cur'] * w).sum() / w.sum()
d_prop = (chk['d_prop'] * w).sum() / w.sum()

print(f"SKUs compared: {len(chk):,}")
print(f"Average walk per pick, current:  {d_cur:.1f} m")
print(f"Average walk per pick, proposed: {d_prop:.1f} m")
print(f"Change: {d_prop - d_cur:+.1f} m ({(d_prop - d_cur) / d_cur:+.1%})")
print(f"Extra walk over 30 days: {(d_prop - d_cur) * w.sum():,.0f} m for {w.sum():,} picks")

SKUs compared: 837
Average walk per pick, current:  41.2 m
Average walk per pick, proposed: 43.5 m
Change: +2.3 m (+5.5%)
Extra walk over 30 days: 61,609 m for 26,995 picks


**Results.**
- Walk per pick for the fast SKUs: **41.2 m → 43.5 m** (+2.3 m, +5.5%).
- Extra walking in 30 days: **61,609 m**, about **17 hours** at 1 m/s (the speed we used in AS2).
- Refill trips saved: 535, about **45 hours** at 5 minutes per trip. Net gain: about **28 hours per month**.
- The change pays off if a refill trip takes more than about **1.9 minutes**.
- *Note for the report:* this section measures only the average walk of the fast SKUs to the packing area. AS5 measures the walking with the real batch routes of all the lines, and finds that the layout alone pays off above **3.3 minutes** and gains about 16 hours (not 28). Use the AS5 values in the report. The recommendation does not change.

**6.5 Recommendation**

We recommend the **targeted option**: give 968 extra active locations to the fast SKUs where they save the most trips.

- **Why:** it gets 92% of the saving with half of the space, and moving the slow SKUs out costs almost nothing.
- **Why not the full option:** the last 967 locations add only 46 more saved trips, and more walking.
- **Link to AS1:** more active space should also reduce Class A stockouts, but we cannot measure this.

**6.6 Other option: keep one location per SKU and refill more often.** It needs no moves and no extra walking. But it keeps 881 refill trips in 30 days (29 per day) instead of 346 (12 per day), and fast SKUs keep running empty between refills. We prefer the targeted option because its net saving is positive. The other option is a good fallback if management cannot give the extra space.

**6.7 Limitations**
- Average demand from only 30 days of data.
- The time per refill trip is an assumption.
- We did not measure the distance to reserve. Zone C has no reserve, so moved SKUs go to reserve in other zones.

**Summary — WH-AS3**
- **Problem:** 838 fast SKUs have only one active location, but they cause 63.4% of the units picked. Their locations empty fast and need many refills.
- **Proposal:** give these SKUs 968 extra locations (99 m²). Refill trips fall 61% (535 trips in 30 days).
- **Cost:** 2.3 m more walking per pick (+5.5%). The net gain is still about 28 hours per month.

# WH-AS4 — Order-Picking Improvement (Zones B & C)

## **1. Baseline: How Picking Works Today**

**1.1 Scope.** Pick lines of Zones B and C from the active pick batches, 25 February – 25 March 2005 (22 days with picking). Same zones and map as AS2 and AS3. We measure walking distance and picking time.

**1.2 How picking works (from the data).**
- Orders are released in **waves**. Each wave is split into **batches** (one batch = one picking tour).
- In Zones B and C, a typical batch has 13 orders, 31 pick lines and 19 aisles.
- 45% of the batches also visit other zones (A, D, F, L). We only measure the walk inside Zones B and C.

**1.3 Assumptions.**
- The picker starts and ends at the packing area (I/O point of AS2).
- Distance is the walking distance on the map grid (1 square = 42 inches = 1.07 m), between aisles, as in AS2. Consecutive picks in the same aisle count as one stop.
- The data does not show the real path of the picker (see 4.2). So we use two baselines: **A** = the data order, and **B** = one pass sorted by location code. B is the main baseline.
- Walking speed, picking time and working hours are assumptions (see 5.1).

In [50]:
# --- WH-AS4 Baseline: B/C pick lines linked to the map ---
# Keep the order of the lines inside each batch (historical sequence)
df_trc['seq_in_file'] = df_trc.groupby('truck_nbr').cumcount()

bc = df_trc[df_trc['zone'].isin(SCOPE_ZONES)].copy()
bc['aisle'] = bc['aisle'].astype(str).str.strip().str.zfill(3)

# Coordinates: one point per aisle (from the AP1 map)
bc = bc.merge(aisle_coords, on=['zone', 'aisle'], how='left')

n_all = len(bc)
print(f"B/C pick lines: {n_all:,} in {bc['truck_nbr'].nunique():,} batches "
      f"(of {len(df_trc):,} lines in all zones, {n_all/len(df_trc):.1%})")
print(f"Lines with a map coordinate: {bc['row'].notna().mean():.1%}")
print(f"Lines with a SKU in our 3,500-SKU scope: {bc['sku'].isin(set(item_opt['sku'])).mean():.1%}")
print(f"Lines with missing order number: {bc['pickticket_nbr'].isna().sum():,}")

no_map = bc[bc['row'].isna()]
if len(no_map):
    print("\nAisles without coordinate (lines):")
    print(no_map.groupby(['zone', 'aisle']).size().sort_values(ascending=False).head(10).to_string())

bc = bc.dropna(subset=['row', 'col']).copy()
bc[['row', 'col']] = bc[['row', 'col']].astype(int)

per_batch = bc.groupby('truck_nbr').agg(lines=('sku', 'size'),
                                        aisles=('aisle', 'nunique'),
                                        orders=('pickticket_nbr', 'nunique'))
print(f"\nBatches with at least one B/C line: {len(per_batch):,}")
print(per_batch.describe().round(1).to_string())
print(f"Batches that visit a single aisle in B/C: {(per_batch['aisles'] == 1).mean():.1%}")

B/C pick lines: 93,257 in 2,800 batches (of 138,674 lines in all zones, 67.2%)
Lines with a map coordinate: 100.0%
Lines with a SKU in our 3,500-SKU scope: 80.0%
Lines with missing order number: 0



Batches with at least one B/C line: 2,800
        lines  aisles  orders
count  2800.0  2800.0  2800.0
mean     33.3    18.4    17.1
std      26.3    12.7    17.2
min       1.0     1.0     1.0
25%       7.0     5.0     4.0
50%      31.0    19.0    13.0
75%      55.0    29.0    21.0
max     171.0    54.0    85.0
Batches that visit a single aisle in B/C: 9.2%


**1.4 Result.** We study **93,257 pick lines in 2,800 batches** (67% of all lines). All have a position on the map, and 80% belong to the 3,500 SKUs of AS2. Only 9% of the batches visit a single aisle, so most batches need a real walking route.

## **2. Walking Distance and a Small Test**

**2.1 Distance.** The picker walks from the packing area to each aisle and back. Distance = rows + columns on the grid, in meters (as in AS2).

**2.2 Small test.** We test the model on one small batch (2 orders) and check every leg by hand.

In [51]:
# --- WH-AS4 Distance functions and a small test ---
DEPOT = (io_point['row'], io_point['col'])

def leg_m(a, b):
    """Walking distance between two grid points, in meters."""
    return (abs(a[0] - b[0]) + abs(a[1] - b[1])) * GRID_TO_M

def route_length(stops):
    """Tour: depot -> stops in the given order -> depot (meters)."""
    path = [DEPOT] + list(stops) + [DEPOT]
    return sum(leg_m(path[i], path[i + 1]) for i in range(len(path) - 1))

def historical_stops(batch_lines):
    """Stops in the order of the data; consecutive picks in the same aisle = one stop."""
    b = batch_lines.sort_values('seq_in_file')
    pts = list(zip(b['row'], b['col']))
    return [p for i, p in enumerate(pts) if i == 0 or p != pts[i - 1]]

# --- Small test: one batch with 2 orders and 4-6 aisles ---
toy_id = per_batch[(per_batch['orders'] == 2) & (per_batch['aisles'].between(4, 6))].index[0]
toy = bc[bc['truck_nbr'] == toy_id].sort_values('seq_in_file')
print(f"Test batch: {toy_id} ({len(toy)} lines, {toy['pickticket_nbr'].nunique()} orders)")
print(toy[['seq_in_file', 'pickticket_nbr', 'zone', 'aisle', 'row', 'col']].to_string(index=False))

stops = historical_stops(toy)
print(f"\nI/O point (depot): {DEPOT}")
print("Stops in historical order:", stops)

path = [DEPOT] + stops + [DEPOT]
for a, b in zip(path[:-1], path[1:]):
    print(f"  {a} -> {b}: {leg_m(a, b):6.1f} m")
print(f"Total route length: {route_length(stops):.1f} m")

Test batch: T230058133 (5 lines, 2 orders)
 seq_in_file pickticket_nbr zone aisle  row  col
           0       23245357    C   352   14  122
           1       23245357    C   358   14  104
           2       23245357    C   362   14   92
           8       23245865    C   362   14   92
           9       23245865    C   364   14   86

I/O point (depot): (7, np.int64(10))
Stops in historical order: [(14, 122), (14, 104), (14, 92), (14, 86)]
  (7, np.int64(10)) -> (14, 122):  126.9 m
  (14, 122) -> (14, 104):   19.2 m
  (14, 104) -> (14, 92):   12.8 m
  (14, 92) -> (14, 86):    6.4 m
  (14, 86) -> (7, np.int64(10)):   88.5 m
Total route length: 253.9 m


**2.3 Result.** By hand: 238 grid squares x 1.07 m = 253.9 m, the same as the code. This batch cannot be improved, because its route is already the shortest. We need bigger batches to see differences.

## **3. Routing Methods**

**3.1 Methods.** For each batch, all methods visit the same aisles and start and end at the packing area:
- **A. Data order:** the lines in the order of the data (order by order).
- **B. One pass by location code:** one walk through the batch, sorted by zone, aisle, bay, level and position.
- **C. Nearest neighbour:** always go to the closest aisle not yet visited.
- **D. Nearest neighbour + 2-opt:** improve C by reversing parts of the route while it gets shorter.
- **E. B + 2-opt:** the same improvement, starting from B.
- **Best possible route:** tries all orders. It works only for batches with up to 10 aisles, and we use it to check how close the methods are to the best.

**3.2 Why not S-shape or largest-gap?** They need the inside of each aisle (length and cross-aisles). Our map has one point per aisle, so we use methods that work on points.

In [52]:
# --- WH-AS4 Routing methods: nearest neighbour, 2-opt, exact (small batches) ---
def nn_route(stops):
    """Nearest neighbour: go to the closest aisle not yet visited."""
    left = list(dict.fromkeys(stops))
    cur, route = DEPOT, []
    while left:
        nxt = min(left, key=lambda p: leg_m(cur, p))
        route.append(nxt)
        left.remove(nxt)
        cur = nxt
    return route

def two_opt(route):
    """Reverse parts of the route while this makes it shorter."""
    best = list(route)
    improved = True
    while improved:
        improved = False
        for i in range(len(best) - 1):
            for j in range(i + 1, len(best)):
                new = best[:i] + best[i:j + 1][::-1] + best[j + 1:]
                if route_length(new) < route_length(best) - 1e-9:
                    best, improved = new, True
    return best

def exact_route(stops, max_stops=10):
    """Best possible route (dynamic programming). Only for small batches."""
    pts = list(dict.fromkeys(stops))
    n = len(pts)
    if n > max_stops:
        return None
    d0 = [leg_m(DEPOT, p) for p in pts]
    d = [[leg_m(a, b) for b in pts] for a in pts]
    INF = float('inf')
    dp = [[INF] * n for _ in range(1 << n)]
    par = [[-1] * n for _ in range(1 << n)]
    for i in range(n):
        dp[1 << i][i] = d0[i]
    for mask in range(1 << n):
        for last in range(n):
            if dp[mask][last] == INF:
                continue
            for nxt in range(n):
                if mask & (1 << nxt):
                    continue
                m2, c = mask | (1 << nxt), dp[mask][last] + d[last][nxt]
                if c < dp[m2][nxt]:
                    dp[m2][nxt], par[m2][nxt] = c, last
    full = (1 << n) - 1
    last = min(range(n), key=lambda i: dp[full][i] + d0[i])
    order, mask = [], full
    while last != -1:
        order.append(pts[last])
        prev = par[mask][last]
        mask ^= (1 << last)
        last = prev
    return order[::-1]

def is_feasible(route, stops):
    """A route is feasible if it visits every needed aisle exactly once."""
    return sorted(route) == sorted(set(stops))

# --- Test on a medium batch (7-9 aisles) ---
mid_id = per_batch[per_batch['aisles'].between(7, 9)].index[0]
mid = bc[bc['truck_nbr'] == mid_id]
needed = list(dict.fromkeys(zip(mid.sort_values('seq_in_file')['row'],
                                mid.sort_values('seq_in_file')['col'])))
print(f"Test batch {mid_id}: {len(mid)} lines, {mid['pickticket_nbr'].nunique()} orders, "
      f"{len(set(needed))} aisles")

methods = {
    'Historical': historical_stops(mid),
    'Nearest neighbour': nn_route(needed),
}
methods['NN + 2-opt'] = two_opt(methods['Nearest neighbour'])
methods['Best possible'] = exact_route(needed)

base = route_length(methods['Historical'])
print(f"\n{'Method':20s} {'Stops':>6s} {'Length (m)':>11s} {'vs historical':>14s} {'Feasible':>9s}")
for name, r in methods.items():
    L = route_length(r)
    ok = is_feasible(r, needed) if name != 'Historical' else 'n/a'
    print(f"{name:20s} {len(r):6d} {L:11.1f} {(L - base) / base:14.1%} {str(ok):>9s}")

Test batch T230057491: 8 lines, 3 orders, 8 aisles

Method                Stops  Length (m)  vs historical  Feasible
Historical                8       322.2           0.0%       n/a
Nearest neighbour         8       262.4         -18.5%      True
NN + 2-opt                8       247.5         -23.2%      True
Best possible             8       247.5         -23.2%      True


**3.3 Result.** In a batch with 8 aisles: data order 322.2 m, nearest neighbour 262.4 m (-18.5%), nearest neighbour + 2-opt 247.5 m (-23.2%), the same as the best possible route. All routes visit every aisle once, so they are feasible.

## **4. Task 1: Comparison on All Batches (Current Warehouse)**

**4.1 What we do.** We take the 2,800 batches of the data. For each batch we build the route of every method (A to E, see 3.1) and measure its length in meters. Then we add up all batches.
- All methods use the same batches, the same aisles, and the same start and end point (the packing area). Storage, layout and equipment do not change. Only the order of the visits changes.
- No random numbers are used, so the results can be repeated exactly.
- For batches with up to 10 aisles we also compute the best possible route, to see how close the methods are to the best.
- *Code note:* the small test in Section 3 uses a simple version of nearest neighbour + 2-opt. Here we use a faster version (the same method, on a distance matrix).

In [53]:
# --- WH-AS4 Task 1: all methods on all 2,800 batches ---
import time

def dist_matrix(pts):
    P = np.array([DEPOT] + list(pts), dtype=float)
    return (np.abs(P[:, None, 0] - P[None, :, 0]) + np.abs(P[:, None, 1] - P[None, :, 1])) * GRID_TO_M

def nn_2opt(pts):
    """Nearest neighbour, then 2-opt. Returns (NN length, NN+2-opt length) in meters."""
    n = len(pts)
    D = dist_matrix(pts)
    tour, left, cur = [0], set(range(1, n + 1)), 0
    while left:
        nxt = min(left, key=lambda k: D[cur, k])
        tour.append(nxt); left.remove(nxt); cur = nxt
    tour.append(0)
    length = lambda t: sum(D[t[k], t[k + 1]] for k in range(len(t) - 1))
    nn_len = length(tour)
    improved = True
    while improved:
        improved = False
        for i in range(1, n):
            for j in range(i + 1, n + 1):
                a, b, c, d = tour[i - 1], tour[i], tour[j], tour[j + 1]
                if D[a, c] + D[b, d] < D[a, b] + D[c, d] - 1e-9:
                    tour[i:j + 1] = tour[i:j + 1][::-1]
                    improved = True
    return nn_len, length(tour)

def two_opt_from(order):
    """2-opt starting from a given visiting order (list of grid points). Returns length in m."""
    n = len(order)
    D = dist_matrix(order)                     # index 0 = depot, index k = order[k-1]
    tour = list(range(n + 1)) + [0]
    length = lambda t: sum(D[t[k], t[k + 1]] for k in range(len(t) - 1))
    improved = True
    while improved:
        improved = False
        for i in range(1, n):
            for j in range(i + 1, n + 1):
                a, b, c, d = tour[i - 1], tour[i], tour[j], tour[j + 1]
                if D[a, c] + D[b, d] < D[a, b] + D[c, d] - 1e-9:
                    tour[i:j + 1] = tour[i:j + 1][::-1]
                    improved = True
    return length(tour)

t0 = time.time()
rows = []
for tid, g in bc.groupby('truck_nbr'):
    g = g.sort_values('seq_in_file')
    pts = list(dict.fromkeys(zip(g['row'], g['col'])))      # aisles to visit, once each
    nn_len, opt_len = nn_2opt(pts)
    ex = exact_route(pts)                                    # None if more than 10 aisles

    # Baseline B: one pass sorted by location code (zone, aisle number, ...)
    gs = g.copy()
    gs['aisle_n'] = pd.to_numeric(gs['aisle'], errors='coerce')
    gs = gs.sort_values(['zone', 'aisle_n', 'bay', 'level', 'position'])
    code_stops = list(dict.fromkeys(zip(gs['row'], gs['col'])))

    rows.append({
        'truck_nbr': tid, 'lines': len(g), 'orders': g['pickticket_nbr'].nunique(),
        'aisles': len(pts),
        'hist_m': route_length(historical_stops(g)),         # A. data order
        'B_m': route_length(code_stops),                     # B. code order
        'nn_m': nn_len,                                      # C. nearest neighbour
        'opt2_m': opt_len,                                   # D. NN + 2-opt
        'B2opt_m': two_opt_from(code_stops),                 # E. code order + 2-opt
        'exact_m': route_length(ex) if ex is not None else np.nan,
    })
res = pd.DataFrame(rows)
print(f"Done in {time.time() - t0:.0f} s for {len(res):,} batches")

# Quick summary: total km per method (nothing is explained yet, only the totals)
tot = (res[['hist_m', 'B_m', 'nn_m', 'opt2_m', 'B2opt_m']].sum() / 1000).round(0)
tot.index = ['A. Data order', 'B. Code order', 'C. Nearest neighbour',
             'D. NN + 2-opt', 'E. Code order + 2-opt']
print(tot.to_string())

# Sanity checks
assert (res['opt2_m'] <= res['nn_m'] + 1e-6).all(), "2-opt must not be longer than NN"
assert (res['B2opt_m'] <= res['B_m'] + 1e-6).all(), "2-opt must not be longer than B"
small = res.dropna(subset=['exact_m'])
assert (small['exact_m'] <= small['opt2_m'] + 1e-6).all(), "best route must be the shortest"

Done in 7 s for 2,800 batches
A. Data order            2699.0
B. Code order            1017.0
C. Nearest neighbour     1046.0
D. NN + 2-opt             938.0
E. Code order + 2-opt     927.0


**4.2 First look at the totals.**

The cell above computed the route length of every method for all 2,800 batches. It prints the total km of each one.

Method A is much longer than all the others (about 2.6 times B). This looks strange, so we check it in the next cell.

In [54]:
# --- WH-AS4 Check: is the file order really a walking route? ---
rows = []
for tid, g in bc.groupby('truck_nbr'):
    g = g.sort_values('seq_in_file')
    uniq = list(dict.fromkeys(zip(g['row'], g['col'])))
    hs = historical_stops(g)                                   # file order

    # Alternative baseline: pick list sorted by location code (zone, aisle, bay, level, position)
    gs = g.copy()
    gs['aisle_n'] = pd.to_numeric(gs['aisle'], errors='coerce')
    gs = gs.sort_values(['zone', 'aisle_n', 'bay', 'level', 'position'])
    ls = historical_stops(gs.assign(seq_in_file=range(len(gs))))

    rows.append({'truck_nbr': tid, 'aisles': len(uniq),
                 'file_visits': len(hs), 'code_visits': len(ls),
                 'file_m': route_length(hs), 'code_m': route_length(ls)})

chk = pd.DataFrame(rows).merge(res[['truck_nbr', 'opt2_m']], on='truck_nbr')

print("Stops per batch divided by different aisles (1.0 = no aisle visited twice):")
print(f"  file order: {(chk['file_visits'] / chk['aisles']).mean():.2f}   "
      f"location-code order: {(chk['code_visits'] / chk['aisles']).mean():.2f}")

print("\nTotal distance (km):")
print(f"  file order {chk['file_m'].sum()/1000:,.0f} | location-code order {chk['code_m'].sum()/1000:,.0f} "
      f"| NN + 2-opt {chk['opt2_m'].sum()/1000:,.0f}")

chk['size'] = pd.cut(chk['aisles'], [0, 5, 10, 20, 30, 100],
                     labels=['1-5', '6-10', '11-20', '21-30', '31+'])
tab = (chk.groupby('size', observed=True)
          .agg(batches=('truck_nbr', 'size'), file_m=('file_m', 'mean'),
               code_m=('code_m', 'mean'), best_m=('opt2_m', 'mean')).round(0))
print("\nMean route length (m) by batch size (number of aisles):")
print(tab.to_string())

# Look at one big batch: the order of its stops in the file
big_id = chk[chk['aisles'].between(28, 32)]['truck_nbr'].iloc[0]
big = bc[bc['truck_nbr'] == big_id].sort_values('seq_in_file')
print(f"\nBatch {big_id}: first 20 lines in file order (column = distance from packing):")
print(big[['seq_in_file', 'pickticket_nbr', 'zone', 'aisle', 'col']].head(20).to_string(index=False))

Stops per batch divided by different aisles (1.0 = no aisle visited twice):
  file order: 1.21   location-code order: 1.00

Total distance (km):
  file order 2,699 | location-code order 1,017 | NN + 2-opt 938

Mean route length (m) by batch size (number of aisles):
       batches  file_m  code_m  best_m
size                                  
1-5        720   276.0   271.0   268.0
6-10       278   418.0   351.0   336.0
11-20      469   641.0   371.0   344.0
21-30      748  1369.0   394.0   357.0
31+        585  1812.0   436.0   382.0

Batch T230057473: first 20 lines in file order (column = distance from packing):
 seq_in_file pickticket_nbr zone aisle  col
           0       23232120    B   208   27
           1       23232120    B   211   19
           2       23232120    B   215   10
           3       23232120    B   217   16
           4       23232120    B   224   37
           5       23232120    B   233   65
           6       23232120    C   310  113
           7       23232120

**4.3 What the check shows: A is not a real walking route.**
- **Why:** in the data, the lines of each order are listed one order after another, and every order starts again from the nearest aisles. In the example batch above: aisle 208, then up to 328, and then again 208 for the next order. A picker who follows this would cross the warehouse once for each order.
- **Aisles visited twice:** A visits each aisle 1.21 times on average (there are walks back). B visits each aisle once.
- **Big batches suffer most:** with 1 to 5 aisles, A and B are almost the same (276 m and 271 m). With 31 or more aisles, A is 1,812 m and B is only 436 m.
- **Our decision:** the data does not show the real path of the pickers, so we use two baselines. **B is the main baseline**, so that our gains do not come from a weak baseline. **A is shown only for reference.**

In [55]:
# --- WH-AS4 Task 1: final comparison of all methods (A and B baselines) ---
def two_opt_from(order):
    """2-opt starting from a given visiting order (list of grid points). Returns length in m."""
    n = len(order)
    D = dist_matrix(order)                     # index 0 = depot, index k = order[k-1]
    tour = list(range(n + 1)) + [0]
    length = lambda t: sum(D[t[k], t[k + 1]] for k in range(len(t) - 1))
    improved = True
    while improved:
        improved = False
        for i in range(1, n):
            for j in range(i + 1, n + 1):
                a, b, c, d = tour[i - 1], tour[i], tour[j], tour[j + 1]
                if D[a, c] + D[b, d] < D[a, b] + D[c, d] - 1e-9:
                    tour[i:j + 1] = tour[i:j + 1][::-1]
                    improved = True
    return length(tour)

rows = []
for tid, g in bc.groupby('truck_nbr'):
    gs = g.copy()
    gs['aisle_n'] = pd.to_numeric(gs['aisle'], errors='coerce')
    gs = gs.sort_values(['zone', 'aisle_n', 'bay', 'level', 'position'])
    code_stops = list(dict.fromkeys(zip(gs['row'], gs['col'])))      # baseline B order
    rows.append({'truck_nbr': tid,
                 'B_m': route_length(code_stops),
                 'B2opt_m': two_opt_from(code_stops)})
res = res.drop(columns=['B_m', 'B2opt_m'], errors='ignore').merge(pd.DataFrame(rows), on='truck_nbr')

# Sanity checks
assert (res['B2opt_m'] <= res['B_m'] + 1e-6).all(), "2-opt from B can never be longer than B"
small = res.dropna(subset=['exact_m'])
assert (small['exact_m'] <= small['B2opt_m'] + 1e-6).all(), "best route must be the shortest"

methods = [('A. Orders one by one (data order)', 'hist_m'),
           ('B. One pass sorted by location code', 'B_m'),
           ('C. Nearest neighbour', 'nn_m'),
           ('D. Nearest neighbour + 2-opt', 'opt2_m'),
           ('E. B + 2-opt', 'B2opt_m')]
tab = pd.DataFrame([{
    'Method': name,
    'Total (km)': res[c].sum() / 1000,
    'Mean per batch (m)': res[c].mean(),
    'vs A (%)': (res[c].sum() / res['hist_m'].sum() - 1) * 100,
    'vs B (%)': (res[c].sum() / res['B_m'].sum() - 1) * 100,
    'Batches shorter than B (%)': (res[c] < res['B_m'] - 1e-6).mean() * 100,
} for name, c in methods]).round(1)
print(tab.to_string(index=False))

print(f"\nBatches with the best possible route (<= 10 aisles): {len(small):,} of {len(res):,}")
for name, c in [('B', 'B_m'), ('E. B + 2-opt', 'B2opt_m')]:
    gp = (small[c] / small['exact_m'] - 1) * 100
    print(f"  {name:14s} vs best possible: mean gap {gp.mean():.2f}%, max {gp.max():.2f}%")
save_table(tab, 'as4_methods_comparison')

                             Method  Total (km)  Mean per batch (m)  vs A (%)  vs B (%)  Batches shorter than B (%)
  A. Orders one by one (data order)      2699.3               964.1       0.0     165.5                         0.0
B. One pass sorted by location code      1016.6               363.1     -62.3       0.0                         0.0
               C. Nearest neighbour      1045.5               373.4     -61.3       2.8                        21.6
       D. Nearest neighbour + 2-opt       937.7               334.9     -65.3      -7.8                        50.6
                       E. B + 2-opt       927.4               331.2     -65.6      -8.8                        51.9

Batches with the best possible route (<= 10 aisles): 998 of 2,800
  B              vs best possible: mean gap 2.34%, max 65.00%
  E. B + 2-opt   vs best possible: mean gap 0.01%, max 8.00%
[saved] artifacts\as4_methods_comparison.csv


**4.4 Results.** Total walking distance of the 2,800 batches (22 picking days):

| Method | Distance (km) | Change vs B |
|---|---|---|
| A. Data order | 2,699 | +165% |
| B. One pass by location code (baseline) | 1,017 | 0% |
| C. Nearest neighbour | 1,046 | +2.8% |
| D. Nearest neighbour + 2-opt | 938 | -7.8% |
| E. B + 2-opt | **927** | **-8.8%** |

- B is 62% shorter than A.
- **E is the best:** 8.8% less walking than B, and no batch gets longer.
- C alone is worse than B, so "go to the closest aisle" is not enough.
- Check with the best possible route (998 batches with up to 10 aisles): E is on average only 0.01% above it (at most 8%). So E is almost optimal.
- **E is our proposal for the current warehouse.** Only the route changes.

## **5. Task 1: From Distance to Labor Time**

**5.1 Assumptions.** Distance is only part of the work, so we add picking time. These values are assumptions, and we test them in the sensitivity analysis:
- Walking speed with a cart: 1.0 m/s.
- Time to find, pick and scan one line: 20 seconds.
- Productive time per picker: 7 hours per day.
- Days with picking: 22.

In [56]:
# --- WH-AS4 Labor time: from distance to hours and pickers ---
SPEED_MPS = 1.0            # walking speed with a cart (assumption)
PICK_S_PER_LINE = 20       # seconds to find, pick and scan one line (assumption)
PRODUCTIVE_H_PER_DAY = 7   # productive hours per picker per day (assumption)

n_lines = len(bc)
pick_days = bc['date_created'].nunique()
print(f"Pick lines: {n_lines:,} | days with picking: {pick_days}")

pick_h = n_lines * PICK_S_PER_LINE / 3600       # same for every method (same lines)

rows = []
for name, c in [('A. Orders one by one', 'hist_m'),
                ('B. One pass by location code', 'B_m'),
                ('E. B + 2-opt (proposal)', 'B2opt_m')]:
    travel_h = res[c].sum() / SPEED_MPS / 3600
    total_h = travel_h + pick_h
    rows.append({'Method': name,
                 'Distance (km)': res[c].sum() / 1000,
                 'Travel (h)': travel_h,
                 'Picking (h)': pick_h,
                 'Total (h)': total_h,
                 'Pickers needed': total_h / pick_days / PRODUCTIVE_H_PER_DAY})
labor_tab = pd.DataFrame(rows)
b_row = labor_tab[labor_tab['Method'].str.startswith('B.')].iloc[0]
labor_tab['Travel vs B (%)'] = (labor_tab['Travel (h)'] / b_row['Travel (h)'] - 1) * 100
labor_tab['Total vs B (%)'] = (labor_tab['Total (h)'] / b_row['Total (h)'] - 1) * 100
print(labor_tab.round(1).to_string(index=False))
print(f"\nTravel is {b_row['Travel (h)'] / b_row['Total (h)']:.0%} of the total time in baseline B")
save_table(labor_tab, 'as4_labor_time')

Pick lines: 93,257 | days with picking: 22
                      Method  Distance (km)  Travel (h)  Picking (h)  Total (h)  Pickers needed  Travel vs B (%)  Total vs B (%)
        A. Orders one by one         2699.3       749.8        518.1     1267.9             8.2            165.5            58.4
B. One pass by location code         1016.6       282.4        518.1      800.5             5.2              0.0             0.0
     E. B + 2-opt (proposal)          927.4       257.6        518.1      775.7             5.0             -8.8            -3.1

Travel is 35% of the total time in baseline B
[saved] artifacts\as4_labor_time.csv


**5.2 Results (time).**
- Walking is 35% of the total time in B (282 h of 800 h in 22 days). Picking takes the rest and does not change with the route.
- E saves 8.8% of the walking time but only **3.1% of the total time**: about 25 hours in 22 days (1.1 hours per day). Pickers needed: 5.2 to 5.0.
- The biggest gain is from A to B: 467 hours less (-37%) and 3 pickers fewer (8.2 to 5.2). It depends on whether pickers really walked order by order, which the data does not show.
- Routing alone gives a small but safe gain. Bigger gains need less walking, like the new storage plan (Task 2).

## **6. Task 1: Feasibility Check**

**6.1 What we check.**
- **Order completion:** every route visits every aisle that has a pick, so all lines and orders are completed. The code below checks this for all 2,800 batches.
- **Capacity:** the batches are the same as today (same lines, same carts), so the load on the cart does not change.
- **Access:** routes only change the order of the visits. No new area is used.
- **Labor:** 5.0 pickers instead of 5.2 (assumptions in 5.1).

**6.2 Not checked (limitations).** Crowding between pickers in the same aisle, and the real walking path inside the aisles (our map has one point per aisle). Replenishment is not changed in Task 1.

In [57]:
# --- WH-AS4 Feasibility: every route visits every needed aisle exactly once ---
def code_order_stops(g):
    """Baseline B order: sorted by location code, one stop per aisle."""
    gs = g.copy()
    gs['aisle_n'] = pd.to_numeric(gs['aisle'], errors='coerce')
    gs = gs.sort_values(['zone', 'aisle_n', 'bay', 'level', 'position'])
    return list(dict.fromkeys(zip(gs['row'], gs['col'])))

def two_opt_order(order):
    """Same as two_opt_from, but returns the final visiting order (list of grid points)."""
    n = len(order)
    D = dist_matrix(order)
    tour = list(range(n + 1)) + [0]
    improved = True
    while improved:
        improved = False
        for i in range(1, n):
            for j in range(i + 1, n + 1):
                a, b, c, d = tour[i - 1], tour[i], tour[j], tour[j + 1]
                if D[a, c] + D[b, d] < D[a, b] + D[c, d] - 1e-9:
                    tour[i:j + 1] = tour[i:j + 1][::-1]
                    improved = True
    return [order[k - 1] for k in tour[1:-1]]

len_E = res.set_index('truck_nbr')['B2opt_m']
routes_E, n_ok, lines_ok, len_mismatch = {}, 0, 0, 0
for tid, g in bc.groupby('truck_nbr'):
    r = two_opt_order(code_order_stops(g))
    routes_E[tid] = r
    if is_feasible(r, set(zip(g['row'], g['col']))):
        n_ok += 1
        lines_ok += len(g)
    if abs(route_length(r) - len_E[tid]) > 1e-6:
        len_mismatch += 1

print(f"Batches with a feasible route E: {n_ok:,} of {len(routes_E):,}")
print(f"Pick lines covered: {lines_ok:,} of {len(bc):,}")
print(f"Route lengths different from the table: {len_mismatch}")

Batches with a feasible route E: 2,800 of 2,800
Pick lines covered: 93,257 of 93,257
Route lengths different from the table: 0


**6.3 Result.** All 2,800 routes of method E are feasible: every route visits every aisle that has a pick, and all 93,257 pick lines are covered. The route lengths are the same as in the comparison table.

**Figure: an example batch.** To see what method E changes, we draw the same batch with route B and with route E. The star is the packing area, the red dots are the aisles to visit, and the numbers give the visiting order. The batch is a typical one among the batches where E is shorter than B (10 to 30 aisles, median saving).

In [58]:
# --- Figure for the report (AS4): one batch, route B against route E on the warehouse map ---
res['gain_pct'] = (res['B_m'] - res['B2opt_m']) / res['B_m'] * 100          # saving of E against B, per batch
improved = res[res['gain_pct'] > 0.01]
print(f"Batches where E is shorter than B: {len(improved):,} of {len(res):,} ({len(improved) / len(res):.1%}). "
      f"Mean saving in those batches: {improved['gain_pct'].mean():.1f}%.")

# Example: a batch with 10 to 30 aisles where E is shorter, with the median saving among them
cand = improved[improved['aisles'].between(10, 30)]
tid = cand.iloc[(cand['gain_pct'] - cand['gain_pct'].median()).abs().argsort().iloc[0]]['truck_nbr']
g = bc[bc['truck_nbr'] == tid]
stops_B = code_order_stops(g)               # method B: one pass by location code
stops_E = two_opt_order(stops_B)            # method E: B improved with 2-opt

def l_path(points):
    """Walking path on the grid: along the columns first, then along the rows."""
    xs, ys = [points[0][1]], [points[0][0]]
    for (r0, c0), (r1, c1) in zip(points[:-1], points[1:]):
        xs += [c1, c1]; ys += [r0, r1]
    return xs, ys

fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.2), sharex=True, sharey=True)
for ax, stops, name, col in [(axes[0], stops_B, 'B: one pass by location code', '#7F7F7F'),
                             (axes[1], stops_E, 'E: B improved with 2-opt', '#4C78A8')]:
    ax.scatter(loc_scope['col'], loc_scope['row'], c='#E6E6E6', s=6, marker='s')
    xs, ys = l_path([DEPOT] + list(stops) + [DEPOT])
    ax.plot(xs, ys, color=col, lw=1.1, alpha=0.9)
    ax.scatter([p[1] for p in stops], [p[0] for p in stops], c='#E45756', s=18, zorder=3)
    for k, (r, c) in enumerate(stops, 1):
        ax.annotate(str(k), (c, r), fontsize=5.5, xytext=(2, 2), textcoords='offset points')
    ax.scatter(DEPOT[1], DEPOT[0], marker='*', s=110, c='navy', zorder=4)
    ax.set_title(f'{name}\n{route_length(stops):.0f} m', fontsize=8)
    ax.set_xlabel('Column (packing area on the left)', fontsize=7)
    ax.tick_params(labelsize=6)
axes[0].set_ylabel('Row', fontsize=7)
axes[0].invert_yaxis()
plt.tight_layout()
save_fig('as4_route_example')

lb, le = route_length(stops_B), route_length(stops_E)
print(f"Batch {tid}: {len(g)} lines, {len(stops_B)} aisles. Route B {lb:.0f} m, route E {le:.0f} m "
      f"({(le / lb - 1) * 100:.1f}%).")
route_df = pd.DataFrame([{'Route': n, 'Stop': k, 'Row': r, 'Column': col} for n, st in [('B', stops_B), ('E', stops_E)] for k, (r, col) in enumerate(st, 1)])
save_table(route_df, 'as4_route_example_stops')

Batches where E is shorter than B: 1,453 of 2,800 (51.9%). Mean saving in those batches: 15.4%.


[saved] artifacts\as4_route_example.png


[saved] artifacts\as4_route_example.pdf
Batch T230062483: 20 lines, 14 aisles. Route B 465 m, route E 388 m (-16.5%).
[saved] artifacts\as4_route_example_stops.csv


C:\Users\joneo\AppData\Local\Temp\ipykernel_15512\2522381755.py:62: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Both routes visit the same aisles. Method B follows the location codes, and the picker crosses the warehouse more than once. Method E keeps the same stops and changes only their order, so there are fewer long returns. In this batch (20 lines, 14 aisles) route E is **16.5% shorter** (388 m against 465 m).

This batch is an example where E helps, not an average case. E is shorter than B in **51.9%** of the 2,800 batches (1,453), with a mean saving of **15.4%** in those. In the other half, the code order is already a good route, because the aisles lie in a few rows. The total saving of 8.8% comes from the batches where it matters.

## **7. Task 2: Retained Designs**

**7.1 Retained designs.** We keep two storage designs from AS2 and AS3. The aisles, the packing area and the equipment (carts) do not change.
- **D1. Full slotting:** the fast SKUs get all the locations they need (1,935 extra), close to the packing area (AS2).
- **D2. Targeted slotting:** only 968 extra locations (99 m²), for the SKUs where they save the most refill trips (the AS3 recommendation).

**7.2 Picking strategy.** Both designs use the same strategy: **one pass sorted by location code, improved with 2-opt (E)**. It fits both designs because only the position of the SKUs changes: the batches and the aisles are the same. When a SKU has several locations, its picks are split evenly between them (as in the AS2 model).

**7.3 How we separate the effects.** We compare everything with baseline B (today's storage, one pass by location code).
- **Picking only:** E on today's storage (Task 1).
- **Design only:** B on the new storage.
- **Combined:** E on the new storage.

**7.4 Assumptions.**

1. SKUs outside the 3,500 stay where they are today. Their picks happen in the same aisles as now.
2. Refill trips come from AS3: 881 (today), 301 (D1) and 346 (D2). Each trip takes 5 minutes.

In [59]:
# --- WH-AS4 Task 2: retained designs, new pick positions and walking distance ---
code_key = {(int(r), int(c)): (z, a) for z, a, r, c in
            zip(aisle_coords['zone'], aisle_coords['aisle'], aisle_coords['row'], aisle_coords['col'])}
assert len(code_key) == len(aisle_coords), "two aisles share the same grid point"
scope_skus = set(item_opt['sku'])

def scope_design(n_extra):
    """Locations of the 3,500 scope SKUs when n_extra of the extra locations are given."""
    extra_by_sku = inc.head(n_extra).groupby('sku').size()
    d = loc_sorted[loc_sorted['assigned_sku'].notna()].copy()
    d['rank_in_sku'] = d.groupby('assigned_sku').cumcount()
    d['n_keep'] = 1 + d['assigned_sku'].map(extra_by_sku).fillna(0)
    return d[d['rank_in_sku'] < d['n_keep']]

def remap_lines(d):
    """Move the pick lines of scope SKUs to the locations of design d (picks split evenly)."""
    lists = {}
    for s, r, c in zip(d['assigned_sku'], d['row'], d['col']):
        lists.setdefault(s, []).append((int(r), int(c)))
    lists = {s: np.array(v) for s, v in lists.items()}
    lines = (bc.sort_values(['pick_datetime', 'truck_nbr', 'seq_in_file'])
               [['truck_nbr', 'seq_in_file', 'sku', 'row', 'col']].copy())
    lines['k'] = lines.groupby('sku').cumcount()
    new_row, new_col = lines['row'].to_numpy().copy(), lines['col'].to_numpy().copy()
    for i, (s, k) in enumerate(zip(lines['sku'].to_numpy(), lines['k'].to_numpy())):
        a = lists.get(s)
        if a is not None:
            new_row[i], new_col[i] = a[k % len(a)]
    lines['row'], lines['col'] = new_row, new_col
    return lines

def evaluate(lines):
    """Route length per batch with rule B (one pass by location code) and E (B + 2-opt)."""
    rows = []
    for tid, g in lines.groupby('truck_nbr'):
        stops = sorted(set(zip(g['row'], g['col'])), key=code_key.get)
        rows.append((tid, route_length(stops), two_opt_from(stops)))
    return pd.DataFrame(rows, columns=['truck_nbr', 'B_m', 'E_m'])

t0 = time.time()
scenarios = {
    'Current storage':      bc[['truck_nbr', 'seq_in_file', 'sku', 'row', 'col']],
    'D1 full slotting':     remap_lines(scope_design(N_ALL)),
    'D2 targeted slotting': remap_lines(scope_design(N_TARGETED)),
}
design_res = {name: evaluate(lines) for name, lines in scenarios.items()}
print(f"Done in {time.time() - t0:.0f} s")

# Check: the current storage must give the same totals as Task 1
cur = design_res['Current storage']
assert abs(cur['B_m'].sum() - res['B_m'].sum()) < 1e-3, "B does not match Task 1"
assert abs(cur['E_m'].sum() - res['B2opt_m'].sum()) < 1e-3, "E does not match Task 1"

base_B = cur['B_m'].sum()
rows = []
for name, r in design_res.items():
    for rule, col in [('B (one pass)', 'B_m'), ('E (B + 2-opt)', 'E_m')]:
        rows.append({'Storage': name, 'Routing': rule,
                     'Walking (km)': r[col].sum() / 1000,
                     'vs baseline B (%)': (r[col].sum() / base_B - 1) * 100})
print("\n", pd.DataFrame(rows).round(1).to_string(index=False))

# Limitation: SKUs outside the 3,500 whose current location is used by the design
line_cnt = bc['sku'].value_counts()
ns = loc_scope[~loc_scope['sku'].isin(scope_skus)]
ns_keys = list(zip(ns['zone'], ns['aisle'], ns['bay'], ns['level'], ns['position']))
for name, n_extra in [('D1', N_ALL), ('D2', N_TARGETED)]:
    d = scope_design(n_extra)
    taken = set(zip(d['zone'], d['aisle'], d['bay'], d['level'], d['position']))
    hit_skus = ns.loc[[k in taken for k in ns_keys], 'sku']
    share = line_cnt.reindex(hit_skus).fillna(0).sum() / len(bc)
    print(f"{name}: {len(hit_skus):,} of {len(ns):,} outside SKUs lose their location "
          f"({share:.1%} of all pick lines)")

Done in 4 s

              Storage       Routing  Walking (km)  vs baseline B (%)
     Current storage  B (one pass)        1016.6                0.0
     Current storage E (B + 2-opt)         927.4               -8.8
    D1 full slotting  B (one pass)        1084.8                6.7
    D1 full slotting E (B + 2-opt)         843.3              -17.0
D2 targeted slotting  B (one pass)        1084.8                6.7
D2 targeted slotting E (B + 2-opt)         843.3              -17.0
D1: 3,372 of 11,078 outside SKUs lose their location (4.6% of all pick lines)
D2: 2,606 of 11,078 outside SKUs lose their location (3.6% of all pick lines)


**7.5 Results (walking distance).**
- D1 and D2 have the same walking distance (843.3 km with E). The extra locations of a SKU are in the same aisle, so they do not change the stops. D1 and D2 differ only in refill trips (next section).
- With the baseline rule B, the new storage is **6.7% worse** (1,085 km against 1,017 km). Sorting by aisle code does not follow the new positions of the fast SKUs, so the picker walks back and forth.
- With rule E (2-opt), the new storage is **9.1% better** than E on today's storage (843 km against 927 km). The routing rule has to fit the design.
- Combined effect (E on the new storage against baseline B): **-17.0%**. Picking alone gives -8.8%, and the design gives the rest.
- The improvement is smaller than in AS2 because a batch is as long as its farthest stop, and 20% of the lines belong to SKUs that do not move.

- **Limitation.** In D1 and D2, the new locations for fast SKUs take places that other SKUs use today. These SKUs (outside the 3,500) would have to move. They are 3.6% (D2) to 4.6% (D1) of the pick lines. We left them in their old aisles, so the savings are slightly optimistic. Section 9 tests this by moving them at random.
## **8. Task 2: Labor Time with Refill Trips**

**8.1 What we add**

So far we counted only walking and picking. Now we add the time to refill the picking locations from the reserve (AS3).

1. **Refill trips.** Each trip takes 5 minutes (AS3 assumption). Today there are 881 trips. With D1 there are 301, and with D2 there are 346.
2. **Total time** = walking + picking + refill.
3. **Why 5.7 pickers and not 5.2?** The baseline now includes 73 h of refill. In Section 5 it did not.
4. **Six cases.** The baseline (B, today), picking only (E, today), and each design (D1, D2) with route B and with route E.

In [60]:
# --- WH-AS4 Check: are the extra locations of a SKU in the same aisle? ---
fast = set(item_opt.loc[item_opt['S_i'] > 1, 'sku'])        # the 843 fast SKUs
for name, n in [('D1', N_ALL), ('D2', N_TARGETED)]:
    d = scope_design(n)
    d_fast = d[d['assigned_sku'].isin(fast)].copy()
    d_fast['pt'] = list(zip(d_fast['row'], d_fast['col']))
    g = d_fast.groupby('assigned_sku')
    print(f"{name}: fast SKUs have {g.size().mean():.2f} locations and "
          f"{g['pt'].nunique().mean():.2f} different aisles on average")

# --- WH-AS4 Task 2: labor time with refill trips, all scenarios ---
REPLEN_MIN_PER_TRIP = 5     # minutes per refill trip (same assumption as AS3)
trips = {'Current storage': trips_cur,
         'D1 full slotting': trips_cur - inc['cum_saved'].iloc[N_ALL - 1],
         'D2 targeted slotting': trips_cur - inc['cum_saved'].iloc[N_TARGETED - 1]}
extra_m2 = {'Current storage': 0, 'D1 full slotting': N_ALL * area_per_location,
            'D2 targeted slotting': N_TARGETED * area_per_location}

plan = [('Baseline (B, today)',      'Current storage',      'B_m'),
        ('Picking only (E, today)',  'Current storage',      'E_m'),
        ('D1 + B (design only)',     'D1 full slotting',     'B_m'),
        ('D1 + E (combined)',        'D1 full slotting',     'E_m'),
        ('D2 + B (design only)',     'D2 targeted slotting', 'B_m'),
        ('D2 + E (combined)',        'D2 targeted slotting', 'E_m')]
rows = []
for label, storage, col in plan:
    walk_km = design_res[storage][col].sum() / 1000
    walk_h = walk_km * 1000 / SPEED_MPS / 3600
    refill_h = trips[storage] * REPLEN_MIN_PER_TRIP / 60
    total_h = walk_h + pick_h + refill_h
    rows.append({'Scenario': label,
                 'Walking (km)': walk_km, 'Walking (h)': walk_h,
                 'Picking (h)': pick_h, 'Refill (h)': refill_h,
                 'Total (h)': total_h,
                 'Pickers': total_h / pick_days / PRODUCTIVE_H_PER_DAY,
                 'Extra space (m2)': extra_m2[storage]})
t2 = pd.DataFrame(rows)
t2['Total vs baseline (%)'] = (t2['Total (h)'] / t2.loc[0, 'Total (h)'] - 1) * 100
print("\n", t2.round(1).to_string(index=False))
save_table(t2, 'as4_task2_table')

D1: fast SKUs have 3.31 locations and 1.02 different aisles on average
D2: fast SKUs have 2.16 locations and 1.01 different aisles on average

                Scenario  Walking (km)  Walking (h)  Picking (h)  Refill (h)  Total (h)  Pickers  Extra space (m2)  Total vs baseline (%)
    Baseline (B, today)        1016.6        282.4        518.1        73.4      873.9      5.7               0.0                    0.0
Picking only (E, today)         927.4        257.6        518.1        73.4      849.2      5.5               0.0                   -2.8
   D1 + B (design only)        1084.8        301.3        518.1        25.1      844.5      5.5             197.4                   -3.4
      D1 + E (combined)         843.3        234.3        518.1        25.1      777.4      5.0             197.4                  -11.0
   D2 + B (design only)        1084.8        301.3        518.1        28.8      848.2      5.5              98.7                   -2.9
      D2 + E (combined)         84

**8.2 Results (labor time)**

Each case is compared with the baseline: 874 h in 22 days (5.7 pickers).

1. **Better route only (E, today's storage).** Total time falls by 2.8% (25 h). Walking is only part of the total, so a better route alone gives a small gain.
2. **New design only (route B).** Total time falls by 3.4% (D1) and 2.9% (D2).
   - Refill time falls a lot: 73 h to 25 h (D1) and 29 h (D2).
   - Walking goes up by 6.7%. Fast SKUs stay in one aisle (1.02 aisles in D1, 1.01 in D2), so this is not about more aisles. Route B follows location codes, and in the new layout the code order no longer follows distance.
3. **New design + route E (combined).** Total time falls by 11.0% (D1) and 10.6% (D2). With D2 this is about 93 h in 22 days, and the pickers needed go from 5.7 to 5.1.
4. **Where the D2 + E saving comes from.** Half from less walking (−48 h) and half from less refill (−45 h). Picking time does not change, because the pick lines are the same.
5. **Design and routing help each other.** Route E cuts walking by 8.8% in today's storage, but by 22% in the new storage. A layout with fast SKUs near packing needs a route that looks at distance.
6. **Choice: D2.** D1 and D2 have the same walking (843 km). D1 only saves 4 h more of refill (−11.0% against −10.6%), but it uses twice the space (197 m² against 99 m²) and moves twice the SKUs. D2 is the better choice.


**Figure: labour time by case.** We split the total time of the four main cases into walking, picking and refill. The cases are the baseline, picking only, the new design with route B, and the new design with route E (D2).

In [61]:
# --- Figure for the report (AS4): labour time by case (walking, picking, refill) ---
labels = ['Baseline\n(B, today)', 'Picking only\n(E, today)', 'D2 design\n+ B', 'D2 design\n+ E']
rows_t2 = t2.set_index('Scenario').loc[['Baseline (B, today)', 'Picking only (E, today)',
                                        'D2 + B (design only)', 'D2 + E (combined)']]
fig, ax = plt.subplots(figsize=(3.5, 2.8))
bottom = np.zeros(len(rows_t2))
for col, color, name in [('Walking (h)', '#4C78A8', 'Walking'), ('Picking (h)', '#B0B0B0', 'Picking'),
                         ('Refill (h)', '#E45756', 'Refill')]:
    vals = rows_t2[col].to_numpy()
    ax.bar(range(len(rows_t2)), vals, bottom=bottom, color=color, label=name)
    for i, v in enumerate(vals):
        if v > 40:                                   # label only the segments that are big enough
            ax.text(i, bottom[i] + v / 2, f'{v:.0f}', ha='center', va='center', fontsize=6,
                    color='white' if name != 'Picking' else 'black')
    bottom += vals
base_h = rows_t2['Total (h)'].iloc[0]
for i, tot in enumerate(rows_t2['Total (h)']):
    ax.text(i, tot + 12, f'{tot:.0f} h\n({(tot / base_h - 1) * 100:+.1f}%)', ha='center', fontsize=6.5)
ax.set_xticks(range(len(rows_t2)))
ax.set_xticklabels(labels, fontsize=6.5)
ax.set_ylabel('Labour time in 22 days (h)', fontsize=8)
ax.set_ylim(0, bottom.max() * 1.4)
ax.tick_params(axis='y', labelsize=7)
ax.legend(fontsize=6.5, loc='upper center', ncol=3, frameon=False)
plt.tight_layout()
save_fig('as4_labour_by_case')
save_table(rows_t2.reset_index(), 'as4_labour_by_case')

[saved] artifacts\as4_labour_by_case.png


[saved] artifacts\as4_labour_by_case.pdf
[saved] artifacts\as4_labour_by_case.csv


C:\Users\joneo\AppData\Local\Temp\ipykernel_15512\2522381755.py:62: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Picking time (518 h in all cases, see the table above) is 59% of the baseline and does not change, because the pick lines are the same.

## **9. Sensitivity Analysis**

**9.1 What we test.** The result depends on assumptions, so we change one at a time:
- Walking speed: 0.8 and 1.4 m/s (base 1.0).
- Time per pick: 10 and 30 seconds (base 20).
- Time per refill trip: 3 and 10 minutes (base 5).
- Number of pickers: how long one day of work takes with 4 to 8 pickers.

In [62]:
# --- WH-AS4 Sensitivity 1: model parameters and number of pickers ---
dist_km = t2.set_index('Scenario')['Walking (km)']

def total_hours(walk_km, n_trips, speed, pick_s, repl_min):
    return walk_km * 1000 / speed / 3600 + n_lines * pick_s / 3600 + n_trips * repl_min / 60

cases = [('Base case',               1.0, 20, 5),
         ('Slow walking (0.8 m/s)',  0.8, 20, 5),
         ('Fast walking (1.4 m/s)',  1.4, 20, 5),
         ('Quick picking (10 s)',    1.0, 10, 5),
         ('Slow picking (30 s)',     1.0, 30, 5),
         ('Short refill (3 min)',    1.0, 20, 3),
         ('Long refill (10 min)',    1.0, 20, 10)]
rows = []
for name, sp, ps, rm in cases:
    base = total_hours(dist_km['Baseline (B, today)'], trips['Current storage'], sp, ps, rm)
    e_only = total_hours(dist_km['Picking only (E, today)'], trips['Current storage'], sp, ps, rm)
    d1e = total_hours(dist_km['D1 + E (combined)'], trips['D1 full slotting'], sp, ps, rm)
    d2e = total_hours(dist_km['D2 + E (combined)'], trips['D2 targeted slotting'], sp, ps, rm)
    rows.append({'Case': name, 'Baseline total (h)': base,
                 'E only (%)': (e_only / base - 1) * 100,
                 'D1 + E (%)': (d1e / base - 1) * 100,
                 'D2 + E (%)': (d2e / base - 1) * 100})
print("Change in total hours against the baseline of each case:")
sens4 = pd.DataFrame(rows)
print(sens4.round(1).to_string(index=False))

# Number of pickers: hours needed per picker to finish one day of work
base_h = total_hours(dist_km['Baseline (B, today)'], trips['Current storage'], 1.0, 20, 5)
d2e_h = total_hours(dist_km['D2 + E (combined)'], trips['D2 targeted slotting'], 1.0, 20, 5)
rows = [{'Pickers': p,
         'Baseline (h per picker per day)': base_h / pick_days / p,
         'D2 + E (h per picker per day)': d2e_h / pick_days / p} for p in [4, 5, 6, 7, 8]]
print(f"\nHours each picker works per day ({PRODUCTIVE_H_PER_DAY} h is the productive limit):")
print(pd.DataFrame(rows).round(1).to_string(index=False))
save_table(sens4, 'as4_sensitivity')
save_table(pd.DataFrame(rows), 'as4_pickers_load')

Change in total hours against the baseline of each case:
                  Case  Baseline total (h)  E only (%)  D1 + E (%)  D2 + E (%)
             Base case               873.9        -2.8       -11.0       -10.6
Slow walking (0.8 m/s)               944.5        -3.3       -11.5       -11.1
Fast walking (1.4 m/s)               793.2        -2.2       -10.4       -10.0
  Quick picking (10 s)               614.9        -4.0       -15.7       -15.1
   Slow picking (30 s)              1133.0        -2.2        -8.5        -8.2
  Short refill (3 min)               844.6        -2.9        -9.1        -8.9
  Long refill (10 min)               947.4        -2.6       -15.3       -14.5

Hours each picker works per day (7 h is the productive limit):
 Pickers  Baseline (h per picker per day)  D2 + E (h per picker per day)
       4                              9.9                            8.9
       5                              7.9                            7.1
       6                    

**9.2 Results.**
- **The recommendation does not change.** D2 + E saves between 8% and 15% of the total time in all cases. E alone saves between 2% and 4%.
- The design gains more when refill trips are long (10 minutes: 14.5%) or picking is fast (10 seconds: 15%). With slow picking (30 seconds) the saving falls to 8%, because picking time does not change.
- D1 and D2 are always within 1 point of each other, except with 10-minute refills (D1 is 0.8 points better).
- **Pickers:** the baseline needs 6 pickers (6.6 hours each per day). With D2 + E, 5 pickers work 7.1 hours each, slightly above the 7-hour limit. So the saving is about one picker, with a small overload.

**9.3 Outside SKUs that lose their location.** In D1 and D2, the fast SKUs take locations that SKUs outside the 3,500 use today. Those SKUs must move. If they are picked, they need a new active location, and **two SKUs cannot share one location**.

Free locations come from the 3,500 SKUs: their old locations are free when the new design does not reuse them. We give each moved SKU its own free location, and test two cases:
- **Best case:** the busiest SKUs choose first and take the closest free location.
- **Worst case:** each SKU gets a random free location (5 random draws).

SKUs outside the 3,500 with no picks in 30 days are assumed to go to reserve. The size of the stock that moves is not checked.

In [63]:
# --- WH-AS4 Sensitivity 2: outside SKUs that lose their location move to a DIFFERENT free location ---
def relocation_sets(n_extra):
    """Design d, the locations that the design takes from outside SKUs, and the locations it frees."""
    d = scope_design(n_extra)
    taken = set(zip(d['zone'], d['aisle'], d['bay'], d['level'], d['position']))
    keys = list(zip(loc_scope['zone'], loc_scope['aisle'], loc_scope['bay'],
                    loc_scope['level'], loc_scope['position']))
    is_out = ~loc_scope['sku'].isin(scope_skus).to_numpy()
    is_taken = np.array([k in taken for k in keys])
    displaced = loc_scope[is_out & is_taken]          # locations of outside SKUs that the design takes
    free = loc_scope[~is_taken & ~is_out]             # old locations of scope SKUs that the design does not reuse
    return d, displaced, free

def relocate(n_extra, mode, seed=0):
    """Move each picked outside SKU that loses its location to its OWN free location.
    mode='random'  : free locations drawn at random, no location used twice (pessimistic).
    mode='nearest' : busiest SKUs first, each takes the closest free location, none used twice (optimistic)."""
    d, displaced, free = relocation_sets(n_extra)
    disp = displaced[displaced['sku'].isin(line_cnt.index)].drop_duplicates('sku')   # only picked SKUs
    if len(disp) > len(free):
        raise ValueError(f"only {len(free)} free locations for {len(disp)} SKUs to move")
    f_rc = free[['row', 'col']].to_numpy()
    used = np.zeros(len(free), dtype=bool)
    new_pos = {}
    if mode == 'random':
        chosen = np.random.default_rng(seed).choice(len(free), size=len(disp), replace=False)
        used[chosen] = True
        for s, k in zip(disp['sku'], chosen):
            new_pos[s] = (int(f_rc[k, 0]), int(f_rc[k, 1]))
    else:
        order = disp.assign(n=disp['sku'].map(line_cnt)).sort_values('n', ascending=False)
        for s, r, c in zip(order['sku'], order['row'], order['col']):
            dist = np.abs(f_rc[:, 0] - r) + np.abs(f_rc[:, 1] - c)
            dist = np.where(used, 10**9, dist)
            k = int(dist.argmin())
            used[k] = True
            new_pos[s] = (int(f_rc[k, 0]), int(f_rc[k, 1]))
    assert used.sum() == len(disp), "a free location was used twice"

    lines = remap_lines(d)
    rr, cc = lines['row'].to_numpy().copy(), lines['col'].to_numpy().copy()
    for i, s in enumerate(lines['sku'].to_numpy()):
        pos = new_pos.get(s)
        if pos is not None:
            rr[i], cc[i] = pos
    lines['row'], lines['col'] = rr, cc
    return lines, len(disp), len(free)

def stress_lines(n_extra, seed):
    """Same name and output as before (used by the final table): the random version."""
    lines, n_disp, _ = relocate(n_extra, 'random', seed)
    return lines, n_disp

t0 = time.time()
rows = []
for name, n_extra, key in [('D1', N_ALL, 'D1 full slotting'), ('D2', N_TARGETED, 'D2 targeted slotting')]:
    ref = design_res[key]['E_m'].sum() / 1000                       # old assumption: left in the old aisle
    near_lines, n_disp, n_free = relocate(n_extra, 'nearest')
    near = evaluate(near_lines)['E_m'].sum() / 1000
    vals = [evaluate(relocate(n_extra, 'random', seed)[0])['E_m'].sum() / 1000 for seed in range(5)]
    rows.append({'Design': name, 'Picked SKUs moved': n_disp, 'Free locations': n_free,
                 'Left in old aisle (km)': ref, 'Nearest free (km)': near,
                 'Random mean (km)': np.mean(vals), 'Random min (km)': min(vals), 'Random max (km)': max(vals),
                 'Nearest vs B (%)': (near * 1000 / base_B - 1) * 100,
                 'Random vs B (%)': (np.mean(vals) * 1000 / base_B - 1) * 100})
print(f"Done in {time.time() - t0:.0f} s")
print(pd.DataFrame(rows).round(1).to_string(index=False))
save_table(pd.DataFrame(rows), 'as4_outside_skus')

Done in 19 s
Design  Picked SKUs moved  Free locations  Left in old aisle (km)  Nearest free (km)  Random mean (km)  Random min (km)  Random max (km)  Nearest vs B (%)  Random vs B (%)
    D1               1260            1434                   843.3              866.9             894.3            892.6            895.7             -14.7            -12.0
    D2                967            1635                   843.3              851.4             883.1            881.7            884.3             -16.2            -13.1
[saved] artifacts\as4_outside_skus.csv


**9.4 Results.**

1. **Enough free locations.** D1 needs to move 1,260 SKUs and has 1,434 free locations. D2 moves 967 and has 1,635. No location is used twice.
2. **Walking saving (against B, 1,017 km):** between 13.1% (random) and 16.2% (nearest free location) for D2, and between 12.0% and 14.7% for D1. If the moved SKUs stayed in their old aisle, the saving would be 17.0%, but that place is taken, so it is not possible.
3. **Total time saving:** between **9.4% and 10.4%** for D2 (about 82 to 90 hours in 22 days) and between 9.4% and 10.3% for D1. The five random draws differ by less than 4 km, so the result is stable.
4. **D2 stays our choice.** D1 and D2 save almost the same time (the difference is about 1 hour), but D2 needs half of the space and moves half of the SKUs.

## **10. Task 2: Feasibility Check**

**10.1 What we check.**
- **Space:** no new floor space. The extra locations already exist (D2: 968 locations, 99 m²; D1: 1,935 locations, 197 m²).
- **Who gives up a location:** the new slotting takes the best locations, so SKUs outside the 3,500 lose theirs. Those with no picks in 30 days (5,118 in zones B and C) go to reserve. In D1 about 2,400 units (198 cases) go back to reserve (AS3). The picked ones (967 in D2, 1,260 in D1) get a free location of their own, taken from the SKUs of the scope (section 9.3). There are enough free locations (1,635 and 1,434).
- **Order completion:** the code below checks that all pick lines and batches are still picked in each design.
- **Equipment and capacity:** same carts and same batches as today, so the load per tour does not change.
- **Replenishment:** 301 (D1) and 346 (D2) refill trips instead of 881. The reserve does not move.
- **Congestion:** the fast SKUs are now in fewer aisles. We check how many pick lines go to the busiest aisles, first with the moved SKUs left in their old aisle, then with them relocated.

In [64]:
# --- WH-AS4 Task 2 feasibility checks: coverage and aisle congestion ---
def top_aisle_share(lines, n=10):
    cnt = lines.groupby(['row', 'col']).size().sort_values(ascending=False)
    return cnt.head(n).sum() / cnt.sum(), cnt.iloc[0] / pick_days

# 1. Moved SKUs left in their old aisle (as assumed before)
rows = []
for name, lines in scenarios.items():
    share, busiest = top_aisle_share(lines)
    rows.append({'Storage': name,
                 'Lines covered': len(lines),
                 'Batches': lines['truck_nbr'].nunique(),
                 'Lines in the 10 busiest aisles (%)': share * 100,
                 'Lines per day, busiest aisle': busiest})
feas = pd.DataFrame(rows)
assert (feas['Lines covered'] == len(bc)).all(), "some pick lines were lost"
assert (feas['Batches'] == bc['truck_nbr'].nunique()).all(), "some batches were lost"
print("Moved SKUs left in their old aisle:")
print(feas.round(1).to_string(index=False))
save_table(feas, 'as4_task2_feasibility')

# 2. Moved SKUs relocated to a free location of their own
rows = []
for name, n_extra in [('D1 full slotting', N_ALL), ('D2 targeted slotting', N_TARGETED)]:
    for mode in ['nearest', 'random']:
        lines, _, _ = relocate(n_extra, mode, 0)
        assert len(lines) == len(bc), "some pick lines were lost"
        assert lines['truck_nbr'].nunique() == bc['truck_nbr'].nunique(), "some batches were lost"
        share, busiest = top_aisle_share(lines)
        rows.append({'Design': name, 'Moved SKUs go to': mode,
                     'Lines in the 10 busiest aisles (%)': share * 100,
                     'Lines per day, busiest aisle': busiest})
print("\nMoved SKUs relocated:")
print(pd.DataFrame(rows).round(1).to_string(index=False))
save_table(pd.DataFrame(rows), 'as4_task2_feasibility_relocated')

Moved SKUs left in their old aisle:
             Storage  Lines covered  Batches  Lines in the 10 busiest aisles (%)  Lines per day, busiest aisle
     Current storage          93257     2800                                24.1                         115.0
    D1 full slotting          93257     2800                                35.8                         232.4
D2 targeted slotting          93257     2800                                35.8                         232.4
[saved] artifacts\as4_task2_feasibility.csv



Moved SKUs relocated:
              Design Moved SKUs go to  Lines in the 10 busiest aisles (%)  Lines per day, busiest aisle
    D1 full slotting          nearest                                34.0                         231.6
    D1 full slotting           random                                34.0                         231.6
D2 targeted slotting          nearest                                34.6                         232.0
D2 targeted slotting           random                                34.3                         231.7
[saved] artifacts\as4_task2_feasibility_relocated.csv


**10.2 Results.**
- All 93,257 pick lines and 2,800 batches are still picked in both designs, so orders are completed.
- Congestion goes up. With the moved SKUs left in their old aisle, the 10 busiest aisles get 36% of the lines (24% today), and the busiest aisle gets 232 lines per day (115 today), about twice as many. At 20 seconds per line, this is 1.3 hours of picking per day in that aisle (18% of a work day). Pickers will sometimes wait, but the aisle is not blocked.
- D1 and D2 have the same congestion because their extra locations are in the same aisles.
- With the moved SKUs relocated (closest free location or random), congestion stays about the same: the 10 busiest aisles get 34% to 35% of the lines and the busiest aisle still gets about 232 lines per day. So the conclusion does not change.

**10.3 Not checked (limitations).**
- The real size of the locations (240 units is an estimate).
- The reserve space for the moved SKUs (conditions in AS3), and whether the stock of a moved SKU fits in its new location.
- The waiting time in busy aisles. We only count lines, and a simulation would be needed to measure it.
- The real path inside the aisles (our map has one point per aisle).
- Where the SKUs outside the 3,500 go. We tested two placements in 9.3 (closest free location and random), not an optimised one.

In [65]:
# --- WH-AS4 Final scenario table (for the report) ---
stress_km, near_km = {}, {}
for name, n_extra in [('D1 full slotting', N_ALL), ('D2 targeted slotting', N_TARGETED)]:
    stress_km[name] = np.mean([evaluate(relocate(n_extra, 'random', s)[0])['E_m'].sum() / 1000
                               for s in range(5)])
    near_km[name] = evaluate(relocate(n_extra, 'nearest')[0])['E_m'].sum() / 1000

def hours(walk_km, storage):
    return (walk_km * 1000 / SPEED_MPS / 3600 + pick_h
            + trips[storage] * REPLEN_MIN_PER_TRIP / 60)

moved = {'Current storage': 0, 'D1 full slotting': N_ALL, 'D2 targeted slotting': N_TARGETED}
plan2 = [('Baseline',             'Current storage',      'B (one pass by code)', dist_km['Baseline (B, today)']),
         ('Task 1: picking only', 'Current storage',      'E (B + 2-opt)',        dist_km['Picking only (E, today)']),
         ('Task 2: D1 + E',       'D1 full slotting',     'E (B + 2-opt)',        dist_km['D1 + E (combined)']),
         ('Task 2: D2 + E',       'D2 targeted slotting', 'E (B + 2-opt)',        dist_km['D2 + E (combined)'])]
base_total = hours(dist_km['Baseline (B, today)'], 'Current storage')

rows = []
for label, storage, rule, km in plan2:
    tot = hours(km, storage)
    row = {'Scenario': label, 'Storage': storage, 'Routing': rule,
           'SKUs moved out of active': moved[storage],
           'Extra space (m2)': extra_m2[storage],
           'Walking (km)': km, 'Refill trips': trips[storage],
           'Total (h)': tot, 'Pickers': tot / pick_days / PRODUCTIVE_H_PER_DAY,
           'Change, SKUs stay in old aisle (%)': (tot / base_total - 1) * 100}
    if storage in stress_km:
        row['Change, nearest free location (%)'] = (hours(near_km[storage], storage) / base_total - 1) * 100
        row['Change, random relocation (%)'] = (hours(stress_km[storage], storage) / base_total - 1) * 100
    rows.append(row)
final_table = pd.DataFrame(rows)
print(final_table.round(1).to_string(index=False))
save_table(final_table, 'as4_final_table')

            Scenario              Storage              Routing  SKUs moved out of active  Extra space (m2)  Walking (km)  Refill trips  Total (h)  Pickers  Change, SKUs stay in old aisle (%)  Change, nearest free location (%)  Change, random relocation (%)
            Baseline      Current storage B (one pass by code)                         0               0.0        1016.6         881.3      873.9      5.7                                 0.0                                NaN                            NaN
Task 1: picking only      Current storage        E (B + 2-opt)                         0               0.0         927.4         881.3      849.2      5.5                                -2.8                                NaN                            NaN
      Task 2: D1 + E     D1 full slotting        E (B + 2-opt)                      1935             197.4         843.3         300.6      777.4      5.0                               -11.0                              -10.3    

## 11. Conclusion (WH-AS4)

**1. Current warehouse.** Sort each pick list by location code and improve the route with 2-opt (method E). It cuts walking by 8.8% with no investment. Check how pickers walk today: if they follow the data order, the gain is much bigger.

**2. Best design: D2 + E.** Add 968 locations (99 m²) for the fast SKUs and use E routes. It saves between 9.4% and 10.4% of the total time (82 to 90 h in 22 days) and about 0.6 pickers (5.7 to about 5.1). D1 gives almost the same with twice the space.

**3. Real saving: 9% to 10%.** The SKUs that lose their location need a free location of their own. If each takes the closest free one, the total saving is 10.4%. If they move at random, it is 9.4%. The sensitivity cases (8% to 15%) use the walking distance before this relocation, so they are about 0.2 to 1.2 points too high.

**4. Both changes are needed.** The new design with a simple route saves only about 3% (before moving the outside SKUs), and better routing alone saves 2.8%. Together they save 9.4% to 10.4%.

**5. Limits.** Walking inside aisles, crowding and the real picker path are not measured. Speed, picking time and refill time are assumptions. Where the moved SKUs go is tested with two simple rules, not optimised.

# WH-AS5 — Intervention Trade-Off Analysis (Zones B & C)

**Goal.** Compare the three improvements of AS2, AS3 and AS4, alone and together, and recommend what to do.

| Decision | Baseline (today) | Proposed | From |
|---|---|---|---|
| **Storage assignment** | Every SKU stays where it is | Fast SKUs go to the locations closest to the packing area | AS2 |
| **Layout** | One active location per SKU | 968 extra locations (99 m²) for the fast SKUs | AS3 |
| **Picking** | Method B: one pass over the aisles, sorted by location code | Method E: B improved with 2-opt | AS4 |

Two options per decision give **8 scenarios**, S0 to S7. S0 is the current warehouse. The numbers are names, not a ranking. We keep one proposed option per decision. The layout is the targeted option (968 locations, called D2 in AS4). The full option (1,935 locations, D1 in AS4) only appears in the sensitivity analysis.

**Fact, assumption or result?**
- *Facts:* the pick lines, batches, SKUs, locations and the map of aisles (AS1).
- *Assumptions (the same in all scenarios):* 1.0 m/s walking, 20 s per pick line, 5 min per refill trip, 7 productive hours per picker per day, one map point per aisle, 240-unit locations.
- *Results:* walking distance, refill trips, labour hours and pickers.
- *Baseline:* B is a baseline we built (the data do not show the real path of the pickers, see AS4). Section 7 tests what happens if the real baseline is the data order (method A).

**Fair comparison.** All scenarios use the same 93,257 pick lines, the same 2,800 batches, the same 22 picking days (25 February to 25 March 2005, the same window as the 30 calendar days of AS3) and the same assumptions. Only the three decisions change.

**Compatibility.** The baseline storage and picking rule must work in the new layout, and the other way round. We checked each combination:

| Combination | How it works | Status |
|---|---|---|
| New layout on today's storage (S2, S3) | Each fast SKU keeps its location and gets extra locations, taken from SKUs with no picks, the closest free one to its aisle | Feasible |
| New storage with rule B (S4, S6) | Rule B still gives a valid route, but sorting by location code no longer follows distance | Feasible, but walks more |
| New storage (S4 to S7) | 3,498 of the 3,500 SKUs change location, and picked SKUs outside the 3,500 lose their location. Each gets its own free location (two SKUs never share one), tested in a best and a worst case | Feasible |
| All eight scenarios | None is infeasible and none is excluded | Feasible |

## 1. The 8 scenarios

The next cell lists what each scenario changes, and the common basis used by all of them.

In [66]:
# --- WH-AS5: scenario definitions and common basis ---

LAYOUT_TXT  = {0: 'Baseline (1 active location per SKU)',
               1: f'Proposed (+{N_TARGETED} locations for fast SKUs)'}
STORAGE_TXT = {0: 'Baseline (today\'s positions)',
               1: 'Proposed (AS2 optimal slotting)'}
PICKING_TXT = {0: 'Baseline B (one pass by location code)',
               1: 'Proposed E (B + 2-opt)'}
NAMES = ['Current warehouse', 'Picking change only', 'Layout change only',
         'New layout + new picking', 'Storage change only', 'New storage + new picking',
         'New layout + new storage', 'All three changes']

def decode(code):
    """Scenario number -> (storage, layout, picking), each 0 = baseline, 1 = proposed."""
    return code >> 2, (code >> 1) & 1, code & 1

scen_def = pd.DataFrame([{'Scenario': f'S{k}', 'Name': NAMES[k],
                          'Layout (AS3)': LAYOUT_TXT[decode(k)[1]],
                          'Storage assignment (AS2)': STORAGE_TXT[decode(k)[0]],
                          'Picking (AS4)': PICKING_TXT[decode(k)[2]]} for k in range(8)])
print(scen_def.to_string(index=False))

# Common basis: all scenarios use exactly the same lines, batches, days and assumptions
common = pd.DataFrame([
    ('Pick lines (zones B and C)', f"{len(bc):,}"),
    ('Batches', f"{bc['truck_nbr'].nunique():,}"),
    ('Picking days', f"{pick_days}"),
    ('SKUs in the AS2 scope', f"{len(item_opt):,}"),
    ('Walking speed (m/s)', f"{SPEED_MPS}"),
    ('Time per pick line (s)', f"{PICK_S_PER_LINE}"),
    ('Time per refill trip (min)', f"{REPLEN_MIN_PER_TRIP}"),
    ('Productive hours per picker per day', f"{PRODUCTIVE_H_PER_DAY}"),
    ('Extra locations in the proposed layout', f"{N_TARGETED}"),
    ('Random draws for the worst case (seeds)', '0 to 4'),
], columns=['Item', 'Value'])
print("\n", common.to_string(index=False))
save_table(scen_def, 'as5_scenario_definitions')
save_table(common, 'as5_common_basis')

Scenario                      Name                            Layout (AS3)        Storage assignment (AS2)                          Picking (AS4)
      S0         Current warehouse    Baseline (1 active location per SKU)    Baseline (today's positions) Baseline B (one pass by location code)
      S1       Picking change only    Baseline (1 active location per SKU)    Baseline (today's positions)                 Proposed E (B + 2-opt)
      S2        Layout change only Proposed (+968 locations for fast SKUs)    Baseline (today's positions) Baseline B (one pass by location code)
      S3  New layout + new picking Proposed (+968 locations for fast SKUs)    Baseline (today's positions)                 Proposed E (B + 2-opt)
      S4       Storage change only    Baseline (1 active location per SKU) Proposed (AS2 optimal slotting) Baseline B (one pass by location code)
      S5 New storage + new picking    Baseline (1 active location per SKU) Proposed (AS2 optimal slotting)                 P

**Results (scenarios).** S0 is the current warehouse. S1, S2 and S4 change one decision, and S3, S5, S6 and S7 combine them. All scenarios use the same lines, batches, days and assumptions, so a difference between two scenarios comes only from the decisions that are different.

## 2. The four warehouses and their walking distance

The 8 scenarios use only four different warehouses, because the picking rule (B or E) is applied afterwards: today's or new storage, each with one location per SKU or with the extra locations. Two of them already exist from AS4. The other two are new: the new layout on today's storage, and the new storage without the extra locations. The next cell builds them, measures the walking with route B and route E, and checks that no pick line or batch is lost.

In [67]:
# --- WH-AS5: build the four storage/layout designs (the picking rule is applied afterwards) ---
def layout_only_design(n_extra):
    """Proposed layout on TODAY's storage: each fast SKU keeps its current location and gets
    extra locations. The extra locations are taken from SKUs with no picks in 30 days, always
    the closest free one to the SKU's current aisle (same pool as the AS3 feasibility check)."""
    top = inc.head(n_extra)['sku'].tolist()
    cur = loc_scope[loc_scope['sku'].isin(set(top))].drop_duplicates('sku')
    cur_pos = {s: (int(r), int(c)) for s, r, c in zip(cur['sku'], cur['row'], cur['col'])}
    pool = loc_scope[loc_scope['sku'].isin(zero_pick_skus)][['row', 'col']].to_numpy()
    assert len(pool) >= n_extra, "not enough zero-pick locations for the extra locations"
    used = np.zeros(len(pool), dtype=bool)
    out = [(s, r, c) for s, (r, c) in cur_pos.items()]          # the current location is kept
    skipped = 0
    for s in top:
        if s not in cur_pos:                                      # SKU without a current location
            skipped += 1
            continue
        r, c = cur_pos[s]
        dist = np.abs(pool[:, 0] - r) + np.abs(pool[:, 1] - c)
        dist = np.where(used, 10**9, dist)
        k = int(dist.argmin())
        used[k] = True
        out.append((s, int(pool[k, 0]), int(pool[k, 1])))
    print(f"Layout only: {len(top) - skipped} extra locations placed, {skipped} skipped (no current location)")
    return pd.DataFrame(out, columns=['assigned_sku', 'row', 'col'])

t0 = time.time()
d_layout_only = layout_only_design(N_TARGETED)

# Check: for SKUs that get extra locations, is their current location the aisle where the
# trace file says they are picked? (if yes, S0 is reproduced exactly for those SKUs)
fast_cur = d_layout_only.groupby('assigned_sku').head(1).set_index('assigned_sku')
ln = bc[bc['sku'].isin(fast_cur.index)]
same = ((ln['row'].to_numpy() == ln['sku'].map(fast_cur['row']).to_numpy()) &
        (ln['col'].to_numpy() == ln['sku'].map(fast_cur['col']).to_numpy())).mean()
print(f"Lines of the fast SKUs picked in the aisle of their current location: {same:.1%}")

designs = {
    (0, 0): scenarios['Current storage'],                                   # S0 / S1
    (1, 0): remap_lines(scope_design(0)),                                   # S4 / S5
    (0, 1): remap_lines(d_layout_only),                                     # S2 / S3
    (1, 1): scenarios['D2 targeted slotting'],                              # S6 / S7
}
res5 = {(0, 0): design_res['Current storage'],
        (1, 1): design_res['D2 targeted slotting'],
        (1, 0): evaluate(designs[(1, 0)]),
        (0, 1): evaluate(designs[(0, 1)])}
print(f"Routes evaluated in {time.time() - t0:.0f} s")

# Every design must still pick every line of every batch
for key, lines in designs.items():
    assert len(lines) == len(bc), f"design {key} lost pick lines"
    assert lines['truck_nbr'].nunique() == bc['truck_nbr'].nunique(), f"design {key} lost batches"

print("\nWalking (km) by design and routing rule:")
walk4 = pd.DataFrame({f'storage={k[0]}, layout={k[1]}': {'B (one pass)': v['B_m'].sum() / 1000,
                                                       'E (B + 2-opt)': v['E_m'].sum() / 1000}
                      for k, v in res5.items()})
print(walk4.round(1).to_string())
save_table(walk4, 'as5_walking_four_designs', index=True)

Layout only: 967 extra locations placed, 1 skipped (no current location)
Lines of the fast SKUs picked in the aisle of their current location: 100.0%


Routes evaluated in 3 s

Walking (km) by design and routing rule:
               storage=0, layout=0  storage=1, layout=1  storage=1, layout=0  storage=0, layout=1
B (one pass)                1016.6               1084.8               1084.6               1121.2
E (B + 2-opt)                927.4                843.3                843.3                939.4
[saved] artifacts\as5_walking_four_designs.csv


**Results (walking distance of the four designs).**

- **Check passed.** The fast SKUs are picked in the aisle of their current location in 100% of their lines, so the current warehouse is reproduced exactly. 967 of the 968 extra locations were placed (one SKU has no current location on the map). The values of AS4 are reproduced: 1,016.6 km (B) and 927.4 km (E) today, and 1,085 km (B) and 843 km (E) with the new storage.
- **New storage only (S4, S5).** With route B the walking goes **up** 6.7%. With route E it goes **down** 17.0%. The new storage needs a route that follows distance.
- **New layout only (S2, S3).** The walking goes up: +10.3% with B (1,121 km) and +1.3% with E (939 km against 927 km). The extra locations of a SKU are in other aisles, so the picker makes more stops (in AS3 the walk per pick of the fast SKUs rose 5.5%).
- **New storage and new layout (S6, S7).** The walking is the same as with the new storage alone. The extra locations are in the same aisle as the SKU, so they add no stops.
- **Meaning.** The layout costs walking on today's storage and costs no walking on the new storage. This is a first sign that the decisions are not independent. Refill time is added later.

## 3. Resources, and what the new storage does to other SKUs

The AS2 slotting puts the fast SKUs in the best locations. Other SKUs (outside our 3,500) use some of those locations today. If they are picked, they need a new location, and **two SKUs cannot share one location**. This is the same test as in AS4 (section 9.3):
- **Best case:** the busiest SKUs choose first and take the closest free location.
- **Worst case:** each SKU gets a random free location (5 random draws, seeds 0 to 4).

Free locations are the old locations of the 3,500 SKUs that the new design does not reuse. SKUs with no picks in 30 days go to reserve. Only the new storage displaces SKUs: the new layout takes its extra locations from SKUs with no picks.

We also check how much of the AS2 space requirement (S_i locations per SKU) each layout meets.

In [68]:
# --- WH-AS5: resources and the moved SKUs (best case and worst case) ---
trips5 = {0: trips['Current storage'], 1: trips['D2 targeted slotting']}   # refill trips by layout
extra_n = {0: 0, 1: N_TARGETED}                                              # extra locations by layout

# 1. Is the AS2 space requirement (S_i locations per SKU) met?
extra_by_sku5 = inc.head(N_TARGETED).groupby('sku').size()
S_i = item_opt.set_index('sku')['S_i']
f_i = item_opt.set_index('sku')['f_i']
have = {0: pd.Series(1, index=S_i.index),
        1: 1 + extra_by_sku5.reindex(S_i.index).fillna(0)}                  # locations each SKU has
req_met = {l: have[l] >= S_i for l in (0, 1)}
req_skus = {l: req_met[l].mean() * 100 for l in (0, 1)}                     # % of SKUs
req_picks = {l: f_i[req_met[l]].sum() / f_i.sum() * 100 for l in (0, 1)}    # % of picks

# 2. How many SKUs of the scope change location with the AS2 slotting?
cur_first = loc_scope.drop_duplicates('sku')
cur_d = {s: k for s, k in zip(cur_first['sku'], zip(cur_first['zone'], cur_first['aisle'],
                              cur_first['bay'], cur_first['level'], cur_first['position']))}
d0 = scope_design(0)
new_d = {s: k for s, k in zip(d0['assigned_sku'], zip(d0['zone'], d0['aisle'], d0['bay'],
                              d0['level'], d0['position']))}
n_reassigned = sum(1 for s, k in new_d.items() if cur_d.get(s) != k)
print(f"SKUs of the scope that change location with the AS2 slotting: {n_reassigned:,} of {len(new_d):,}")
print(f"Space requirement met: {req_skus[0]:.0f}% of SKUs with 1 location per SKU, "
      f"{req_skus[1]:.0f}% with the targeted layout ({req_picks[1]:.0f}% of the picks)")

# 3. SKUs outside the scope that lose their location (only the new storage does this):
#    best case = each takes the closest free location, worst case = a random free location (5 draws)
t0 = time.time()
near5, rand5 = {}, {}
for key in [(1, 0), (1, 1)]:
    lines_n, n_moved, n_free = relocate(extra_n[key[1]], 'nearest')
    ev = evaluate(lines_n)
    near5[key] = {'B': ev['B_m'].sum() / 1000, 'E': ev['E_m'].sum() / 1000,
                  'moved': n_moved, 'free': n_free}
    b, e = [], []
    for seed in range(5):
        ev = evaluate(relocate(extra_n[key[1]], 'random', seed)[0])
        b.append(ev['B_m'].sum() / 1000)
        e.append(ev['E_m'].sum() / 1000)
    rand5[key] = {'B': float(np.mean(b)), 'E': float(np.mean(e))}
print(f"Relocation tests done in {time.time() - t0:.0f} s")
reloc5 = (pd.DataFrame({f'storage={k[0]}, layout={k[1]}': {
        'Picked SKUs to move': near5[k]['moved'], 'Free locations': near5[k]['free'],
        'Walking B, nearest (km)': near5[k]['B'], 'Walking B, random (km)': rand5[k]['B'],
        'Walking E, nearest (km)': near5[k]['E'], 'Walking E, random (km)': rand5[k]['E']}
    for k in near5}).round(1))
print(reloc5.to_string())
save_table(reloc5, 'as5_resources_moved_skus', index=True)

SKUs of the scope that change location with the AS2 slotting: 3,498 of 3,500
Space requirement met: 76% of SKUs with 1 location per SKU, 82% with the targeted layout (77% of the picks)


Relocation tests done in 25 s
                         storage=1, layout=0  storage=1, layout=1
Picked SKUs to move                    683.0                967.0
Free locations                        1927.0               1635.0
Walking B, nearest (km)               1087.7               1099.5
Walking B, random (km)                1117.8               1131.6
Walking E, nearest (km)                844.1                851.4
Walking E, random (km)                 870.4                883.1
[saved] artifacts\as5_resources_moved_skus.csv


**Results (resources and moved SKUs).**

- **Almost every SKU of the scope changes location.** 3,498 of 3,500 SKUs get a new location. The new storage means re-slotting the whole active area of zones B and C.
- **Space requirement.** 76% of the SKUs need only one location. With the targeted layout, 82% of the SKUs (77% of the picks) have the space that AS2 asks for. The rest do not get all their extra locations because we give only half of them.
- **There are enough free locations.** The new storage alone moves 683 picked SKUs and has 1,927 free locations. With the new layout it moves 967 and has 1,635 free locations: 284 more SKUs must move because the layout takes more of the best locations.
- **Where the moved SKUs go matters.** With route E the walking of the new storage goes from 844.0 km (best case) to 870.4 km (worst case), 3.1% or about 7 hours in 22 days. With the new layout it goes from 851.5 to 883.1 km (3.7%, about 9 hours).
- **The layout adds a little walking on top of the new storage:** 8 to 13 km (2 to 4 hours), small against the 45 hours of refill time that it saves. AS4 did not see this because the moved SKUs stayed in their old aisle.

## 4. Results of the 8 scenarios

Now we put everything together. For each scenario we compute walking, picking and refill hours, and the pickers needed. We show a best and a worst case for the SKUs that must move. Walking time comes from the routes, refill time from the refill trips of the layout, and picking time is the same in all scenarios. The last lines check that S0, S1, S6 and S7 give exactly the AS4 results when the moved SKUs stay in their old aisle.

In [69]:
# --- WH-AS5: results of the 8 scenarios (same assumptions, best and worst case for the moved SKUs) ---
def hours_of(walk_km, n_trips):
    return walk_km * 1000 / SPEED_MPS / 3600, pick_h, n_trips * REPLEN_MIN_PER_TRIP / 60

rows = []
for code in range(8):
    s, l, p = decode(code)
    rule = 'E' if p else 'B'
    walk_old = res5[(s, l)][rule + '_m'].sum() / 1000          # moved SKUs stay in their old aisle (AS4 assumption)
    if s:
        walk_best, walk_worst = near5[(s, l)][rule], rand5[(s, l)][rule]
    else:                                                      # nothing is displaced: best = worst
        walk_best = walk_worst = walk_old
    row = {'Scenario': f'S{code}', 'Name': NAMES[code],
           'Layout': 'P' if l else 'B', 'Storage': 'P' if s else 'B', 'Picking': 'P' if p else 'B'}
    for tag, wk in [('old aisle', walk_old), ('best', walk_best), ('worst', walk_worst)]:
        w_h, p_h, r_h = hours_of(wk, trips5[l])
        row[f'Walking, {tag} (km)'] = wk
        row[f'Total, {tag} (h)'] = w_h + p_h + r_h
    w_h, p_h, r_h = hours_of(walk_best, trips5[l])
    row.update({'Walking (h)': w_h, 'Picking (h)': p_h, 'Refill (h)': r_h,
                'Refill trips': trips5[l],
                'Extra space (m2)': extra_n[l] * area_per_location,
                'Scope SKUs reassigned': n_reassigned if s else 0,
                'Zero-pick SKUs moved to reserve': extra_n[l],
                'Picked outside SKUs displaced': near5[(s, l)]['moved'] if s else 0,
                'SKUs with space requirement met (%)': req_skus[l],
                'Picks with space requirement met (%)': req_picks[l]})
    share, busiest = top_aisle_share(designs[(s, l)])
    row.update({'Lines in 10 busiest aisles (%)': share * 100, 'Lines per day, busiest aisle': busiest})
    rows.append(row)
sc = pd.DataFrame(rows)

T0 = sc.loc[0, 'Total, best (h)']
for tag in ['best', 'worst']:
    sc[f'Change vs S0, {tag} (%)'] = (sc[f'Total, {tag} (h)'] / T0 - 1) * 100
    sc[f'Pickers, {tag}'] = sc[f'Total, {tag} (h)'] / pick_days / PRODUCTIVE_H_PER_DAY
sc['Walking vs S0, worst (%)'] = (sc['Walking, worst (km)'] / sc.loc[0, 'Walking, worst (km)'] - 1) * 100

def status(r):
    if r['Scenario'] == 'S0':
        return 'Observed baseline'
    notes = []
    if r['Picking'] == 'B' and r['Walking vs S0, worst (%)'] > 0.05:
        notes.append(f"rule B walks {r['Walking vs S0, worst (%)']:.1f}% more than S0")
    if r['Picked outside SKUs displaced'] > 0:
        notes.append(f"{int(r['Picked outside SKUs displaced'])} picked SKUs must move")
    return 'Feasible' + (': ' + '; '.join(notes) if notes else '')
sc['Status'] = sc.apply(status, axis=1)

# Table 1: performance (the walking and the totals are given for the best and the worst case of the moved SKUs)
perf = ['Scenario', 'Name', 'Walking, best (km)', 'Walking, worst (km)', 'Refill trips',
        'Total, best (h)', 'Total, worst (h)', 'Change vs S0, best (%)', 'Change vs S0, worst (%)']
print(sc[perf].round(1).to_string(index=False))

# Table 2: resources and feasibility
res_cols = ['Scenario', 'Pickers, worst', 'Extra space (m2)', 'Scope SKUs reassigned',
            'Picked outside SKUs displaced', 'SKUs with space requirement met (%)',
            'Lines per day, busiest aisle', 'Status']
print("\nResources and feasibility:")
print(sc[res_cols].round(1).to_string(index=False))

# Check: with the moved SKUs left in their old aisle, S0, S1, S6 and S7 must be the AS4 results
for code, row_t2 in [(0, 0), (1, 1), (6, 4), (7, 5)]:
    assert abs(sc.loc[code, 'Total, old aisle (h)'] - t2.loc[row_t2, 'Total (h)']) < 1e-6, f"S{code} differs from AS4"
print("\nChecks passed: S0, S1, S6 and S7 match the AS4 results (moved SKUs in their old aisle).")
save_table(sc, 'as5_scenario_results')

Scenario                      Name  Walking, best (km)  Walking, worst (km)  Refill trips  Total, best (h)  Total, worst (h)  Change vs S0, best (%)  Change vs S0, worst (%)
      S0         Current warehouse              1016.6               1016.6         881.3            873.9             873.9                     0.0                      0.0
      S1       Picking change only               927.4                927.4         881.3            849.2             849.2                    -2.8                     -2.8
      S2        Layout change only              1121.2               1121.2         345.9            858.4             858.4                    -1.8                     -1.8
      S3  New layout + new picking               939.4                939.4         345.9            807.9             807.9                    -7.6                     -7.6
      S4       Storage change only              1087.7               1117.8         881.3            893.7             902.0      

In [70]:
# --- WH-AS5: report tables for section 4 (formatted copies; units are in the headers) ---
from IPython.display import display

def span(a, b, d=0):
    """One number if best and worst case are equal after rounding, otherwise 'best to worst'."""
    return f'{a:.{d}f}' if round(a, d) == round(b, d) else f'{a:.{d}f} to {b:.{d}f}'

word = {'B': 'Baseline', 'P': 'Proposed'}
best_row = sc['Total, worst (h)'].idxmin()                      # the scenario with the lowest worst-case labour time

def style_table(df, caption, best=None):
    def mark(row):
        return ['background-color: #E8F1E4' if row.name == best else '' for _ in row]
    return (df.style.hide(axis='index').set_caption(caption).apply(mark, axis=1)
            .set_properties(**{'text-align': 'center', 'font-size': '11px'})
            .set_table_styles([{'selector': 'caption', 'props': 'font-weight: bold; font-size: 12px; text-align: left; padding-bottom: 6px;'},
                               {'selector': 'th', 'props': 'text-align: center; font-size: 11px; background-color: #F2F2F2; border-bottom: 2px solid #444;'},
                               {'selector': 'td', 'props': 'border-bottom: 1px solid #DDD; padding: 3px 8px;'}]))

# Table 1: what each scenario is, what it needs and whether it is feasible
t_def = pd.DataFrame({
    'Scenario': sc['Scenario'], 'Name': sc['Name'],
    'Layout': sc['Layout'].map(word), 'Storage': sc['Storage'].map(word), 'Picking': sc['Picking'].map(word),
    'Extra space (m²)': sc['Extra space (m2)'].map(lambda v: f'{v:.0f}'),
    'SKUs re-slotted (count)': sc['Scope SKUs reassigned'].map(lambda v: f'{v:,.0f}'),
    'Picked SKUs displaced (count)': sc['Picked outside SKUs displaced'].map(lambda v: f'{v:,.0f}'),
    'Busiest aisle (lines/day)': sc['Lines per day, busiest aisle'].map(lambda v: f'{v:.0f}'),
    'Status': sc['Status']})
display(style_table(t_def, 'Table 1. The eight scenarios: decisions, resources and feasibility', best_row))

# Table 2: performance, with the change from the current situation (S0)
t_perf = pd.DataFrame({
    'Scenario': sc['Scenario'],
    'Walking (km)': [span(a, b) for a, b in zip(sc['Walking, best (km)'], sc['Walking, worst (km)'])],
    'Refill trips (count)': sc['Refill trips'].map(lambda v: f'{v:,.0f}'),
    'Labour time (h)': [span(a, b) for a, b in zip(sc['Total, best (h)'], sc['Total, worst (h)'])],
    'Change vs S0 (%)': [span(a, b, 1) for a, b in zip(sc['Change vs S0, best (%)'], sc['Change vs S0, worst (%)'])],
    'Pickers (count)': sc['Pickers, worst'].map(lambda v: f'{v:.1f}')})
display(style_table(t_perf, 'Table 2. Performance over 22 picking days (best–worst case for the SKUs that must move)', best_row))

save_table(t_def, 'as5_table1_scenarios', caption='The eight scenarios: decisions, resources and feasibility.',
           label='tab:as5-scenarios', col_format='llcccccccl', wide=True)
save_table(t_perf, 'as5_table2_performance',
           caption='Performance over 22 picking days (best to worst case for the SKUs that must move).',
           label='tab:as5-performance')

Scenario,Name,Layout,Storage,Picking,Extra space (m²),SKUs re-slotted (count),Picked SKUs displaced (count),Busiest aisle (lines/day),Status
S0,Current warehouse,Baseline,Baseline,Baseline,0,0,0,115,Observed baseline
S1,Picking change only,Baseline,Baseline,Proposed,0,0,0,115,Feasible
S2,Layout change only,Proposed,Baseline,Baseline,99,0,0,191,Feasible: rule B walks 10.3% more than S0
S3,New layout + new picking,Proposed,Baseline,Proposed,99,0,0,191,Feasible
S4,Storage change only,Baseline,Proposed,Baseline,0,"3,498",683,236,Feasible: rule B walks 9.9% more than S0; 683 picked SKUs must move
S5,New storage + new picking,Baseline,Proposed,Proposed,0,"3,498",683,236,Feasible: 683 picked SKUs must move
S6,New layout + new storage,Proposed,Proposed,Baseline,99,"3,498",967,232,Feasible: rule B walks 11.3% more than S0; 967 picked SKUs must move
S7,All three changes,Proposed,Proposed,Proposed,99,"3,498",967,232,Feasible: 967 picked SKUs must move


Scenario,Walking (km),Refill trips (count),Labour time (h),Change vs S0 (%),Pickers (count)
S0,1017,881,874,0.0,5.7
S1,927,881,849,-2.8,5.5
S2,1121,346,858,-1.8,5.6
S3,939,346,808,-7.6,5.2
S4,1088 to 1118,881,894 to 902,2.3 to 3.2,5.9
S5,844 to 870,881,826 to 833,-5.5 to -4.6,5.4
S6,1100 to 1132,346,852 to 861,-2.5 to -1.4,5.6
S7,851 to 883,346,783 to 792,-10.4 to -9.4,5.1


[saved] artifacts\as5_table1_scenarios.csv
[saved] artifacts\as5_table1_scenarios.tex
[saved] artifacts\as5_table2_performance.csv
[saved] artifacts\as5_table2_performance.tex


**Results (the 8 scenarios).**

- **Check passed.** The new numbers (best and worst case) are slightly higher than in AS4 because the moved SKUs now get a real free location.
- **Best scenario: S7 (all three changes).** It needs 784 h (best case) to 792 h (worst case) instead of 873.9 h today: **10.4% to 9.4% less** labour, and 5.1 pickers instead of 5.7. The range between best and worst case is only 9 h.
- **Each change alone gives little.** Picking alone (S1) saves 2.8%. The layout alone (S2) saves 1.8%: it cuts refill time by 45 h but adds 105 km of walking.
- **Negative result: the new storage alone (S4) is worse than today** (+2.3% to +3.2%). Supported by the data: with route B the walking increases (+7% to +10%), and the storage change does not reduce refill. Explanation (from AS4): sorting by location code does not follow the new positions, so the picker walks back and forth. With route E (S5) the storage change helps (-5.5% to -4.6%).
- **S3 is a simpler option.** The new layout with the new picking rule (S3) saves 7.6% and needs no re-slotting. It is 16 to 24 h behind S7 (1.8 to 2.8 points). Those points need 3,498 SKUs to change location and 967 other picked SKUs to move.
- **Congestion trade-off.** The new storage puts the fast SKUs together: the busiest aisle goes from 115 lines per day (S0) to 232–236 (S4 to S7), and the 10 busiest aisles hold 36% of the lines instead of 24%. The new layout alone raises the busiest aisle to 191 (S2, S3). AS1 suggested spreading the traffic, so the two goals conflict. The walking model has no queues, so the real gain of S4 to S7 may be smaller.
- **Feasibility.** All scenarios are feasible (none excluded). The layout needs only 99 m².
- **Why AS2 and AS3 show bigger numbers.** AS2 counts every pick line as a separate trip from the packing area (4,463 km against 1,017 km of real walking), and AS3 measures only the average walk of the fast SKUs. AS5 uses the real batch routes of all lines, so these are the numbers to use in the report. For the layout alone, AS3 gives a break-even of 1.9 min and a net gain of 28 h. AS5 gives 3.3 min and 16 h.

## 5. Effect of each decision, alone and together

Each decision can be switched on in four settings, depending on the other two decisions. Comparing the four shows if the effect is stable or depends on the context.

We also test the **interaction**. If the decisions did not influence each other, the combined effect (S7) would be the sum of the three isolated effects (S1, S2, S4). The difference is the interaction. A negative value means the decisions help each other.

In [71]:
# --- WH-AS5: effect of each decision, alone and in the other settings ---
T_b = sc['Total, best (h)'].to_dict()      # best case for the moved SKUs
T_w = sc['Total, worst (h)'].to_dict()     # worst case for the moved SKUs
W_b = sc['Walking (h)'].to_dict()          # walking hours (best case)
R_b = sc['Refill (h)'].to_dict()           # refill hours

pairs = {'picking': [(0, 1), (2, 3), (4, 5), (6, 7)],
         'layout':  [(0, 2), (1, 3), (4, 6), (5, 7)],
         'storage': [(0, 4), (1, 5), (2, 6), (3, 7)]}

rows = []
for dec, prs in pairs.items():
    for off, on in prs:
        rows.append({'Decision': dec, 'Comparison': f'S{off} -> S{on}',
                     'Change in total, best (h)': T_b[on] - T_b[off],
                     'Change in total, worst (h)': T_w[on] - T_w[off],
                     'Change in walking, best (h)': W_b[on] - W_b[off],
                     'Change in refill (h)': R_b[on] - R_b[off]})
marg = pd.DataFrame(rows)            # all 12 comparisons (kept for the report)

# Table 1: effect of each decision, over the four settings of the other two decisions
summary = marg.groupby('Decision', sort=False).agg(
    best_min=('Change in total, best (h)', 'min'), best_max=('Change in total, best (h)', 'max'),
    worst_min=('Change in total, worst (h)', 'min'), worst_max=('Change in total, worst (h)', 'max'),
    walking_min=('Change in walking, best (h)', 'min'), walking_max=('Change in walking, best (h)', 'max'),
    refill=('Change in refill (h)', 'mean'))
summary.columns = ['Best case, smallest change (h)', 'Best case, largest change (h)',
                   'Worst case, smallest change (h)', 'Worst case, largest change (h)',
                   'Walking, smallest change (h)', 'Walking, largest change (h)', 'Refill change (h)']
print("Change in total labour time when one decision is switched on (4 settings of the other two):")
print(summary.round(1).to_string())

# Table 2: interaction = combined effect minus the sum of the three isolated effects
inter = pd.DataFrame({tag: {'Picking only (S1)': T[1] - T[0],
                            'Layout only (S2)': T[2] - T[0],
                            'Storage only (S4)': T[4] - T[0],
                            'Sum of the three isolated effects': (T[1] - T[0]) + (T[2] - T[0]) + (T[4] - T[0]),
                            'Combined effect (S7)': T[7] - T[0],
                            'Interaction (combined - sum)': (T[7] - T[0]) - ((T[1] - T[0]) + (T[2] - T[0]) + (T[4] - T[0]))}
                      for tag, T in [('Best case (h)', T_b), ('Worst case (h)', T_w)]})
print("\nInteraction (change in total labour time vs S0):")
print(inter.round(1).to_string())
save_table(marg, 'as5_marginal_effects')

Change in total labour time when one decision is switched on (4 settings of the other two):
          Best case, smallest change (h)  Best case, largest change (h)  Worst case, smallest change (h)  Worst case, largest change (h)  Walking, smallest change (h)  Walking, largest change (h)  Refill change (h)
Decision                                                                                                                                                                                                              
picking                            -68.9                          -24.8                            -69.1                           -24.8                         -68.9                        -24.8                0.0
layout                             -42.6                          -15.6                            -41.3                           -15.6                           2.0                         29.1              -44.6
storage                            -24.4        

In [72]:
# --- WH-AS5: report tables for section 5 ---
from IPython.display import display

# Section 5: effect of each decision, and the interaction
t_eff = summary.round(1).reset_index().rename(columns={'Decision': 'Decision switched on'})
t_eff['Decision switched on'] = t_eff['Decision switched on'].str.capitalize()
t_eff.columns = [c.replace('smallest change', 'min').replace('largest change', 'max') for c in t_eff.columns]
display(t_eff.style.hide(axis='index').format({c: '{:+.1f}' for c in t_eff.columns[1:]})
        .set_caption('Table 3. Change in labour time (h) when one decision is switched on, over the four settings of the other two')
        .set_properties(**{'text-align': 'center', 'font-size': '11px'})
        .set_table_styles([{'selector': 'caption', 'props': 'font-weight: bold; font-size: 12px; text-align: left; padding-bottom: 6px;'},
                           {'selector': 'th', 'props': 'text-align: center; font-size: 11px; background-color: #F2F2F2; border-bottom: 2px solid #444;'}]))

t_int = inter.round(1).reset_index().rename(columns={'index': 'Effect (change vs S0)'})
last = len(t_int) - 1
display(t_int.style.hide(axis='index').format({c: '{:+.1f}' for c in t_int.columns[1:]})
        .apply(lambda r: ['font-weight: bold; background-color: #E8F1E4' if r.name == last else '' for _ in r], axis=1)
        .set_caption('Table 4. Interaction: the combined effect is much larger than the sum of the isolated effects (h)')
        .set_properties(**{'font-size': '11px'})
        .set_table_styles([{'selector': 'caption', 'props': 'font-weight: bold; font-size: 12px; text-align: left; padding-bottom: 6px;'},
                           {'selector': 'th', 'props': 'font-size: 11px; background-color: #F2F2F2; border-bottom: 2px solid #444;'}]))

save_table(summary.reset_index(), 'as5_effects_summary')
save_table(t_eff, 'as5_table3_effects',
           caption='Change in labour time (h) when one decision is switched on, over the four settings of the other two.',
           label='tab:as5-effects', wide=True)
save_table(t_int, 'as5_table4_interaction',
           caption='Interaction: the combined effect is much larger than the sum of the isolated effects (h).',
           label='tab:as5-interaction')


Decision switched on,"Best case, min (h)","Best case, max (h)","Worst case, min (h)","Worst case, max (h)","Walking, min (h)","Walking, max (h)",Refill change (h)
Picking,-68.9,-24.8,-69.1,-24.8,-68.9,-24.8,+0.0
Layout,-42.6,-15.6,-41.3,-15.6,+2.0,+29.1,-44.6
Storage,-24.4,+19.7,-15.8,+28.1,-24.4,+19.7,+0.0


Effect (change vs S0),Best case (h),Worst case (h)
Picking only (S1),-24.8,-24.8
Layout only (S2),-15.6,-15.6
Storage only (S4),+19.7,+28.1
Sum of the three isolated effects,-20.6,-12.2
Combined effect (S7),-90.5,-81.7
Interaction (combined - sum),-69.9,-69.5


[saved] artifacts\as5_effects_summary.csv
[saved] artifacts\as5_table3_effects.csv
[saved] artifacts\as5_table3_effects.tex
[saved] artifacts\as5_table4_interaction.csv
[saved] artifacts\as5_table4_interaction.tex


**Results (effect of each decision).**

- **Picking (E instead of B) always helps, and more with the new storage.** It saves 25 h with today's storage and up to 69 h with the new storage and layout. Route E needs the new storage to show its value.
- **The layout always saves 44.6 h of refill time** (it depends only on the layout). What changes is the walking it costs: +29 h with route B on today's storage, only +2 to +4 h in the other settings. So the net gain is 16 h in the worst setting and 41 to 43 h in the others.
- **Storage depends most on the context.** In the best case it costs +20 h with route B on today's layout, and saves 24 h with route E on the new layout. In the worst case it goes from +28 h to -16 h.
- **Strong positive interaction.** The three isolated effects add up to only -21 h (best case) or -12 h (worst case), but the three changes together save 90 h or 82 h. The interaction is **-70 h**. Each decision removes a problem that limits another one.
- **Meaning.** We cannot choose the decisions one by one. The storage change alone looks bad (+20 h), but it is part of the best scenario. The decisions must be judged as a package.

## 6. Figure: labour time by scenario

The figure splits the labour time of each scenario into walking, picking and refill (best case). The black diamond is the total in the worst case, and the number above is the change vs S0 (best case).

In [73]:
# --- WH-AS5: figure for the report ---
import textwrap
colors = {'Walking (h)': '#4C78A8', 'Picking (h)': '#B0B0B0', 'Refill (h)': '#E45756'}
x = np.arange(8)

fig, ax = plt.subplots(figsize=(7.6, 3.8))
bottom = np.zeros(8)
for c in ['Walking (h)', 'Picking (h)', 'Refill (h)']:
    ax.bar(x, sc[c], bottom=bottom, color=colors[c], label=c.replace(' (h)', ''))
    if c != 'Refill (h)':                                   # hours inside the walking and picking parts
        for i in range(8):
            ax.text(i, bottom[i] + sc[c].iloc[i] / 2, f'{sc[c].iloc[i]:.0f}', ha='center', va='center',
                    fontsize=6.5, color='white' if c == 'Walking (h)' else 'black')
    bottom += sc[c].to_numpy()
ax.scatter(x, sc['Total, worst (h)'], color='black', marker='D', s=14, zorder=3, label='Worst case (total)')

# Above each bar: total hours (best case, and worst case if different) and the change vs S0 (best case)
for i in range(8):
    b, w = sc['Total, best (h)'].iloc[i], sc['Total, worst (h)'].iloc[i]
    total_txt = f'{b:.0f} h' if abs(w - b) < 0.5 else f'{b:.0f}-{w:.0f} h'
    ax.text(i, w + 12, f"{total_txt}\n{sc['Change vs S0, best (%)'].iloc[i]:+.1f}%", ha='center', va='bottom',
            fontsize=6.5, linespacing=1.2)

ax.set_xticks(x)
ax.set_xticklabels(['S%d\n%s\nrefill %.0f h' % (k, textwrap.fill(NAMES[k], 12), sc['Refill (h)'].iloc[k])
                    for k in range(8)], fontsize=6.3)
ax.set_ylabel('Labour time in 22 days (h)', fontsize=8)
ax.set_ylim(0, max(bottom.max(), sc['Total, worst (h)'].max()) * 1.25)
ax.tick_params(axis='y', labelsize=7)
ax.legend(fontsize=6.5, loc='upper center', ncol=4, frameon=False)
plt.tight_layout()
save_fig('as5_labour_by_scenario')
save_table(sc[['Scenario', 'Name', 'Walking (h)', 'Picking (h)', 'Refill (h)', 'Total, best (h)', 'Total, worst (h)', 'Change vs S0, best (%)']], 'as5_figure_data')

[saved] artifacts\as5_labour_by_scenario.png
[saved] artifacts\as5_labour_by_scenario.pdf
[saved] artifacts\as5_figure_data.csv


C:\Users\joneo\AppData\Local\Temp\ipykernel_15512\2522381755.py:62: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Results (figure).**

- **Picking time is fixed.** The grey part (518 h) is the same in all scenarios because the pick lines do not change. It is about 60% of the total, so only walking and refill can be reduced. This is why the best scenario saves only about 10%.
- **Walking (blue) and refill (red) move in different ways.** The layout scenarios (S2, S3, S6, S7) almost remove the red part (73 h to 29 h). The new storage with route E (S5, S7) reduces the blue part. Route B with the new layout or storage (S2, S4, S6) makes it bigger.
- **S4 is the only scenario above today (+2.3%). S7 is the lowest bar (-10.4%), and S3 is the next best (-7.6%).**
- **Best and worst case are close.** The diamonds are only slightly above the bars, and only in the scenarios with new storage (S4 to S7).

## 7. Sensitivity to the model parameters

The results depend on the walking speed, the time per picked line and the time per refill trip. We change one at a time and check whether the best scenario changes. We also add two things:
- the full layout (D1, 1,935 locations) for S7, and
- a row where **today's picking is the data order (method A)** instead of method B. This is the biggest uncertainty about the baseline: the data do not show how pickers really walk.

The table shows the worst case for the moved SKUs. At the end we compute the **break-even** refill time: the time above which the layout change pays off.

In [74]:
# --- WH-AS5: sensitivity to the model parameters (all scenarios, worst case for the moved SKUs) ---
trip_v = sc['Refill trips'].to_numpy()
full_D1 = {'best': (near_km['D1 full slotting'], trips['D1 full slotting']),     # full layout, from the AS4 table
           'worst': (stress_km['D1 full slotting'], trips['D1 full slotting'])}
walk_A = res['hist_m'].sum() / 1000          # AS4 method A: pick lines in the order of the data file

cases5 = [('Base case', 1.0, 20, 5), ('Slow walking (0.8 m/s)', 0.8, 20, 5),
          ('Fast walking (1.4 m/s)', 1.4, 20, 5), ('Quick picking (10 s)', 1.0, 10, 5),
          ('Slow picking (30 s)', 1.0, 30, 5), ('Short refill (3 min)', 1.0, 20, 3),
          ('Long refill (10 min)', 1.0, 20, 10), ('Very long refill (20 min)', 1.0, 20, 20)]

def sens_table(tag, with_A=False):
    walks = sc[f'Walking, {tag} (km)'].to_numpy()
    rows = []
    cases = cases5 + ([('Baseline A: pickers follow the data order today', 1.0, 20, 5)] if with_A else [])
    for name, sp, ps, rm in cases:
        tot = np.array([total_hours(walks[k], trip_v[k], sp, ps, rm) for k in range(8)])
        if name.startswith('Baseline A'):                       # S0 is replaced by method A
            tot[0] = total_hours(walk_A, trip_v[0], sp, ps, rm)
        d = (tot / tot[0] - 1) * 100
        r = {'Case': name, 'S0 total (h)': tot[0]}
        r.update({f'S{k} (%)': d[k] for k in range(1, 8)})
        r['S7 with full layout (%)'] = (total_hours(full_D1[tag][0], full_D1[tag][1], sp, ps, rm) / tot[0] - 1) * 100
        r['Best scenario'] = f'S{int(tot.argmin())}'
        rows.append(r)
    return pd.DataFrame(rows)

sens_worst = sens_table('worst', with_A=True)
sens_best = sens_table('best', with_A=True)
print("Change in total hours vs S0, worst case for the moved SKUs:")
print(sens_worst.round(1).to_string(index=False))
s7 = pd.concat([sens_best['S7 (%)'], sens_worst['S7 (%)']])
s7_main = pd.concat([sens_best['S7 (%)'][:-1], sens_worst['S7 (%)'][:-1]])
print(f"\nS7 vs S0, 8 parameter cases (best and worst case): {s7_main.max():.1f}% to {s7_main.min():.1f}%")
print(f"Best scenario in all {len(sens_worst) * 2} cases (best and worst case): "
      f"{', '.join(sorted(set(sens_best['Best scenario']) | set(sens_worst['Best scenario'])))}")

# Break-even: how long must a refill trip take for the layout change to pay off?
print("\nBreak-even refill time for the layout change (best case for the moved SKUs):")
for off, on in [(0, 2), (1, 3), (4, 6), (5, 7)]:
    d_walk_h = sc.loc[on, 'Walking (h)'] - sc.loc[off, 'Walking (h)']
    d_trips = trip_v[off] - trip_v[on]
    if d_walk_h <= 0:
        print(f"  S{off} -> S{on}: the layout also reduces walking, so it pays off for any refill time.")
    else:
        print(f"  S{off} -> S{on}: walking +{d_walk_h:.1f} h, trips -{d_trips:.0f}: "
              f"pays off if a refill trip takes more than {d_walk_h * 60 / d_trips:.1f} min (base: {REPLEN_MIN_PER_TRIP}).")

be_rows = []
for off, on in [(0, 2), (1, 3), (4, 6), (5, 7)]:
    d_walk_h = sc.loc[on, 'Walking (h)'] - sc.loc[off, 'Walking (h)']
    d_trips = trip_v[off] - trip_v[on]
    be_rows.append({'Comparison': f'S{off} -> S{on}', 'Extra walking (h)': d_walk_h, 'Refill trips saved (count)': d_trips,
                    'Break-even refill time (min)': max(d_walk_h, 0) * 60 / d_trips})
save_table(sens_worst, 'as5_sensitivity_worst')
save_table(sens_best, 'as5_sensitivity_best')
save_table(pd.DataFrame(be_rows), 'as5_break_even')

Change in total hours vs S0, worst case for the moved SKUs:
                                           Case  S0 total (h)  S1 (%)  S2 (%)  S3 (%)  S4 (%)  S5 (%)  S6 (%)  S7 (%)  S7 with full layout (%) Best scenario
                                      Base case         873.9    -2.8    -1.8    -7.6     3.2    -4.6    -1.4    -9.4                     -9.4            S7
                         Slow walking (0.8 m/s)         944.5    -3.3    -0.9    -7.6     3.7    -5.4    -0.5    -9.6                     -9.6            S7
                         Fast walking (1.4 m/s)         793.2    -2.2    -3.0    -7.6     2.5    -3.7    -2.7    -9.0                     -9.2            S7
                           Quick picking (10 s)         614.9    -4.0    -2.5   -10.7     4.6    -6.6    -2.1   -13.3                    -13.4            S7
                            Slow picking (30 s)        1133.0    -2.2    -1.4    -5.8     2.5    -3.6    -1.1    -7.2                     -7.3            S

In [75]:
# --- WH-AS5: report table for section 7 ---
from IPython.display import display

# Sensitivity (worst case); the colour shows the size of the saving, the S0 total is in hours
pct_cols = [c for c in sens_worst.columns if c.endswith('(%)')]
display(sens_worst.style.hide(axis='index')
        .format({'S0 total (h)': '{:,.0f}', **{c: '{:+.1f}' for c in pct_cols}})
        .background_gradient(subset=pct_cols[:-1], cmap='RdYlGn_r', axis=None, vmin=-25, vmax=5)
        .set_caption('Table 5. Sensitivity: change in labour time vs S0 (%), worst case for the SKUs that must move')
        .set_properties(**{'text-align': 'center', 'font-size': '11px'})
        .set_table_styles([{'selector': 'caption', 'props': 'font-weight: bold; font-size: 12px; text-align: left; padding-bottom: 6px;'},
                           {'selector': 'th', 'props': 'text-align: center; font-size: 11px; background-color: #F2F2F2; border-bottom: 2px solid #444;'}]))

save_table(sens_worst, 'as5_table5_sensitivity',
           caption='Sensitivity: change in labour time vs S0 (%), worst case for the SKUs that must move.',
           label='tab:as5-sensitivity', wide=True)

Case,S0 total (h),S1 (%),S2 (%),S3 (%),S4 (%),S5 (%),S6 (%),S7 (%),S7 with full layout (%),Best scenario
Base case,874,-2.8,-1.8,-7.6,+3.2,-4.6,-1.4,-9.4,-9.4,S7
Slow walking (0.8 m/s),945,-3.3,-0.9,-7.6,+3.7,-5.4,-0.5,-9.6,-9.6,S7
Fast walking (1.4 m/s),793,-2.2,-3.0,-7.6,+2.5,-3.7,-2.7,-9.0,-9.2,S7
Quick picking (10 s),615,-4.0,-2.5,-10.7,+4.6,-6.6,-2.1,-13.3,-13.4,S7
Slow picking (30 s),"1,133",-2.2,-1.4,-5.8,+2.5,-3.6,-1.1,-7.2,-7.3,S7
Short refill (3 min),845,-2.9,+0.3,-5.7,+3.3,-4.8,+0.6,-7.6,-7.5,S7
Long refill (10 min),947,-2.6,-6.4,-11.7,+3.0,-4.3,-6.0,-13.3,-13.8,S7
Very long refill (20 min),"1,094",-2.3,-13.7,-18.3,+2.6,-3.7,-13.4,-19.7,-20.8,S7
Baseline A: pickers follow the data order today,"1,341",-36.7,-36.0,-39.8,-32.8,-37.9,-35.8,-40.9,-41.0,S7


[saved] artifacts\as5_table5_sensitivity.csv
[saved] artifacts\as5_table5_sensitivity.tex


**Results (sensitivity).**

- **S7 is the best scenario in every case.** With the other parameters changed, S7 saves between 7% and 20% (best and worst case). It saves less with slow picking (30 s) or quick refill (3 min), because walking and refill are then a smaller part of the total, and more with long refill trips.
- **S4 (new storage alone) is above today in every case** (+2.5% to +4.6% in the worst case).
- **The layout alone depends on the refill time.** S2 saves nothing with 3 min per refill trip (+0.3%) and 13.7% with 20 min. The break-even is **3.3 min**, close to our 5 min. With route E or with the new storage (S3, S6, S7) the break-even is only 0.2 to 0.4 min, so the layout should not go alone.
- **S3 stays close to S7.** In the worst case S7 is only 1.4 to 2.6 points better than S3.
- **The full layout adds almost nothing.** S7 with 1,935 locations is within 0.1 points of S7 with 968 in the base case, and at most 1.1 points better (20 min per refill trip).
- **If pickers follow the data order today (method A),** the baseline is much worse (about 1,341 h). Then picking alone (S1) saves about 37% and S7 about 41%. The recommendation does not change, and the routing rule becomes the most valuable part.

## 8. Ranking and recommendation

We rank the scenarios by labour time in the **worst case**, so the recommendation does not depend on lucky placements of the moved SKUs. Among the scenarios within 1% of the best, we would prefer the one with the fewest changes. The cell also prints the second scenario, to compare the effort.

In [76]:
# --- WH-AS5: ranking and recommendation (computed from the evidence above) ---
rank = sc[['Scenario', 'Name', 'Total, best (h)', 'Total, worst (h)', 'Pickers, worst',
           'Extra space (m2)', 'Scope SKUs reassigned', 'Picked outside SKUs displaced']].copy()
rank['Worst vs S0 (%)'] = (rank['Total, worst (h)'] / T0 - 1) * 100
rank['Changes made'] = sc[['Layout', 'Storage', 'Picking']].eq('P').sum(axis=1)
rank = rank.sort_values('Total, worst (h)').reset_index(drop=True)
print(rank.round(1).to_string(index=False))

best = rank.iloc[0]
close = rank[rank['Total, worst (h)'] <= best['Total, worst (h)'] * 1.01]     # within 1% of the best
simplest = close.sort_values(['Changes made', 'Total, worst (h)']).iloc[0]
print(f"\nBest scenario by worst-case labour time: {best['Scenario']} ({best['Name']}), "
      f"{best['Worst vs S0 (%)']:+.1f}% vs S0.")
print(f"Scenarios within 1% of it: {', '.join(close['Scenario'])}.")
second = rank.iloc[1]
print(f"Second: {second['Scenario']} ({second['Name']}), {second['Worst vs S0 (%)']:+.1f}% vs S0, "
      f"{second['Total, worst (h)'] - best['Total, worst (h)']:.1f} h behind in the worst case, "
      f"{int(second['Scope SKUs reassigned'])} scope SKUs reassigned, "
      f"{int(second['Picked outside SKUs displaced'])} picked SKUs displaced.")
print(f"Pickers (worst case): S0 {sc.loc[0, 'Pickers, worst']:.1f} -> "
      f"{best['Scenario']} {best['Pickers, worst']:.1f}.")
save_table(rank, 'as5_ranking')

Scenario                      Name  Total, best (h)  Total, worst (h)  Pickers, worst  Extra space (m2)  Scope SKUs reassigned  Picked outside SKUs displaced  Worst vs S0 (%)  Changes made
      S7         All three changes            783.4             792.2             5.1              98.7                   3498                            967             -9.4             3
      S3  New layout + new picking            807.9             807.9             5.2              98.7                      0                              0             -7.6             2
      S5 New storage + new picking            826.0             833.3             5.4               0.0                   3498                            683             -4.6             2
      S1       Picking change only            849.2             849.2             5.5               0.0                      0                              0             -2.8             1
      S2        Layout change only            858.4    

**Results (ranking).**

- **S7 is first by worst-case labour time:** 792 h instead of 873.9 h (-9.4%, about 82 h in 22 days or 3.7 h per day), and 5.1 pickers instead of 5.7. No other scenario is within 1% of it.
- **S3 is second and has no uncertainty:** 808 h (-7.6%, 66 h saved, 5.2 pickers). It keeps 73% (best case) to 81% (worst case) of the S7 saving. It has the same layout and picking rule as S7, but it does not re-slot any SKU and does not move other SKUs.
- **The last step costs more than it gives.** From S3 to S7 we gain 16 to 24 h (1.8 to 2.8 points) and must change 3,498 SKUs and move 967 other picked SKUs.
- **Avoid:** S4 (storage alone) is the only scenario worse than today (+3.2% in the worst case). S6 (new layout and storage with route B) and S2 (layout alone) save less than 2%.

## Management recommendation

**Target: S7 (new layout, new storage and route E), reached in two phases.** It is the best scenario in every sensitivity case, with 9.4% to 10.4% less labour (792 to 784 h instead of 874 h in 22 days).

**Phase 1: S3 (layout + route E).** Add 968 locations (99 m²) for the fastest SKUs and change the picking rule from B to E (B + 2-opt). It saves 7.6% (66 h in 22 days) and refill trips go from 881 to 346. It needs no re-slotting and does not displace other SKUs, and its result does not depend on where moved SKUs are placed.

**Phase 2: new storage (S7).** Re-slot the active area of zones B and C only if Phase 1 works as expected. It adds 16 to 24 h (1.8 to 2.8 points), changes 3,498 SKUs and moves 683 to 967 other picked SKUs.

**Never do the storage change alone or with route B.** S4 is worse than today in every case. The layout alone (S2) pays off only if a refill trip takes more than 3.3 min.

**Risks and limitations.**
- *Congestion.* The new storage doubles the lines per day in the busiest aisle (115 today, 232 in S7; 191 already with the new layout). The model has no queues, so the real gain may be smaller. This is the main reason to check Phase 1 before Phase 2.
- *Baseline.* B is a baseline we built. If pickers follow the data order today, the savings are much larger (S7 about 41%).
- *Assumed times.* The saving is 7% to 20% depending on the picking time (10 to 30 s per line) and the refill time (3 to 20 min per trip). These two times should be measured before the decision.
- *Moved SKUs.* Where the displaced SKUs go changes the result by about 9 h, and the stock that moves with them was not checked.
- *Service.* We did not measure stockouts. Fewer refill trips and more active locations should help the Class A stockouts of AS1, but this is a hypothesis.
- *Labour.* 0.6 pickers fewer in S7 (0.5 in S3) is a reduction of labour hours, not necessarily of headcount.

**What to monitor after each phase:** refill trips per day (target about 12, today 29), walking distance per pick line, lines per day in the busiest aisle, measured time per pick line and per refill trip, and picker hours per day against the model (S3: 808 h and S7: 792 h per 22 days).

**Summary — WH-AS5**
- Eight scenarios from three decisions, compared with the same lines, batches, days and assumptions.
- Best: S7, with 9.4% to 10.4% less labour and 0.6 pickers fewer. Second: S3, with 7.6% and no uncertainty.
- The decisions help each other: the isolated effects add up to -21 h, the combination saves 90 h.
- Negative result: the new storage alone is worse than today (+2.3% to +3.2%).
- The result holds in all sensitivity cases (saving 7% to 20%), and the main risks are congestion and the unknown real baseline.
- Recommendation: do S3 first, then the new storage if the pilot works.

## 9. Tables for the report

Short tables used in the report (settings, KPIs, checks, options, methods, scenarios). They are built from the files saved above.

In [77]:
# --- Report tables for AS1 to AS4 (built from the evidence saved above) ---------------------
def load(name):
    return pd.read_csv(ARTIFACTS / f"{name}.csv")

# Table: settings and assumptions
assump = pd.DataFrame([
    ('Picking period (zones B and C)', '25 Feb to 25 Mar 2005, 22 picking days', 'data'),
    ('Pick lines and batches', '93,257 lines in 2,800 batches', 'data'),
    ('Walking speed', '1.0 m/s', 'assumption'),
    ('Time to find, pick and scan one line', '20 s', 'assumption'),
    ('Time per refill trip', '5 min', 'assumption'),
    ('Productive time per picker', '7 h per day', 'assumption'),
    ('Distance', 'rectilinear on the map grid, 1 square = 1.07 m, one point per aisle', 'assumption'),
    ('Packing area (start and end of tours)', 'grid point (row 7, column 10)', 'assumption'),
    ('Capacity of one active location', '240 units (75th percentile of Max)', 'estimate'),
    ('Cap on Max', '2,200 units (99th percentile)', 'data rule'),
    ('Random draws for relocated SKUs', 'seeds 0 to 4 (worst case)', 'setting'),
], columns=['Parameter', 'Value', 'Basis'])
save_table(assump, 'report_assumptions', caption='Settings and assumptions used in all comparisons.',
           label='tab:assumptions', col_format='p{3.3cm}p{3.7cm}l')

# Table: the five KPIs of AS1
kpi = load('kpi_summary')
note = {'KPI1 Time': 'not a real lead time (files from different years)',
        'KPI2 Quality': '93% of the ordered quantity is shipped',
        'KPI3 Quality': 'about 1 line in 14 is short-shipped',
        'KPI4 Productivity': 'all pickers together, varies from 1 to 21,569',
        'KPI5 Productivity': 'all pickers together, varies from 1 to 3,998'}
kpi_t = pd.DataFrame({'KPI': kpi['Name'], 'Value': kpi['Value'].map(lambda v: f'{v:,.1f}'),
                      'Unit': kpi['Unit'], 'Reading': kpi['KPI'].map(note)})
save_table(kpi_t, 'report_kpis', caption='The five KPIs of the current operation.', label='tab:kpis',
           col_format='p{2.4cm}rlp{3.2cm}')

# Table: stockouts by ABC class
so = load('kpi3_stockout_by_abc')
so_t = pd.DataFrame({'Class': so['abc_class'], 'Lines (count)': so['total_lines'].map('{:,.0f}'.format),
                     'Short-shipped lines (count)': so['stockout_lines'].map('{:,.0f}'.format),
                     'Stockout rate (%)': so['stockout_rate'].map('{:.1f}'.format),
                     'Share of all stockouts (%)': so['pct_of_total_stockouts'].map('{:.1f}'.format)})
save_table(so_t, 'report_stockouts', caption='Stockouts are rare in class A (4.3\\%) but frequent in class C (19.7\\%); A and C give the same share of all stockouts.',
           label='tab:stockouts')

# Table: checks of the AS2 storage plan
a2 = load('as2_assignment'); kp = load('as2_management_kpis').set_index('KPI')['Value']
rep = load('as2_replenishment_check'); svc = load('as2_service_check')
cap_ok = (a2['S_i'] * 240 >= a2['Max_capped'] - 1e-9).mean() * 100
checks = pd.DataFrame([
    ('Model travel cost (picks x m)', f"{kp['Current travel cost']:,.0f} to {kp['Proposed travel cost']:,.0f} ({-kp['Travel distance saved']:.1f}%)"),
    ('Locations used', f"{kp['Locations used']:,.0f} of {kp['Locations available']:,.0f} ({kp['Space utilization']:.1f}%)"),
    ('Capacity: SKUs whose space covers Max', f'{cap_ok:.1f}%'),
    ('Replenishment: median days of stock', f"{rep['days_of_stock'].median():.0f} days"),
    ('Replenishment: refills of the 3,500 SKUs', f"{rep['refills_30d'].sum():,.0f} in 30 days ({rep['refills_30d'].sum() / 30:.0f} per day)"),
    ('Service: fewest days of stock', f"{svc['days_of_stock'].min():.1f} days"),
], columns=['Measure', 'Result'])
save_table(checks, 'report_as2_checks', caption='The proposed storage assignment passes the capacity, replenishment and service checks.',
           label='tab:as2-checks', col_format='p{4.3cm}p{3.7cm}')

# Table: layout options (AS3)
sens3 = load('as3_sensitivity'); cmp3 = load('as3_comparison')
trips0 = cmp3.loc[cmp3['option'] == 'Current', 'trips_30d'].iloc[0]
lay = [('Current', 0, 0.0, trips0, 0.0)]
for _, r in sens3.iterrows():
    lay.append((r['share_of_extra_locations'], r['extra_locations'], r['extra_area_m2'], trips0 - r['trips_saved_30d'], r['trips_saved_30d']))
lay_t = pd.DataFrame(lay, columns=['Extra locations given', 'Locations (count)', 'Floor area (m2)', 'Refill trips in 30 days (count)', 'Trips saved (count)'])
lay_t['Time saved at 5 min per trip (h)'] = lay_t['Trips saved (count)'] * 5 / 60
lay_t['Locations (count)'] = lay_t['Locations (count)'].map('{:,.0f}'.format)
for c in ['Floor area (m2)', 'Refill trips in 30 days (count)', 'Trips saved (count)', 'Time saved at 5 min per trip (h)']:
    lay_t[c] = lay_t[c].map('{:,.0f}'.format)
save_table(lay_t, 'report_layout_options', caption='Each extra location saves fewer refill trips than the one before: 50\\% of the locations give 92\\% of the saving.',
           label='tab:layout-options')

# Table: routing methods (AS4)
m4 = load('as4_methods_comparison'); lt = load('as4_labor_time')
lab = {r['Method'][0]: r for _, r in lt.iterrows()}
rows = []
for _, r in m4.iterrows():
    k = r['Method'][0]
    rows.append({'Method': r['Method'], 'Distance (km)': f"{r['Total (km)']:,.0f}", 'Change vs B (%)': f"{r['vs B (%)']:+.1f}",
                 'Batches shorter than B (%)': f"{r['Batches shorter than B (%)']:.1f}",
                 'Labour time (h)': f"{lab[k]['Total (h)']:,.0f}" if k in lab else '--',
                 'Pickers (count)': f"{lab[k]['Pickers needed']:.1f}" if k in lab else '--'})
save_table(pd.DataFrame(rows), 'report_routing_methods',
           caption='Walking distance of the five routing methods over 2,800 batches. Method E is the shortest; labour time is shown for A, B and E.',
           label='tab:routing', col_format='p{2.5cm}rrrrr', wide=False)

# Table: scenario definitions, compact version of Table 1 of WH-AS5
t1 = load('as5_table1_scenarios')
t1c = pd.DataFrame({'Scenario': t1['Scenario'], 'Name': t1['Name'],
                    'Layout': t1['Layout'].map({'Baseline': 'no', 'Proposed': 'yes'}),
                    'Storage': t1['Storage'].map({'Baseline': 'no', 'Proposed': 'yes'}),
                    'Picking': t1['Picking'].map({'Baseline': 'no', 'Proposed': 'yes'}),
                    'Extra space (m2)': t1['Extra space (m²)'].astype(str),
                    'SKUs re-slotted (count)': t1['SKUs re-slotted (count)'].astype(str),
                    'Picked SKUs displaced (count)': t1['Picked SKUs displaced (count)'].astype(str),
                    'Busiest aisle (lines/day)': t1['Busiest aisle (lines/day)'].astype(str)})
save_table(t1c, 'as5_table1_compact', caption='The eight scenarios: decisions switched on (yes/no) and their resources. All are feasible.',
           label='tab:as5-scenarios', col_format='llcccrrrr', wide=True)


[saved] artifacts\report_assumptions.csv
[saved] artifacts\report_assumptions.tex
[saved] artifacts\report_kpis.csv
[saved] artifacts\report_kpis.tex
[saved] artifacts\report_stockouts.csv
[saved] artifacts\report_stockouts.tex
[saved] artifacts\report_as2_checks.csv
[saved] artifacts\report_as2_checks.tex
[saved] artifacts\report_layout_options.csv
[saved] artifacts\report_layout_options.tex
[saved] artifacts\report_routing_methods.csv
[saved] artifacts\report_routing_methods.tex
[saved] artifacts\as5_table1_compact.csv
[saved] artifacts\as5_table1_compact.tex


## 10. Summary table of the intervention comparison

One table with scenario definitions, feasibility status, resources, performance and change from the current situation. It is built from `as5_scenario_results.csv`.

In [78]:
# --- WH-FINAL: one intervention-comparison table (definitions, feasibility, resources, performance, change vs S0)
import pandas as pd
_sc = pd.read_csv(ARTIFACTS / "as5_scenario_results.csv")
_lab = {'B': 'B', 'P': 'new'}
def _rng(lo, hi, fmt):
    return fmt.format(lo) if abs(lo - hi) < 0.05 else (fmt + " to " + fmt).format(lo, hi)
_rows = []
for _, r in _sc.iterrows():
    _feas = ('Baseline' if r['Scenario'] == 'S0' else 'Feasible' if r['Status'] == 'Feasible'
             else 'Feasible, SKUs must move' if r['Picked outside SKUs displaced'] > 0 else 'Feasible, more walking with B')
    _rows.append({
        'Scenario': r['Scenario'], 'Name': r['Name'],
        'Layout': 'yes' if r['Layout'] == 'P' else 'no',
        'Storage': 'yes' if r['Storage'] == 'P' else 'no',
        'Picking rule': 'E' if r['Picking'] == 'P' else 'B',
        'Feasibility': _feas,
        'Extra space (m2)': round(r['Extra space (m2)']),
        'SKUs re-slotted (count)': int(r['Scope SKUs reassigned']),
        'Picked SKUs displaced (count)': int(r['Picked outside SKUs displaced']),
        'Refill trips (count)': round(r['Refill trips']),
        'Labour time (h)': _rng(r['Total, best (h)'], r['Total, worst (h)'], "{:.0f}"),
        'Change vs S0 (h)': _rng(r['Total, best (h)'] - _sc.loc[0, 'Total, best (h)'], r['Total, worst (h)'] - _sc.loc[0, 'Total, worst (h)'], "{:+.0f}"),
        'Change vs S0 (%)': _rng(r['Change vs S0, best (%)'], r['Change vs S0, worst (%)'], "{:+.1f}"),
        'Pickers, worst case (count)': round(r['Pickers, worst'], 1),
        'Busiest aisle (lines/day)': round(r['Lines per day, busiest aisle']),
    })
_tab = pd.DataFrame(_rows)
_tab.to_csv(ARTIFACTS / "as5_table_summary.csv", index=False)
def _tex(v): return str(v).replace('%', r'\%').replace('_', r'\_')
_cols = list(_tab.columns)
_head = ['Scenario', 'Layout', 'Storage', 'Pick rule', 'Feasible', r'\shortstack{Extra space\\(m$^2$)}', r'\shortstack{Re-slotted\\(SKUs)}',
         r'\shortstack{Displaced\\(SKUs)}', r'\shortstack{Refill trips\\(count)}', r'\shortstack{Labour time\\(h)}',
         r'\shortstack{Change\\(h)}', r'\shortstack{Change\\(\%)}', r'\shortstack{Pickers\\(count)}', r'\shortstack{Busiest aisle\\(lines/day)}']
_use = ['Scenario', 'Layout', 'Storage', 'Picking rule', 'Feasibility', 'Extra space (m2)', 'SKUs re-slotted (count)',
        'Picked SKUs displaced (count)', 'Refill trips (count)', 'Labour time (h)', 'Change vs S0 (h)', 'Change vs S0 (%)',
        'Pickers, worst case (count)', 'Busiest aisle (lines/day)']
_lines = [r"\begin{table*}[t]", r"\small",
          r"\caption{Comparison of the eight scenarios (S0 = current warehouse; S1 to S7 switch on the layout change, the storage change and the new picking rule E). Labour time is given from the best to the worst placement of the displaced SKUs over 22 picking days. S7 saves most (9.4\% to 10.4\%); S3 keeps most of that saving with no displaced SKUs; the storage change alone (S4) is worse than today.}",
          r"\label{tab:as5-scenarios}",
          r"\fitwidth{\textwidth}{\begin{tabular}{l" + "c" * 4 + "r" * 9 + "}", r"\toprule",
          " & ".join(_head) + r" \\", r"\midrule"]
for _, r in _tab.iterrows():
    _lines.append(" & ".join(_tex(r[c]) for c in _use) + r" \\")
_lines += [r"\bottomrule", r"\end{tabular}}", r"\end{table*}"]
(ARTIFACTS / "as5_table_summary.tex").write_text("\n".join(_lines) + "\n", encoding="utf-8")
print(_tab.to_string(index=False))


Scenario                      Name Layout Storage Picking rule                   Feasibility  Extra space (m2)  SKUs re-slotted (count)  Picked SKUs displaced (count)  Refill trips (count) Labour time (h) Change vs S0 (h) Change vs S0 (%)  Pickers, worst case (count)  Busiest aisle (lines/day)
      S0         Current warehouse     no      no            B                      Baseline                 0                        0                              0                   881             874               +0             +0.0                          5.7                        115
      S1       Picking change only     no      no            E                      Feasible                 0                        0                              0                   881             849              -25             -2.8                          5.5                        115
      S2        Layout change only    yes      no            B Feasible, more walking with B                99     

## 11. Break-even cost of Phase 2

We have no cost data, so the cell computes how many minutes a SKU move may take for Phase 2 (S7 instead of S3) to pay back its relocation work, over one or twelve periods of 22 picking days (assuming the demand repeats).

**Limits of the model (not tested here):** each aisle is one point, so S-shape and largest-gap routing are not tested; there is no queuing at the busiest aisle; the 2,800 batches are fixed as in the logs (no batching optimisation); demand comes from one 22-day window, so a periodic re-slotting policy should be tested; and each refill is one trip of an assumed time, with no batched refill routes.

In [79]:
# --- WH-FINAL: break-even cost of Phase 2 (S7 compared with S3), no cost data needed -------------
import pandas as pd
_sc = pd.read_csv(ARTIFACTS / "as5_scenario_results.csv").set_index("Scenario")
_moves = _sc.loc["S7", "Scope SKUs reassigned"] + _sc.loc["S7", "Picked outside SKUs displaced"]
_rows = []
for _case, _col in [("best", "Total, best (h)"), ("worst", "Total, worst (h)")]:
    _saved = _sc.loc["S3", _col] - _sc.loc["S7", _col]          # hours saved by Phase 2 per 22 picking days
    for _periods in (1, 12):                                    # one period = 22 picking days (assumption: demand repeats)
        _rows.append({"Relocation case": _case, "Hours saved per 22 days (h)": round(_saved, 1),
                      "SKU moves (count)": int(_moves), "Periods of 22 days (count)": _periods,
                      "Break-even time per SKU move (min)": round(_saved * _periods * 60 / _moves, 2)})
pd.DataFrame(_rows).to_csv(ARTIFACTS / "as5_phase2_breakeven.csv", index=False)
print(pd.DataFrame(_rows).to_string(index=False))


Relocation case  Hours saved per 22 days (h)  SKU moves (count)  Periods of 22 days (count)  Break-even time per SKU move (min)
           best                         24.4               4465                           1                                0.33
           best                         24.4               4465                          12                                3.94
          worst                         15.7               4465                           1                                0.21
          worst                         15.7               4465                          12                                2.53
